## 00 — Configuration de l'environnement expérimental

 Objectif

Cette étape consiste à préparer et vérifier l'environnement logiciel et matériel nécessaire au développement d'ADAS-FUSION X 2030.

L'environnement expérimental repose sur Python, OpenCV, NumPy, Pandas et PyTorch. Les expériences d'apprentissage profond pourront utiliser l'accélérateur GPU de Kaggle.

 Environnement

Plateforme : Kaggle Notebooks

Langage : Python

Traitement vidéo : OpenCV

Calcul scientifique : NumPy, Pandas

Deep Learning : PyTorch

Accélération : GPU CUDA si disponible

 Méthodologie

Nous vérifions les versions des bibliothèques, la disponibilité du GPU et l'organisation des répertoires de travail.

Critères de validation

L'environnement est considéré comme opérationnel si les bibliothèques principales sont importables, si les répertoires de sortie sont accessibles et si le dispositif de calcul est correctement identifié.

In [ ]:
# ============================================================
# ADAS-FUSION X 2030
# ETAPE 00 : ENVIRONNEMENT EXPERIMENTAL
# ============================================================

import os
import sys
import platform
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt

# Informations système
print("=" * 60)
print("ADAS-FUSION X 2030 - ENVIRONMENT CHECK")
print("=" * 60)

print("Python :", sys.version.split()[0])
print("Platform :", platform.platform())
print("NumPy :", np.__version__)
print("Pandas :", pd.__version__)
print("OpenCV :", cv2.__version__)
print("PyTorch :", torch.__version__)

# Vérification GPU
print("\n--- COMPUTING DEVICE ---")

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU disponible :", torch.cuda.get_device_name(0))
    print("CUDA :", torch.version.cuda)
else:
    DEVICE = "cpu"
    print("GPU CUDA indisponible : utilisation du CPU")

print("Device sélectionné :", DEVICE)

# Organisation des répertoires
PROJECT_DIR = Path("/kaggle/working/ADAS_FUSION_X2030")

DIRS = {
    "videos": PROJECT_DIR / "videos",
    "frames": PROJECT_DIR / "frames",
    "detections": PROJECT_DIR / "detections",
    "tracking": PROJECT_DIR / "tracking",
    "segmentation": PROJECT_DIR / "segmentation",
    "depth": PROJECT_DIR / "depth",
    "trajectories": PROJECT_DIR / "trajectories",
    "risk": PROJECT_DIR / "risk_analysis",
    "metrics": PROJECT_DIR / "metrics",
    "figures": PROJECT_DIR / "figures",
    "reports": PROJECT_DIR / "reports"
}

for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

print("\n--- PROJECT DIRECTORIES ---")

for name, directory in DIRS.items():
    print(f"{name:15s}: {directory}")

print("\nConfiguration terminée.")

# 01 — Importation et caractérisation de la vidéo de référence

 Objectif

Cette étape consiste à importer et caractériser la vidéo de référence utilisée dans les expériences d'ADAS-FUSION X 2030.

Fichier attendu :

`ADAS_FUSION_X2030_Reference_Driving_Video.mp4`

Paramètres à caractériser

- Durée totale de la séquence.
- Résolution spatiale.
- Fréquence d'images (FPS).
- Nombre total d'images.
- Codec vidéo lorsque disponible.
- Lisibilité et intégrité des images.

 Méthodologie

OpenCV est utilisé pour ouvrir le fichier vidéo, extraire ses métadonnées et lire plusieurs images réparties dans la séquence.

La vidéo originale est conservée. Une éventuelle normalisation à 180 secondes sera réalisée dans une étape séparée, sans modifier la source.

 Résultats attendus

Une fiche descriptive de la vidéo, des images représentatives et une confirmation de la capacité de décodage de la séquence.

In [ ]:
# ============================================================
# ETAPE 01 : RECHERCHE DE LA VIDEO
# ============================================================

from pathlib import Path

VIDEO_NAME = "ADAS_FUSION_X2030_Reference_Driving_Video"

SEARCH_DIRS = [
    Path("/kaggle/input"),
    Path("/kaggle/working")
]

video_candidates = []

for root in SEARCH_DIRS:
    if root.exists():
        for extension in ["*.mp4", "*.webm", "*.avi", "*.mov", "*.mkv"]:
            video_candidates.extend(root.rglob(extension))

# Priorité au nom de notre vidéo
matching_videos = [
    p for p in video_candidates
    if p.stem.lower() == VIDEO_NAME.lower()
]

if matching_videos:
    VIDEO_PATH = matching_videos[0]
elif len(video_candidates) == 1:
    VIDEO_PATH = video_candidates[0]
else:
    print("Vidéos détectées :")
    for i, path in enumerate(video_candidates):
        print(i, path)

    raise FileNotFoundError(
        "Vidéo non identifiée automatiquement. "
        "Définir VIDEO_PATH avec le chemin exact."
    )

print("Vidéo sélectionnée :")
print(VIDEO_PATH)
print("Extension :", VIDEO_PATH.suffix)
print("Taille (MB) :", round(VIDEO_PATH.stat().st_size / (1024**2), 2))

In [ ]:
# ============================================================
# CARACTERISATION DE LA VIDEO
# ============================================================

import cv2
import pandas as pd

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError(
        "Impossible d'ouvrir la vidéo. "
        "Vérifier le fichier et son codec."
    )

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = frame_count / fps if fps > 0 else 0

video_info = {
    "Fichier": VIDEO_PATH.name,
    "Résolution": f"{width} × {height}",
    "FPS": round(fps, 3),
    "Nombre d'images": frame_count,
    "Durée (secondes)": round(duration, 3),
    "Durée (minutes)": round(duration / 60, 3),
    "Taille (MB)": round(VIDEO_PATH.stat().st_size / (1024**2), 2)
}

video_df = pd.DataFrame(
    video_info.items(),
    columns=["Paramètre", "Valeur"]
)

display(video_df)

cap.release()

In [ ]:
# ============================================================
# ADAS-FUSION X 2030
# ETAPE 01.4 : REPARATION ET INSPECTION VIDEO
# ============================================================

import cv2
import shutil
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Vérification de FFmpeg
FFMPEG = shutil.which("ffmpeg")

if FFMPEG is None:
    raise RuntimeError("FFmpeg est introuvable.")

# 2. Fichier de sortie
RECOVERED_VIDEO = (
    DIRS["videos"] / "ADAS_FUSION_X2030_Recovered.mp4"
)

# 3. Conversion avec récupération des images décodables
command = [
    FFMPEG,
    "-y",
    "-hide_banner",
    "-loglevel", "warning",
    "-fflags", "+discardcorrupt",
    "-err_detect", "ignore_err",
    "-i", str(VIDEO_PATH),
    "-map", "0:v:0",
    "-an",
    "-c:v", "libx264",
    "-preset", "veryfast",
    "-crf", "20",
    "-pix_fmt", "yuv420p",
    str(RECOVERED_VIDEO)
]

print("Conversion de la vidéo en cours...")

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

if result.returncode != 0 or not RECOVERED_VIDEO.exists():
    print(result.stderr[-4000:])
    raise RuntimeError(
        "La conversion a échoué. "
        "Le fichier source est peut-être endommagé."
    )

print("Conversion terminée.")

# 4. Vérification du MP4
cap = cv2.VideoCapture(str(RECOVERED_VIDEO))

if not cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir le MP4 converti.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = frame_count / fps if fps > 0 else 0

print("\n--- VIDEO RECUPEREE ---")
print("Résolution :", width, "x", height)
print("FPS :", fps)
print("Images :", frame_count)
print("Durée :", round(duration, 2), "secondes")

# 5. Extraction de cinq images
timestamps = np.linspace(0, max(duration - 0.1, 0), 5)

fig, axes = plt.subplots(1, 5, figsize=(22, 5))

valid_frames = 0

for ax, timestamp in zip(axes, timestamps):

    cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)

    success, frame = cap.read()

    if success:
        frame_rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        ax.imshow(frame_rgb)
        ax.set_title(f"t = {timestamp:.1f} s")
        valid_frames += 1
    else:
        ax.set_title("Image illisible")

    ax.axis("off")

plt.suptitle(
    "ADAS-FUSION X 2030 - Video Quality Inspection"
)
plt.tight_layout()
plt.show()

cap.release()

# 6. Validation
VIDEO_PATH = RECOVERED_VIDEO

print("\nImages correctement extraites :", valid_frames, "/ 5")
print("Nouveau chemin vidéo :", VIDEO_PATH)

if valid_frames == 5:
    print("VALIDATION : SUCCES")
else:
    print("VALIDATION : ECHEC PARTIEL")

## Validation expérimentale de la vidéo

La vidéo sera considérée comme exploitable si :

1. Le fichier peut être ouvert et décodé.
2. Les images sont lisibles et ne présentent pas de corruption importante.
3. La résolution est suffisante pour identifier les usagers de la route.
4. La fréquence d'images est connue et cohérente.
5. La durée réelle est mesurée et enregistrée.

Une durée de 180 secondes sera utilisée comme référence si nous décidons de normaliser la séquence à exactement trois minutes.

La qualité visuelle, la diversité des situations routières et la visibilité des objets devront également être évaluées avant l'entraînement des modèles.

In [ ]:
# ============================================================
# NORMALISATION : 180 SECONDES
# ============================================================

import subprocess
import shutil

TARGET_DURATION = 180

FFMPEG = shutil.which("ffmpeg")

if FFMPEG is None:
    raise RuntimeError("FFmpeg n'est pas disponible dans l'environnement.")

NORMALIZED_VIDEO = DIRS["videos"] / "ADAS_FUSION_X2030_Reference_180s.mp4"

if duration < TARGET_DURATION:
    raise ValueError(
        f"La vidéo dure seulement {duration:.2f} secondes. "
        "Impossible de produire une séquence complète de 180 secondes."
    )

command = [
    FFMPEG, "-y",
    "-i", str(VIDEO_PATH),
    "-t", str(TARGET_DURATION),
    "-an",
    "-c:v", "libx264",
    "-preset", "fast",
    "-crf", "18",
    "-pix_fmt", "yuv420p",
    str(NORMALIZED_VIDEO)
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("Échec de la normalisation vidéo.")

VIDEO_PATH = NORMALIZED_VIDEO

print("Vidéo de travail créée :", VIDEO_PATH)
print("Durée cible :", TARGET_DURATION, "secondes")

In [ ]:
# ============================================================
# ETAPE 01.7 : VALIDATION FINALE
# ============================================================

import cv2
from pathlib import Path

FINAL_VIDEO = Path(VIDEO_PATH)

cap = cv2.VideoCapture(str(FINAL_VIDEO))

assert cap.isOpened(), "ERREUR : vidéo impossible à ouvrir."

fps = cap.get(cv2.CAP_PROP_FPS)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = frames / fps if fps > 0 else 0

# Vérification de plusieurs images
timestamps = [0, 30, 60, 90, 120, 150, 179]
valid = 0

for t in timestamps:
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    success, frame = cap.read()
    if success and frame is not None:
        valid += 1

cap.release()

print("========== FINAL VIDEO VALIDATION ==========")
print("Fichier :", FINAL_VIDEO.name)
print("Résolution :", width, "x", height)
print("FPS :", round(fps, 2))
print("Nombre d'images :", frames)
print("Durée :", round(duration, 2), "secondes")
print("Images lisibles :", valid, "/", len(timestamps))

if abs(duration - 180) <= 1 and valid == len(timestamps):
    print("\nVALIDATION : SUCCESS")
else:
    print("\nVALIDATION : CHECK REQUIRED")

# Étape 02 — Détection des objets avec YOLO

## 02.1 — Vérifier l'environnement et installer Ultralytics

In [ ]:
# ============================================================
# ETAPE 02.1 — CONFIGURATION DU DETECTEUR YOLO
# ============================================================

!pip install -q ultralytics

import torch
import cv2
import ultralytics

from ultralytics import YOLO

print("========== ENVIRONNEMENT YOLO ==========")
print("Ultralytics :", ultralytics.__version__)
print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("GPU non détecté : utilisation du CPU")

## Étape 02.2 — Chargement du modèle YOLO

In [ ]:
# ============================================================
# ETAPE 02.2 — CHARGEMENT DU MODELE YOLO26
# ============================================================

from ultralytics import YOLO
import torch
from pathlib import Path

# Modèle initial
MODEL_NAME = "yolo26n.pt"

print("Chargement du modèle :", MODEL_NAME)

model = YOLO(MODEL_NAME)

# Vérification du modèle
print("\n========== MODELE YOLO ==========")
print("Modèle :", MODEL_NAME)
print("Tâche :", model.task)
print("GPU disponible :", torch.cuda.is_available())

# Vérification des classes COCO
class_names = model.names

print("\nNombre de classes :", len(class_names))
print("Classes disponibles :")

for class_id, class_name in class_names.items():
    print(f"{class_id:02d} : {class_name}")

print("\nVALIDATION : MODELE CHARGE")

## Étape 02.3 — Première détection sur notre vidéo

In [ ]:
# ============================================================
# ETAPE 02.3 — PREMIERE INFERENCE YOLO26n
# ============================================================

import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Vérification du chemin vidéo
VIDEO_PATH = Path(VIDEO_PATH)

assert VIDEO_PATH.exists(), f"Vidéo introuvable : {VIDEO_PATH}"

# Extraction d'une image à 30 secondes
cap = cv2.VideoCapture(str(VIDEO_PATH))

assert cap.isOpened(), "Impossible d'ouvrir la vidéo."

cap.set(cv2.CAP_PROP_POS_MSEC, 30_000)
success, frame = cap.read()

cap.release()

assert success, "Impossible d'extraire l'image."

# Inférence YOLO sur GPU
results = model.predict(
    source=frame,
    imgsz=640,
    conf=0.25,
    device=0,
    verbose=False
)

result = results[0]

# Image annotée
annotated_frame = result.plot()

# Conversion BGR -> RGB
annotated_rgb = cv2.cvtColor(
    annotated_frame,
    cv2.COLOR_BGR2RGB
)

# Affichage
plt.figure(figsize=(16, 9))
plt.imshow(annotated_rgb)
plt.title("ADAS-FUSION X 2030 — Première détection YOLO26n")
plt.axis("off")
plt.show()

# Tableau textuel des détections
print("\n========== RESULTATS DE DETECTION ==========")

if len(result.boxes) > 0:

    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()

        print(
            f"Objet : {model.names[class_id]:15s} | "
            f"Confiance : {confidence:.3f} | "
            f"Bounding Box : {xyxy}"
        )

else:
    print("Aucun objet détecté.")

print("\nNombre total d'objets :", len(result.boxes))

## Étape 02.4 — Analyse multi-images et filtrage ADAS

In [ ]:
# ============================================================
# ETAPE 02.4 — EVALUATION PRELIMINAIRE MULTI-IMAGES
# ============================================================

import cv2
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Classes pertinentes pour ADAS
ADAS_CLASSES = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck",
    9: "traffic light",
    11: "stop sign"
}

# Instants à analyser
TIMESTAMPS = [5, 15, 30, 45, 60, 90, 120, 150, 175]

cap = cv2.VideoCapture(str(VIDEO_PATH))

assert cap.isOpened(), "Impossible d'ouvrir la vidéo."

records = []
inference_times = []

for timestamp in TIMESTAMPS:

    cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)
    success, frame = cap.read()

    if not success:
        print(f"Image inaccessible à {timestamp}s")
        continue

    start = time.perf_counter()

    result = model.predict(
        source=frame,
        imgsz=640,
        conf=0.25,
        device=0,
        classes=list(ADAS_CLASSES.keys()),
        verbose=False
    )[0]

    # Synchronisation GPU pour mesurer correctement le temps
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    inference_times.append(elapsed)

    detections = result.boxes

    for box in detections:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        records.append({
            "timestamp_s": timestamp,
            "class": ADAS_CLASSES[class_id],
            "confidence": confidence,
            "inference_time_s": elapsed
        })

cap.release()

# Résultats
df_detections = pd.DataFrame(records)

print("========== EVALUATION MULTI-IMAGES ==========")
print("Images analysées :", len(TIMESTAMPS))
print("Images valides :", len(set(
    df_detections["timestamp_s"]
)) if not df_detections.empty else 0)

print("Nombre total de détections :", len(df_detections))

if inference_times:
    print("Temps moyen d'inférence :",
          round(np.mean(inference_times) * 1000, 2), "ms")

    print("Temps P95 :",
          round(np.percentile(inference_times, 95) * 1000, 2), "ms")

if not df_detections.empty:

    print("\nDétections par classe :")
    print(df_detections["class"].value_counts())

    print("\nConfiance moyenne :",
          round(df_detections["confidence"].mean(), 3))

    print("\nDétail des détections :")
    display(df_detections)

else:
    print("Aucune détection routière sur les images analysées.")

## Étape 02.4 — Version corrigée de l'évaluation

In [ ]:
# ============================================================
# ETAPE 02.4 — BENCHMARK YOLO26n CORRIGE
# ============================================================

import cv2
import time
import torch
import numpy as np
import pandas as pd
from pathlib import Path

ADAS_CLASSES = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck",
    9: "traffic light",
    11: "stop sign"
}

TIMESTAMPS = [5, 15, 30, 45, 60, 90, 120, 150, 175]

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Impossible d'ouvrir la vidéo."

# Warm-up GPU : première inférence non comptabilisée
cap.set(cv2.CAP_PROP_POS_MSEC, 30_000)
success, warmup_frame = cap.read()
assert success, "Impossible de lire l'image de warm-up."

_ = model.predict(
    source=warmup_frame,
    imgsz=640,
    conf=0.25,
    device=0,
    classes=list(ADAS_CLASSES.keys()),
    verbose=False
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

# Stockage des résultats
detection_records = []
image_records = []

for timestamp in TIMESTAMPS:

    cap.set(cv2.CAP_PROP_POS_MSEC, timestamp * 1000)
    success, frame = cap.read()

    if not success or frame is None:
        image_records.append({
            "timestamp_s": timestamp,
            "read_success": False,
            "detections": 0,
            "inference_ms": np.nan
        })
        continue

    # Synchronisation avant chronométrage
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    result = model.predict(
        source=frame,
        imgsz=640,
        conf=0.25,
        device=0,
        classes=list(ADAS_CLASSES.keys()),
        verbose=False
    )[0]

    # Synchronisation après inférence
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed_ms = (time.perf_counter() - start) * 1000

    boxes = result.boxes
    number_detections = len(boxes)

    image_records.append({
        "timestamp_s": timestamp,
        "read_success": True,
        "detections": number_detections,
        "inference_ms": elapsed_ms
    })

    for box in boxes:
        class_id = int(box.cls[0])

        detection_records.append({
            "timestamp_s": timestamp,
            "class": ADAS_CLASSES[class_id],
            "confidence": float(box.conf[0])
        })

cap.release()

# DataFrames indépendants
df_images = pd.DataFrame(image_records)
df_detections = pd.DataFrame(detection_records)

# Statistiques
valid_images = df_images["read_success"].sum()
images_with_detections = (
    df_images["detections"] > 0
).sum()

times = df_images.loc[
    df_images["read_success"], "inference_ms"
].dropna()

print("========== BENCHMARK YOLO26n ==========")
print("Instants demandés :", len(TIMESTAMPS))
print("Images correctement lues :", valid_images)
print("Images avec détection :", images_with_detections)
print("Images sans détection :", valid_images - images_with_detections)
print("Nombre total de détections :", len(df_detections))

if len(times) > 0:
    print("\nTemps moyen :", round(times.mean(), 2), "ms")
    print("Temps médian :", round(times.median(), 2), "ms")
    print("Temps P95 :", round(times.quantile(0.95), 2), "ms")
    print("FPS théorique inférence seule :",
          round(1000 / times.mean(), 2))

if not df_detections.empty:
    print("\nDétections par classe :")
    print(df_detections["class"].value_counts())

    print("\nConfiance moyenne :",
          round(df_detections["confidence"].mean(), 3))

print("\n========== DETAIL PAR IMAGE ==========")
display(df_images)

print("\n========== DETAIL DES DETECTIONS ==========")
display(df_detections)

# Étape 03 — Suivi multi-objets avec ByteTrack

## 03.1 — Premier test de suivi sur une séquence vidéo

In [ ]:
# ============================================================
# ETAPE 03.1 — YOLO26n + BYTETRACK
# ADAS-FUSION X 2030
# ============================================================

import cv2
import torch
import time
import pandas as pd
from pathlib import Path
from collections import defaultdict
from IPython.display import Video, display

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

START_TIME = 55       # Début de la séquence (secondes)
DURATION = 15         # Durée du test
CONFIDENCE = 0.25
IMAGE_SIZE = 640

OUTPUT_DIR = Path("/kaggle/working/ADAS_FUSION_X2030")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "ADAS_FUSION_X2030_ByteTrack_Test.mp4"

ADAS_CLASSES = [0, 1, 2, 3, 5, 7, 9, 11]

# ------------------------------------------------------------
# 2. OUVERTURE DE LA VIDEO
# ------------------------------------------------------------

cap = cv2.VideoCapture(str(VIDEO_PATH))

assert cap.isOpened(), "Impossible d'ouvrir la vidéo."

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

cap.set(cv2.CAP_PROP_POS_MSEC, START_TIME * 1000)

total_frames = int(DURATION * fps)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    fourcc,
    fps,
    (width, height)
)

assert writer.isOpened(), "Impossible de créer la vidéo de sortie."

# ------------------------------------------------------------
# 3. INITIALISATION DU TRACKING
# ------------------------------------------------------------

track_observations = []
frame_count = 0
frames_with_tracks = 0
unique_tracks = set()

start_time = time.perf_counter()

# ------------------------------------------------------------
# 4. TRAITEMENT IMAGE PAR IMAGE
# ------------------------------------------------------------

for frame_index in range(total_frames):

    success, frame = cap.read()

    if not success:
        break

    results = model.track(
        source=frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=CONFIDENCE,
        iou=0.5,
        imgsz=IMAGE_SIZE,
        device=0,
        classes=ADAS_CLASSES,
        verbose=False
    )

    result = results[0]
    annotated_frame = result.plot()

    frame_count += 1

    # Récupération des identifiants
    if result.boxes is not None and result.boxes.id is not None:

        frames_with_tracks += 1

        boxes = result.boxes
        track_ids = boxes.id.int().cpu().tolist()
        class_ids = boxes.cls.int().cpu().tolist()
        confidences = boxes.conf.cpu().tolist()

        for track_id, class_id, confidence in zip(
            track_ids, class_ids, confidences
        ):

            class_name = model.names[class_id]
            unique_tracks.add(track_id)

            track_observations.append({
                "frame": frame_count,
                "timestamp_s": START_TIME + frame_count / fps,
                "track_id": track_id,
                "class": class_name,
                "confidence": confidence
            })

    writer.write(annotated_frame)

cap.release()
writer.release()

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start_time

# ------------------------------------------------------------
# 5. RESULTATS
# ------------------------------------------------------------

df_tracks = pd.DataFrame(track_observations)

print("========== RESULTATS BYTETRACK ==========")
print("Images traitées :", frame_count)
print("Durée vidéo traitée :", round(frame_count / fps, 2), "s")
print("Images avec identifiants :", frames_with_tracks)
print("Identifiants uniques :", len(unique_tracks))
print("Observations de tracking :", len(df_tracks))
print("Temps total de traitement :", round(elapsed, 2), "s")

if elapsed > 0:
    print("Débit réel de traitement :",
          round(frame_count / elapsed, 2), "FPS")

print("\nVidéo enregistrée :", OUTPUT_VIDEO)

if not df_tracks.empty:

    print("\nNombre de trajectoires observées :")
    print(df_tracks.groupby("class")["track_id"].nunique())

    print("\nExtrait des observations :")
    display(df_tracks.head(15))

else:
    print("Aucun identifiant de suivi obtenu.")

# ------------------------------------------------------------
# 6. VISUALISATION
# ------------------------------------------------------------

if OUTPUT_VIDEO.exists() and OUTPUT_VIDEO.stat().st_size > 0:
    display(Video(
        str(OUTPUT_VIDEO),
        embed=True,
        width=800
    ))

## Étape 03.2 — Vérification de la cohérence des identifiants

In [ ]:
# ============================================================
# ETAPE 03.2 — AUDIT DES IDENTIFIANTS BYTETRACK
# ============================================================

assert not df_tracks.empty, "Aucune observation disponible."

# Nombre réel d'identifiants distincts
total_unique_ids = df_tracks["track_id"].nunique()

# Nombre de classes associées à chaque ID
id_audit = (
    df_tracks.groupby("track_id")
    .agg(
        classes=("class", lambda x: ", ".join(sorted(set(x)))),
        number_of_classes=("class", "nunique"),
        observations=("frame", "count"),
        first_frame=("frame", "min"),
        last_frame=("frame", "max")
    )
    .reset_index()
)

# IDs associés à plusieurs classes
reused_ids = id_audit[
    id_audit["number_of_classes"] > 1
]

print("========== AUDIT DES TRACK IDs ==========")
print("Nombre total d'observations :", len(df_tracks))
print("IDs uniques globaux :", total_unique_ids)
print("IDs associés à plusieurs classes :", len(reused_ids))

print("\nRépartition des IDs par classe :")
print(
    df_tracks.groupby("class")["track_id"]
    .nunique()
)

print("\nDétail des IDs réutilisés entre classes :")
if not reused_ids.empty:
    display(reused_ids)
else:
    print("Aucun ID associé à plusieurs classes.")

print("\nRésumé des trajectoires :")
display(id_audit.sort_values(
    "observations",
    ascending=False
).head(20))

## Étape 03.3 — Identifier précisément les changements de classe

In [ ]:
# ============================================================
# ETAPE 03.3 — ANALYSE TEMPORELLE DES CHANGEMENTS DE CLASSE
# ============================================================

import pandas as pd

# IDs présentant plusieurs classes
ambiguous_ids = (
    df_tracks.groupby("track_id")["class"]
    .nunique()
)

ambiguous_ids = ambiguous_ids[
    ambiguous_ids > 1
].index.tolist()

print("IDs à examiner :", ambiguous_ids)

# ------------------------------------------------------------
# 1. Chronologie des changements de classe
# ------------------------------------------------------------

transition_records = []

for track_id in ambiguous_ids:

    track = (
        df_tracks[df_tracks["track_id"] == track_id]
        .sort_values("frame")
    )

    previous_class = None

    for _, row in track.iterrows():

        current_class = row["class"]

        if previous_class is not None and current_class != previous_class:

            transition_records.append({
                "track_id": track_id,
                "frame": int(row["frame"]),
                "timestamp_s": round(row["timestamp_s"], 3),
                "previous_class": previous_class,
                "new_class": current_class,
                "confidence": round(row["confidence"], 3)
            })

        previous_class = current_class

transitions_df = pd.DataFrame(transition_records)

print("\n========== CHANGEMENTS DE CLASSE ==========")

if not transitions_df.empty:
    display(transitions_df)
else:
    print("Aucun changement temporel de classe détecté.")

# ------------------------------------------------------------
# 2. Répartition des classes par ID
# ------------------------------------------------------------

print("\n========== DISTRIBUTION PAR IDENTIFIANT ==========")

class_distribution = (
    df_tracks[df_tracks["track_id"].isin(ambiguous_ids)]
    .groupby(["track_id", "class"])
    .agg(
        observations=("frame", "count"),
        first_frame=("frame", "min"),
        last_frame=("frame", "max"),
        mean_confidence=("confidence", "mean")
    )
    .reset_index()
)

display(class_distribution)

# ------------------------------------------------------------
# 3. Recherche de doublons ID / frame
# ------------------------------------------------------------

duplicate_records = df_tracks[
    df_tracks.duplicated(
        subset=["track_id", "frame"],
        keep=False
    )
].sort_values(["track_id", "frame"])

print("\n========== DOUBLONS ID / FRAME ==========")

print("Nombre de lignes concernées :",
      len(duplicate_records))

if not duplicate_records.empty:
    display(duplicate_records)
else:
    print("Aucun doublon ID/frame.")

## Étape 03.4 — Diagnostic visuel des pistes ambiguës

In [ ]:
# ============================================================
# ETAPE 03.4 — DIAGNOSTIC VISUEL DES PISTES AMBIGUES
# ============================================================

import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO
from pathlib import Path

# Nouvelle instance : tracker indépendant
audit_model = YOLO(MODEL_NAME)

START_TIME = 55
DURATION = 15
FPS_VIDEO = 24

# Frames relatives à la séquence de 55 secondes
FRAMES_TO_INSPECT = [
    116, 117, 118, 126, 127,
    135, 138, 167, 168,
    214, 215, 271, 273,
    319, 330, 335, 341, 345, 346, 348, 349, 354, 355
]

cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Vidéo inaccessible."

cap.set(cv2.CAP_PROP_POS_MSEC, START_TIME * 1000)

audit_records = []
selected_frames = {}

for frame_index in range(int(DURATION * FPS_VIDEO)):

    success, frame = cap.read()

    if not success:
        break

    result = audit_model.track(
        source=frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.25,
        iou=0.5,
        imgsz=640,
        device=0,
        classes=[0, 1, 2, 3, 5, 7, 9, 11],
        verbose=False
    )[0]

    if result.boxes is not None and result.boxes.id is not None:

        boxes = result.boxes

        for box, track_id, class_id, confidence in zip(
            boxes.xyxy.cpu().numpy(),
            boxes.id.int().cpu().numpy(),
            boxes.cls.int().cpu().numpy(),
            boxes.conf.cpu().numpy()
        ):

            x1, y1, x2, y2 = box.tolist()

            audit_records.append({
                "frame": frame_index + 1,
                "timestamp_s": START_TIME + (frame_index + 1) / FPS_VIDEO,
                "track_id": int(track_id),
                "class": audit_model.names[int(class_id)],
                "confidence": float(confidence),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2
            })

    if frame_index + 1 in FRAMES_TO_INSPECT:
        selected_frames[frame_index + 1] = result.plot()

cap.release()

df_tracks_bbox = pd.DataFrame(audit_records)

print("========== DIAGNOSTIC VISUEL ==========")
print("Observations enregistrées :", len(df_tracks_bbox))
print("Frames sélectionnées :", len(selected_frames))

# ------------------------------------------------------------
# TABLEAU DES PISTES AUX FRAMES SUSPECTES
# ------------------------------------------------------------

SUSPECT_IDS = [19, 33, 40, 54, 97, 136, 169, 179]

diagnostic_table = df_tracks_bbox[
    df_tracks_bbox["track_id"].isin(SUSPECT_IDS)
].copy()

display(
    diagnostic_table[
        [
            "frame", "track_id", "class", "confidence",
            "x1", "y1", "x2", "y2"
        ]
    ].sort_values(["track_id", "frame"])
)

# ------------------------------------------------------------
# MOSAIQUE DES IMAGES
# ------------------------------------------------------------

frames_to_show = list(selected_frames.keys())

if frames_to_show:

    columns = 3
    rows = (len(frames_to_show) + columns - 1) // columns

    fig, axes = plt.subplots(
        rows, columns,
        figsize=(18, 5 * rows)
    )

    axes = axes.flatten()

    for ax, frame_number in zip(axes, frames_to_show):

        image_rgb = cv2.cvtColor(
            selected_frames[frame_number],
            cv2.COLOR_BGR2RGB
        )

        ax.imshow(image_rgb)
        ax.set_title(
            f"Frame {frame_number} | "
            f"t = {START_TIME + frame_number / FPS_VIDEO:.2f} s"
        )
        ax.axis("off")

    for ax in axes[len(frames_to_show):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Étape 03.5 — Stabilisation temporelle des classes

In [ ]:
# ============================================================
# ETAPE 03.5 — STABILISATION TEMPORELLE DES CLASSES
# ============================================================

import pandas as pd

# Conservation des données brutes
df_tracks_stable = df_tracks_bbox.copy()

# ------------------------------------------------------------
# 1. CLASSE DOMINANTE PAR TRACK ID
# ------------------------------------------------------------

class_statistics = (
    df_tracks_stable
    .groupby(["track_id", "class"])
    .agg(
        observations=("confidence", "count"),
        mean_confidence=("confidence", "mean"),
        total_confidence=("confidence", "sum")
    )
    .reset_index()
)

# Classe dominante selon le nombre d'observations
dominant_classes = (
    class_statistics
    .sort_values(
        ["track_id", "observations", "total_confidence"],
        ascending=[True, False, False]
    )
    .drop_duplicates("track_id")
    [["track_id", "class"]]
    .rename(columns={"class": "stable_class"})
)

# ------------------------------------------------------------
# 2. FUSION AVEC LES DONNEES ORIGINALES
# ------------------------------------------------------------

df_tracks_stable = df_tracks_stable.merge(
    dominant_classes,
    on="track_id",
    how="left"
)

# ------------------------------------------------------------
# 3. INDICATEURS DE STABILITE
# ------------------------------------------------------------

df_tracks_stable["class_corrected"] = (
    df_tracks_stable["class"] !=
    df_tracks_stable["stable_class"]
)

correction_count = df_tracks_stable["class_corrected"].sum()

print("========== STABILISATION TEMPORELLE ==========")

print("Observations originales :", len(df_tracks_stable))
print("Observations corrigées :", correction_count)

print(
    "Taux de corrections :",
    round(100 * correction_count / len(df_tracks_stable), 2),
    "%"
)

print("\nComparaison des classes :")

display(
    df_tracks_stable[
        [
            "track_id",
            "class",
            "stable_class",
            "confidence",
            "class_corrected"
        ]
    ].head(20)
)

print("\nRépartition des classes stabilisées :")

print(
    df_tracks_stable
    .groupby("stable_class")["track_id"]
    .nunique()
)

## Étape 03.6 — Audit de cohérence par piste

In [ ]:
# ============================================================
# ETAPE 03.6 — AUDIT DE COHERENCE DES PISTES
# ============================================================

import pandas as pd

df_audit = df_tracks_stable.copy()
df_audit = df_audit.sort_values(["track_id", "frame"])

# ------------------------------------------------------------
# 1. STATISTIQUES PAR TRACK
# ------------------------------------------------------------

audit_summary = (
    df_audit.groupby("track_id")
    .agg(
        observations=("frame", "count"),
        nombre_classes=("class", "nunique"),
        classe_dominante=("stable_class", "first"),
        premiere_frame=("frame", "min"),
        derniere_frame=("frame", "max"),
        confiance_moyenne=("confidence", "mean")
    )
)

# ------------------------------------------------------------
# 2. PROPORTION DE LA CLASSE DOMINANTE
# ------------------------------------------------------------

dominant_counts = (
    df_audit.groupby(["track_id", "class"])
    .size()
    .rename("effectif")
    .reset_index()
)

dominant_ratio = (
    dominant_counts.groupby("track_id")["effectif"].max()
    / df_audit.groupby("track_id").size()
)

audit_summary["part_classe_dominante"] = dominant_ratio

# ------------------------------------------------------------
# 3. NOMBRE DE TRANSITIONS DE CLASSE
# ------------------------------------------------------------

df_audit["classe_precedente"] = (
    df_audit.groupby("track_id")["class"].shift(1)
)

df_audit["transition"] = (
    df_audit["class"] != df_audit["classe_precedente"]
)

transition_counts = (
    df_audit.groupby("track_id")["transition"].sum()
)

audit_summary["transitions_classe"] = transition_counts

# ------------------------------------------------------------
# 4. DUREE D'OBSERVATION
# ------------------------------------------------------------

audit_summary["duree_s"] = (
    audit_summary["derniere_frame"]
    - audit_summary["premiere_frame"]
) / FPS_VIDEO

# ------------------------------------------------------------
# 5. IDENTIFICATION DES PISTES AMBIGUES
# ------------------------------------------------------------

audit_summary = audit_summary.reset_index()

ambiguous_tracks = audit_summary[
    (audit_summary["nombre_classes"] > 1)
    | (audit_summary["part_classe_dominante"] < 0.95)
    | (audit_summary["transitions_classe"] > 0)
]

print("========== AUDIT DES PISTES ==========")
print("Nombre total de pistes :", len(audit_summary))
print("Pistes ambiguës :", len(ambiguous_tracks))

display(
    audit_summary.sort_values(
        ["nombre_classes", "transitions_classe", "observations"],
        ascending=[False, False, False]
    )
)

print("\n========== PISTES A EXAMINER ==========")

display(
    ambiguous_tracks[
        [
            "track_id",
            "observations",
            "nombre_classes",
            "classe_dominante",
            "part_classe_dominante",
            "transitions_classe",
            "duree_s"
        ]
    ].sort_values("transitions_classe", ascending=False)
)

## Étape 03.7 — Correction de l'audit

In [ ]:
# ============================================================
# ETAPE 03.7 — AUDIT CORRIGE DES PISTES
# ============================================================

df_audit = df_tracks_stable.copy()
df_audit = df_audit.sort_values(["track_id", "frame"])

# 1. Statistiques générales par piste
audit_corrected = (
    df_audit.groupby("track_id")
    .agg(
        observations=("frame", "count"),
        nombre_classes=("class", "nunique"),
        classe_dominante=("stable_class", "first"),
        premiere_frame=("frame", "min"),
        derniere_frame=("frame", "max"),
        confiance_moyenne=("confidence", "mean")
    )
)

# 2. Proportion de la classe dominante
dominant_counts = (
    df_audit.groupby(["track_id", "class"])
    .size()
    .rename("effectif")
    .reset_index()
)

dominant_ratio = (
    dominant_counts.groupby("track_id")["effectif"].max()
    / df_audit.groupby("track_id").size()
)

audit_corrected["part_classe_dominante"] = dominant_ratio

# 3. Transitions : la première observation n'est pas une transition
df_audit["classe_precedente"] = (
    df_audit.groupby("track_id")["class"].shift(1)
)

df_audit["transition"] = (
    df_audit["class"]
    .ne(df_audit["classe_precedente"])
    .fillna(False)
)

audit_corrected["transitions_classe"] = (
    df_audit.groupby("track_id")["transition"].sum()
)

# 4. Durée d'observation
audit_corrected["duree_s"] = (
    audit_corrected["derniere_frame"]
    - audit_corrected["premiere_frame"]
) / FPS_VIDEO

audit_corrected = audit_corrected.reset_index()

# 5. Définition des pistes suspectes
audit_corrected["suspecte"] = (
    (audit_corrected["nombre_classes"] > 1)
    & (audit_corrected["part_classe_dominante"] < 0.95)
)

# 6. Résultats
suspect_tracks = audit_corrected[
    audit_corrected["suspecte"]
].sort_values(
    ["part_classe_dominante", "transitions_classe"]
)

print("========== AUDIT CORRIGE ==========")
print("Nombre total de pistes :", len(audit_corrected))
print("Pistes suspectes :", len(suspect_tracks))

print("\n========== PISTES SUSPECTES ==========")

display(
    suspect_tracks[
        [
            "track_id",
            "observations",
            "nombre_classes",
            "classe_dominante",
            "part_classe_dominante",
            "transitions_classe",
            "duree_s"
        ]
    ]
)

print("\n========== PISTES STABLES ==========")

display(
    audit_corrected[
        ~audit_corrected["suspecte"]
    ][
        [
            "track_id",
            "observations",
            "classe_dominante",
            "part_classe_dominante",
            "transitions_classe"
        ]
    ]
)

## Étape 03.8 — Correction définitive du nombre de transitions

In [ ]:
# Correction du calcul des transitions
df_audit = df_tracks_stable.sort_values(
    ["track_id", "frame"]
).copy()

df_audit["classe_precedente"] = (
    df_audit.groupby("track_id")["class"].shift(1)
)

df_audit["transition"] = (
    df_audit["classe_precedente"].notna()
    & (
        df_audit["class"] != df_audit["classe_precedente"]
    )
)

transitions_finales = (
    df_audit.groupby("track_id")["transition"]
    .sum()
    .astype(int)
)

audit_corrected["transitions_classe"] = (
    audit_corrected["track_id"]
    .map(transitions_finales)
)

print("========== TRANSITIONS CORRIGEES ==========")

display(
    audit_corrected[
        [
            "track_id",
            "observations",
            "nombre_classes",
            "part_classe_dominante",
            "transitions_classe",
            "suspecte"
        ]
    ].sort_values(
        ["suspecte", "transitions_classe"],
        ascending=[False, False]
    )
)

# Étape 03 : clôture et sauvegarde des résultats

In [ ]:
# ============================================================
# ETAPE 03.9 — EXPORT DES RESULTATS DU TRACKING
# ============================================================

from pathlib import Path
import json

OUTPUT_TRACKING = Path("/kaggle/working/ADAS_FUSION_X2030/03_tracking")
OUTPUT_TRACKING.mkdir(parents=True, exist_ok=True)

# Données brutes et données stabilisées
df_tracks_bbox.to_csv(
    OUTPUT_TRACKING / "tracking_raw.csv",
    index=False
)

df_tracks_stable.to_csv(
    OUTPUT_TRACKING / "tracking_stabilized.csv",
    index=False
)

# Audit final
audit_corrected.to_csv(
    OUTPUT_TRACKING / "tracking_audit.csv",
    index=False
)

# Métadonnées
metadata = {
    "detector": "YOLO26n",
    "tracker": "ByteTrack",
    "video_duration_processed_s": 15,
    "video_fps": 24,
    "frames_processed": 360,
    "total_observations": 1052,
    "unique_tracks": 31,
    "processing_fps": 27.93,
    "processing_time_s": 12.89,
    "suspicious_tracks": 6,
    "stable_tracks": 25,
    "classification_stability_threshold": 0.95,
    "ground_truth_available": False
}

with open(
    OUTPUT_TRACKING / "tracking_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("========== EXPORT TRACKING ==========")

for file in OUTPUT_TRACKING.iterdir():
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

print("\nVALIDATION : SUCCESS")

# Étape 04 — Segmentation sémantique de la scène routière

## 04.1 — Préparation de l'environnement

In [ ]:
# ============================================================
# ADAS-FUSION X 2030
# ETAPE 04.1 — ENVIRONNEMENT SEGFORMER
# ============================================================

import sys
import torch
import cv2
import numpy as np

print("========== ENVIRONNEMENT SEGMENTATION ==========")

print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print(
        "Mémoire GPU :",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3, 2
        ),
        "GB"
    )

print("OpenCV :", cv2.__version__)
print("NumPy :", np.__version__)

In [ ]:
!pip -q install transformers

## 04.3 — Chargement du modèle SegFormer-B0

In [ ]:
# ============================================================
# ETAPE 04.3 — CHARGEMENT DU MODELE
# ============================================================

from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation
)

MODEL_SEGMENTATION = (
    "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
)

processor = SegformerImageProcessor.from_pretrained(
    MODEL_SEGMENTATION
)

segformer = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_SEGMENTATION
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

segformer = segformer.to(DEVICE)
segformer.eval()

print("========== MODELE SEGFORMER ==========")
print("Modèle :", MODEL_SEGMENTATION)
print("Device :", DEVICE)
print("Nombre de classes :", segformer.config.num_labels)

for class_id, class_name in segformer.config.id2label.items():
    print(class_id, ":", class_name)

## Étape 04.4 — Première inférence sur notre vidéo

In [ ]:
# ============================================================
# ADAS-FUSION X 2030
# ETAPE 04.4 — PREMIERE INFERENCE SEGFORMER
# ============================================================

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import time

# ------------------------------------------------------------
# 1. EXTRACTION D'UNE IMAGE
# ------------------------------------------------------------

TEST_TIME = 30

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_MSEC, TEST_TIME * 1000)

success, frame = cap.read()
cap.release()

if not success:
    raise RuntimeError("Impossible de lire l'image de test.")

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

print("Image :", frame_rgb.shape)
print("Temps vidéo :", TEST_TIME, "secondes")

# ------------------------------------------------------------
# 2. PRETRAITEMENT
# ------------------------------------------------------------

inputs = processor(
    images=frame_rgb,
    return_tensors="pt"
)

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

# ------------------------------------------------------------
# 3. INFERENCE GPU
# ------------------------------------------------------------

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

start = time.perf_counter()

with torch.inference_mode():
    outputs = segformer(**inputs)

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

inference_time = time.perf_counter() - start

# ------------------------------------------------------------
# 4. RECONSTRUCTION DU MASQUE
# ------------------------------------------------------------

logits = torch.nn.functional.interpolate(
    outputs.logits,
    size=frame_rgb.shape[:2],
    mode="bilinear",
    align_corners=False
)

mask = logits.argmax(dim=1)[0].cpu().numpy()

print("Dimensions du masque :", mask.shape)
print("Temps d'inférence :", round(inference_time, 4), "s")
print("FPS théorique segmentation :",
      round(1 / inference_time, 2))

# ------------------------------------------------------------
# 5. DISTRIBUTION DES CLASSES
# ------------------------------------------------------------

class_counts = np.bincount(
    mask.flatten(),
    minlength=19
)

total_pixels = mask.size

print("\n========== CLASSES DETECTEES ==========")

for class_id, count in enumerate(class_counts):
    if count > 0:
        percentage = 100 * count / total_pixels

        print(
            f"{segformer.config.id2label[class_id]:15s} : "
            f"{percentage:.2f}%"
        )

# ------------------------------------------------------------
# 6. CALCUL DE LA PROPORTION DE CHAUSSEE
# ------------------------------------------------------------

road_ratio = 100 * np.mean(mask == 0)
sidewalk_ratio = 100 * np.mean(mask == 1)

print("\nProportion chaussée :", round(road_ratio, 2), "%")
print("Proportion trottoir :", round(sidewalk_ratio, 2), "%")

# ------------------------------------------------------------
# 7. VISUALISATION
# ------------------------------------------------------------

plt.figure(figsize=(18, 7))

plt.subplot(1, 3, 1)
plt.imshow(frame_rgb)
plt.title("Image originale")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask, cmap="tab20", vmin=0, vmax=19)
plt.title("Masque sémantique")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(frame_rgb)
plt.imshow(mask, cmap="tab20", alpha=0.45,
           vmin=0, vmax=19)
plt.title("Superposition")
plt.axis("off")

plt.tight_layout()
plt.show()

## Étape 04.5 — Inspection visuelle et validation du masque

In [ ]:
# ============================================================
# ETAPE 04.5 — VALIDATION VISUELLE SEGFORMER
# ============================================================

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

OUTPUT_SEG = Path(
    "/kaggle/working/ADAS_FUSION_X2030/04_segmentation"
)
OUTPUT_SEG.mkdir(parents=True, exist_ok=True)

# Palette Cityscapes RGB
CITYSCAPES_COLORS = np.array([
    [128,  64, 128],  # road
    [244,  35, 232],  # sidewalk
    [ 70,  70,  70],  # building
    [102, 102, 156],  # wall
    [190, 153, 153],  # fence
    [153, 153, 153],  # pole
    [250, 170,  30],  # traffic light
    [220, 220,   0],  # traffic sign
    [107, 142,  35],  # vegetation
    [152, 251, 152],  # terrain
    [ 70, 130, 180],  # sky
    [220,  20,  60],  # person
    [255,   0,   0],  # rider
    [  0,   0, 142],  # car
    [  0,   0,  70],  # truck
    [  0,  60, 100],  # bus
    [  0,  80, 100],  # train
    [  0,   0, 230],  # motorcycle
    [119,  11,  32]   # bicycle
], dtype=np.uint8)

# Construction de l'image colorée
colored_mask = CITYSCAPES_COLORS[mask]

# Superposition
overlay = cv2.addWeighted(
    frame_rgb, 0.55,
    colored_mask, 0.45,
    0
)

# Contours des régions segmentées
boundaries = np.zeros(mask.shape, dtype=np.uint8)

for y in range(1, mask.shape[0]):
    boundaries[y] = (
        mask[y] != mask[y - 1]
    ).astype(np.uint8) * 255

overlay_with_boundaries = overlay.copy()
overlay_with_boundaries[boundaries > 0] = [255, 255, 255]

# Export des images
cv2.imwrite(
    str(OUTPUT_SEG / "original_30s.png"),
    cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    str(OUTPUT_SEG / "mask_30s.png"),
    cv2.cvtColor(colored_mask, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    str(OUTPUT_SEG / "overlay_30s.png"),
    cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    str(OUTPUT_SEG / "overlay_boundaries_30s.png"),
    cv2.cvtColor(overlay_with_boundaries, cv2.COLOR_RGB2BGR)
)

# Tableau de distribution
class_stats = pd.DataFrame({
    "class_id": range(19),
    "class_name": [
        segformer.config.id2label[i] for i in range(19)
    ],
    "pixel_count": [
        int(np.sum(mask == i)) for i in range(19)
    ]
})

class_stats["percentage"] = (
    100 * class_stats["pixel_count"] / mask.size
)

class_stats.to_csv(
    OUTPUT_SEG / "segmentation_statistics_30s.csv",
    index=False
)

# Visualisation
fig, axes = plt.subplots(1, 4, figsize=(24, 7))

images = [
    frame_rgb,
    colored_mask,
    overlay,
    overlay_with_boundaries
]

titles = [
    "Image originale",
    "Masque sémantique",
    "Superposition",
    "Contours des régions"
]

for ax, image, title in zip(axes, images, titles):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("========== EXPORT SEGMENTATION ==========")

for file in OUTPUT_SEG.iterdir():
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

In [ ]:
from IPython.display import display, Image
from pathlib import Path

folder = Path(
    "/kaggle/working/ADAS_FUSION_X2030/04_segmentation"
)

display(Image(filename=str(folder / "overlay_30s.png")))
display(Image(filename=str(folder / "overlay_boundaries_30s.png")))

## Étape 04.6 — Segmentation de la vidéo complète  

### Cellule 04.6 — Traitement vidéo

In [ ]:
# ============================================================
# ADAS-FUSION X 2030
# ETAPE 04.6 — SEGMENTATION VIDEO COMPLETE
# ============================================================

import cv2
import torch
import numpy as np
import pandas as pd
import time
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

OUTPUT_SEG = Path(
    "/kaggle/working/ADAS_FUSION_X2030/04_segmentation"
)
OUTPUT_SEG.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_SEG / "ADAS_Segmentation_180s.mp4"
OUTPUT_CSV = OUTPUT_SEG / "segmentation_statistics_full.csv"

TARGET_FPS = 4
INFERENCE_SIZE = (1024, 576)

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir la vidéo.")

original_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / original_fps

frame_interval = max(1, round(original_fps / TARGET_FPS))
expected_frames = int(np.ceil(total_frames / frame_interval))

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("========== CONFIGURATION ==========")
print("Durée :", duration, "secondes")
print("FPS original :", original_fps)
print("Images originales :", total_frames)
print("Résolution :", width, "x", height)
print("Intervalle :", frame_interval)
print("Images à traiter :", expected_frames)

# ------------------------------------------------------------
# 2. INITIALISATION VIDEO
# ------------------------------------------------------------

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    fourcc,
    TARGET_FPS,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Impossible de créer la vidéo de sortie.")

# ------------------------------------------------------------
# 3. PALETTE CITYSCAPES
# ------------------------------------------------------------

palette = CITYSCAPES_COLORS

class_names = [
    segformer.config.id2label[i]
    for i in range(19)
]

global_counts = np.zeros(19, dtype=np.int64)
records = []

processed = 0
start_time = time.perf_counter()

# ------------------------------------------------------------
# 4. TRAITEMENT
# ------------------------------------------------------------

pbar = tqdm(total=expected_frames, desc="Segmentation")

for frame_index in range(0, total_frames, frame_interval):

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    success, frame_bgr = cap.read()

    if not success:
        continue

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    # Réduction de résolution pour accélérer l'inférence
    frame_small = cv2.resize(
        frame_rgb,
        INFERENCE_SIZE,
        interpolation=cv2.INTER_AREA
    )

    inputs = processor(
        images=frame_small,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = segformer(**inputs)

    logits = torch.nn.functional.interpolate(
        outputs.logits,
        size=(height, width),
        mode="bilinear",
        align_corners=False
    )

    mask = (
        logits.argmax(dim=1)[0]
        .cpu()
        .numpy()
        .astype(np.uint8)
    )

    # Comptage des classes
    counts = np.bincount(
        mask.ravel(),
        minlength=19
    )

    global_counts += counts

    # Création du masque coloré
    colored_mask = palette[mask]

    # Superposition
    overlay = cv2.addWeighted(
        frame_rgb,
        0.55,
        colored_mask,
        0.45,
        0
    )

    # Informations affichées
    timestamp = frame_index / original_fps

    cv2.putText(
        overlay,
        f"ADAS-FUSION X2030 | {timestamp:.2f} s",
        (35, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.1,
        (255, 255, 255),
        2
    )

    # Ecriture vidéo
    writer.write(
        cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    )

    records.append({
        "frame_index": frame_index,
        "timestamp_s": timestamp,
        "road_percent": 100 * counts[0] / mask.size,
        "sidewalk_percent": 100 * counts[1] / mask.size,
        "sky_percent": 100 * counts[10] / mask.size,
        "car_percent": 100 * counts[13] / mask.size,
        "truck_percent": 100 * counts[14] / mask.size,
        "person_percent": 100 * counts[11] / mask.size
    })

    processed += 1
    pbar.update(1)

pbar.close()

# ------------------------------------------------------------
# 5. LIBERATION RESSOURCES
# ------------------------------------------------------------

cap.release()
writer.release()

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start_time

# ------------------------------------------------------------
# 6. EXPORT STATISTIQUES
# ------------------------------------------------------------

df_segmentation = pd.DataFrame(records)

df_segmentation.to_csv(
    OUTPUT_CSV,
    index=False
)

total_pixels = global_counts.sum()

class_summary = pd.DataFrame({
    "class": class_names,
    "pixel_count": global_counts,
    "percentage": (
        100 * global_counts / total_pixels
    )
})

class_summary.to_csv(
    OUTPUT_SEG / "class_distribution_full.csv",
    index=False
)

# ------------------------------------------------------------
# 7. RAPPORT
# ------------------------------------------------------------

print("\n========== RAPPORT SEGMENTATION ==========")
print("Images traitées :", processed)
print("Durée vidéo :", round(duration, 2), "s")
print("Temps de traitement :", round(elapsed, 2), "s")
print("Débit réel :", round(processed / elapsed, 2), "FPS")
print("FPS vidéo export :", TARGET_FPS)

print("\nDistribution globale des classes :")
print(
    class_summary.sort_values(
        "percentage",
        ascending=False
    ).to_string(index=False)
)

print("\nFichiers exportés :")
for path in [
    OUTPUT_VIDEO,
    OUTPUT_CSV,
    OUTPUT_SEG / "class_distribution_full.csv"
]:
    print(path.name, "-", round(path.stat().st_size / 1024**2, 2), "MB")

print("\nVALIDATION :", "SUCCESS" if processed > 0 else "FAILED")

In [ ]:
from pathlib import Path
import cv2
import pandas as pd

folder = Path(
    "/kaggle/working/ADAS_FUSION_X2030/04_segmentation"
)

video = folder / "ADAS_Segmentation_180s.mp4"

cap = cv2.VideoCapture(str(video))

print("========== VALIDATION EXPORT ==========")
print("Vidéo existe :", video.exists())

if cap.isOpened():
    print("FPS export :", cap.get(cv2.CAP_PROP_FPS))
    print("Images export :", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    print(
        "Résolution :",
        int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "x",
        int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    )

    ret, test_frame = cap.read()
    print("Première image lisible :", ret)
    cap.release()

csv_path = folder / "segmentation_statistics_full.csv"
class_path = folder / "class_distribution_full.csv"

df = pd.read_csv(csv_path)
classes = pd.read_csv(class_path)

print("\nImages dans CSV :", len(df))
print("\nDistribution des classes :")
print(classes.sort_values("percentage", ascending=False).to_string(index=False))

print("\nFichiers :")
for f in folder.iterdir():
    if f.is_file():
        print(f.name, round(f.stat().st_size / 1024**2, 2), "MB")

# Étape 05 — Estimation de profondeur monoculaire

## 2. Étape 05.1 — Installation et chargement du modèle

In [ ]:
# ============================================================
# ADAS-FUSION X 2030
# ETAPE 05.1 — INITIALISATION DEPTH ANYTHING V2
# ============================================================

import torch
import transformers
import time

from transformers import (
    AutoImageProcessor,
    AutoModelForDepthEstimation
)

MODEL_DEPTH = "depth-anything/Depth-Anything-V2-Small-hf"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("========== ENVIRONNEMENT DEPTH ESTIMATION ==========")
print("PyTorch :", torch.__version__)
print("Transformers :", transformers.__version__)
print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

# Chargement du préprocesseur
processor_depth = AutoImageProcessor.from_pretrained(
    MODEL_DEPTH
)

# Chargement du modèle
start = time.perf_counter()

depth_model = AutoModelForDepthEstimation.from_pretrained(
    MODEL_DEPTH
).to(DEVICE)

depth_model.eval()

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

loading_time = time.perf_counter() - start

print("\n========== MODELE CHARGE ==========")
print("Modèle :", MODEL_DEPTH)
print("Device :", DEVICE)
print("Temps de chargement :", round(loading_time, 2), "s")
print("Nombre de paramètres :",
      sum(p.numel() for p in depth_model.parameters()))
print("VALIDATION : SUCCESS")

## Étape 05.2 — Inférence sur l'image à 30 secondes

In [ ]:
# ============================================================
# ADAS-FUSION X 2030
# ETAPE 05.2 — ESTIMATION DE PROFONDEUR MONOCULAIRE
# ============================================================

import cv2
import numpy as np
import torch
import time
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

OUTPUT_DEPTH = Path(
    "/kaggle/working/ADAS_FUSION_X2030/05_depth"
)
OUTPUT_DEPTH.mkdir(parents=True, exist_ok=True)

TEST_TIME = 30

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_MSEC, TEST_TIME * 1000)

success, frame_bgr = cap.read()
cap.release()

if not success:
    raise RuntimeError(
        "Impossible de lire l'image à 30 secondes."
    )

frame_rgb = cv2.cvtColor(
    frame_bgr,
    cv2.COLOR_BGR2RGB
)

image_pil = Image.fromarray(frame_rgb)

print("Image :", frame_rgb.shape)
print("Instant vidéo :", TEST_TIME, "s")

# ------------------------------------------------------------
# 2. INFERENCE
# ------------------------------------------------------------

inputs = processor_depth(
    images=image_pil,
    return_tensors="pt"
)

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

start = time.perf_counter()

with torch.inference_mode():
    outputs = depth_model(**inputs)

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

inference_time = time.perf_counter() - start

# ------------------------------------------------------------
# 3. RECONSTRUCTION DE LA CARTE DE PROFONDEUR
# ------------------------------------------------------------

predicted_depth = outputs.predicted_depth

depth_map = torch.nn.functional.interpolate(
    predicted_depth.unsqueeze(1),
    size=frame_rgb.shape[:2],
    mode="bicubic",
    align_corners=False
).squeeze().cpu().numpy()

depth_map = np.nan_to_num(
    depth_map,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

# Normalisation pour visualisation uniquement
depth_min = float(depth_map.min())
depth_max = float(depth_map.max())

if depth_max > depth_min:
    depth_normalized = (
        (depth_map - depth_min)
        / (depth_max - depth_min)
    )
else:
    depth_normalized = np.zeros_like(depth_map)

depth_uint8 = (
    depth_normalized * 255
).astype(np.uint8)

# ------------------------------------------------------------
# 4. EXPORT
# ------------------------------------------------------------

np.save(
    OUTPUT_DEPTH / "depth_map_30s.npy",
    depth_map.astype(np.float32)
)

cv2.imwrite(
    str(OUTPUT_DEPTH / "depth_map_30s.png"),
    depth_uint8
)

# ------------------------------------------------------------
# 5. INDICATEURS NUMERIQUES
# ------------------------------------------------------------

print("\n========== RESULTATS DEPTH ESTIMATION ==========")
print("Dimensions :", depth_map.shape)
print("Temps d'inférence :", round(inference_time, 4), "s")
print("FPS théorique :", round(1 / inference_time, 2))
print("Minimum brut :", round(depth_min, 5))
print("Maximum brut :", round(depth_max, 5))
print("Moyenne brute :", round(float(depth_map.mean()), 5))
print("Écart-type brut :", round(float(depth_map.std()), 5))
print("Valeurs non finies après nettoyage :",
      int(np.sum(~np.isfinite(depth_map))))

print("\nVALIDATION :", "SUCCESS" if depth_map.size > 0 else "FAILED")

# ------------------------------------------------------------
# 6. VISUALISATION
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].imshow(frame_rgb)
axes[0].set_title("Image originale — 30 s")

axes[1].imshow(depth_map, cmap="inferno")
axes[1].set_title("Profondeur estimée — valeurs brutes")

axes[2].imshow(depth_normalized, cmap="inferno")
axes[2].set_title("Profondeur normalisée pour visualisation")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("\nFichiers exportés :")
for path in OUTPUT_DEPTH.iterdir():
    if path.is_file():
        print(
            path.name,
            "-",
            round(path.stat().st_size / 1024, 2),
            "KB"
        )

## Étape 05.4 — Vérifier numériquement la profondeur du véhicule

In [ ]:
# Étape 05.4 — Vérification ciblée de la profondeur
# Aucune distance métrique n'est calculée.

import numpy as np
import matplotlib.pyplot as plt
import cv2

# Vérifier la disponibilité des variables issues des étapes précédentes
required = ["depth_map", "frame"]

missing = [name for name in required if name not in globals()]

if missing:
    print("Variables manquantes :", missing)
    print("Réexécute la cellule d'inférence de l'étape 05.2.")
else:
    depth = np.asarray(depth_map, dtype=np.float32)
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    h, w = depth.shape

    # Régions de contrôle dans la partie basse de l'image.
    # Elles sont indicatives : il faudra vérifier leur position visuellement.
    regions = {
        "Route proche": (0.70, 0.90, 0.40, 0.75),
        "Route intermédiaire": (0.60, 0.75, 0.40, 0.75),
        "Route éloignée": (0.48, 0.60, 0.40, 0.75),
    }

    print("========== CONTRÔLE DES RÉGIONS ==========")

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    axes[0].imshow(image)
    axes[0].set_title("Image originale et régions de contrôle")
    axes[0].axis("off")

    axes[1].imshow(depth, cmap="inferno")
    axes[1].set_title("Carte de profondeur brute")
    axes[1].axis("off")

    for name, (y1, y2, x1, x2) in regions.items():
        xa, xb = int(x1 * w), int(x2 * w)
        ya, yb = int(y1 * h), int(y2 * h)

        crop = depth[ya:yb, xa:xb]
        valid = crop[np.isfinite(crop)]

        if valid.size == 0:
            print(f"{name} : aucune valeur finie")
            continue

        print(
            f"{name} | "
            f"médiane={np.median(valid):.4f} | "
            f"moyenne={np.mean(valid):.4f} | "
            f"écart-type={np.std(valid):.4f}"
        )

        rect = plt.Rectangle(
            (xa, ya), xb - xa, yb - ya,
            fill=False, edgecolor="cyan", linewidth=2
        )
        axes[0].add_patch(rect)
        axes[1].add_patch(
            plt.Rectangle(
                (xa, ya), xb - xa, yb - ya,
                fill=False, edgecolor="cyan", linewidth=2
            )
        )

    plt.tight_layout()
    plt.show()

## Étape 05.5 — Vérifier l'orientation de la profondeur

In [ ]:
# ADAS-FUSION X 2030 — Étape 05.5
# Analyse de la distribution de profondeur par bandes horizontales

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

depth = np.asarray(depth_map, dtype=np.float32)

if depth.ndim != 2:
    raise ValueError(f"Carte attendue en 2D, forme reçue : {depth.shape}")

valid = np.isfinite(depth)

# Bandes verticales couvrant la hauteur de l'image
bands = [
    ("Haut", 0.00, 0.20),
    ("Supérieur", 0.20, 0.40),
    ("Central", 0.40, 0.60),
    ("Inférieur", 0.60, 0.80),
    ("Bas", 0.80, 1.00),
]

h, w = depth.shape
rows = []

for name, y_start, y_end in bands:
    y1 = int(y_start * h)
    y2 = int(y_end * h)

    crop = depth[y1:y2, :]
    crop_valid = crop[np.isfinite(crop)]

    rows.append({
        "Bande": name,
        "Position début (%)": int(y_start * 100),
        "Position fin (%)": int(y_end * 100),
        "Pixels valides (%)": round(
            100 * crop_valid.size / crop.size, 2
        ),
        "Médiane brute": float(np.median(crop_valid)),
        "Moyenne brute": float(np.mean(crop_valid)),
        "Écart-type": float(np.std(crop_valid)),
        "P10": float(np.percentile(crop_valid, 10)),
        "P90": float(np.percentile(crop_valid, 90)),
    })

df_depth_bands = pd.DataFrame(rows)

display(df_depth_bands.round(4))

plt.figure(figsize=(10, 5))

plt.plot(
    df_depth_bands["Position début (%)"] + 10,
    df_depth_bands["Médiane brute"],
    marker="o",
    label="Médiane"
)

plt.xlabel("Position verticale dans l'image (%)")
plt.ylabel("Valeur brute prédite")
plt.title("Distribution de profondeur par bande horizontale")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### Corrigeons l'affichage avant de conclure

In [ ]:
# ADAS-FUSION X 2030 — Vérification du tableau de profondeur

print("Colonnes disponibles :")
print(df_depth_bands.columns.tolist())

display(
    df_depth_bands[
        [
            "Bande",
            "Position début (%)",
            "Position fin (%)",
            "Pixels valides (%)",
            "Médiane brute",
            "Moyenne brute",
            "Écart-type",
            "P10",
            "P90",
        ]
    ].round(4)
)

## Étape 05.6 — Traitement temporel de la profondeur sur la vidéo

### Première cellule — Préparer le traitement

In [ ]:
# ============================================================
# ADAS-FUSION X 2030 — Étape 05.6.1
# Préparation du traitement temporel de la profondeur
# ============================================================

from pathlib import Path
import cv2
import torch
import numpy as np
import pandas as pd
import time

# Vérification des variables et modèles nécessaires
assert "VIDEO_PATH" in globals(), "VIDEO_PATH est introuvable."
assert "processor_depth" in globals(), "Le processeur de profondeur est introuvable."
assert "depth_model" in globals(), "Le modèle de profondeur est introuvable."
assert "DEVICE" in globals(), "DEVICE est introuvable."

video_path = Path(VIDEO_PATH)

assert video_path.exists(), f"Vidéo introuvable : {video_path}"

# Paramètres de l'expérience
SAMPLE_FPS = 2
OUTPUT_DIR = Path("/kaggle/working/ADAS_FUSION_X2030/05_depth")
RAW_DIR = OUTPUT_DIR / "raw_maps"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Inspection du fichier vidéo
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError(f"Impossible d'ouvrir la vidéo : {video_path}")

video_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = total_frames / video_fps if video_fps > 0 else 0

cap.release()

print("========== PRÉPARATION DU TRAITEMENT ==========")
print(f"Vidéo              : {video_path}")
print(f"Résolution         : {width} x {height}")
print(f"FPS source         : {video_fps:.3f}")
print(f"Nombre d'images    : {total_frames}")
print(f"Durée              : {duration:.2f} s")
print(f"Échantillonnage    : {SAMPLE_FPS} image(s)/s")
print(f"Images prévues     : environ {int(duration * SAMPLE_FPS)}")
print(f"Appareil           : {DEVICE}")
print(f"Dossier de sortie  : {OUTPUT_DIR}")
print(f"CUDA disponible    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU                : {torch.cuda.get_device_name(0)}")

print("\nVALIDATION : PRÉPARATION TERMINÉE")

## Étape 05.6.2 — Traitement temporel de la profondeur

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — Étape 05.6.2
# Estimation temporelle de la profondeur monoculaire
# Modèle : Depth Anything V2 Small
# GPU cible : NVIDIA Tesla T4
# ============================================================

from pathlib import Path
import time
import gc
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SAMPLE_FPS = 2

VIDEO_PATH = str(VIDEO_PATH)

OUTPUT_DIR = Path(
    "/kaggle/working/ADAS_FUSION_X2030/05_depth"
)
RAW_DIR = OUTPUT_DIR / "raw_maps"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Nettoyage du cache CUDA avant traitement
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 2. Ouverture de la vidéo
# ------------------------------------------------------------

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Impossible d'ouvrir la vidéo : {VIDEO_PATH}")

source_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if source_fps <= 0 or total_frames <= 0:
    cap.release()
    raise RuntimeError("Métadonnées vidéo invalides.")

duration_s = total_frames / source_fps

# Pour 24 FPS et un échantillonnage de 2 FPS : intervalle de 12 images.
frame_step = max(1, round(source_fps / SAMPLE_FPS))

expected_samples = len(range(0, total_frames, frame_step))

print("========== CONFIGURATION ==========")
print(f"FPS source             : {source_fps:.2f}")
print(f"Images source          : {total_frames}")
print(f"Durée                  : {duration_s:.2f} s")
print(f"Intervalle             : {frame_step} images")
print(f"Échantillons prévus    : {expected_samples}")
print(f"Appareil               : {DEVICE}")
print(f"Sortie                 : {OUTPUT_DIR}")
print("===================================")

# ------------------------------------------------------------
# 3. Inférence image par image
# ------------------------------------------------------------

records = []
processed = 0
failed = 0
inference_times = []

start_total = time.perf_counter()

try:
    for frame_idx in range(0, total_frames, frame_step):

        # Accès direct à l'image demandée
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame_bgr = cap.read()

        if not success:
            failed += 1
            print(f"[AVERTISSEMENT] Lecture impossible : image {frame_idx}")
            continue

        timestamp_s = frame_idx / source_fps

        # BGR -> RGB -> PIL
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)

        # Prétraitement
        inputs = processor_depth(
            images=pil_image,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        # Inférence
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_inference = time.perf_counter()

        with torch.inference_mode():
            outputs = depth_model(**inputs)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        inference_s = time.perf_counter() - start_inference
        inference_times.append(inference_s)

        # Mise à l'échelle vers la résolution de la vidéo
        predicted_depth = torch.nn.functional.interpolate(
            outputs.predicted_depth.unsqueeze(1),
            size=(frame_rgb.shape[0], frame_rgb.shape[1]),
            mode="bicubic",
            align_corners=False
        ).squeeze().float().cpu().numpy()

        # Sécurisation des valeurs non finies
        predicted_depth = np.nan_to_num(
            predicted_depth,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        ).astype(np.float32)

        # Conservation de la carte brute
        map_filename = f"depth_{frame_idx:06d}.npy"
        map_path = RAW_DIR / map_filename
        np.save(map_path, predicted_depth)

        # Statistiques descriptives
        records.append({
            "frame_index": int(frame_idx),
            "timestamp_s": float(timestamp_s),
            "map_file": map_filename,
            "height": int(predicted_depth.shape[0]),
            "width": int(predicted_depth.shape[1]),
            "inference_time_s": float(inference_s),
            "inference_fps": float(1.0 / inference_s)
                if inference_s > 0 else np.nan,
            "depth_min": float(np.min(predicted_depth)),
            "depth_max": float(np.max(predicted_depth)),
            "depth_mean": float(np.mean(predicted_depth)),
            "depth_median": float(np.median(predicted_depth)),
            "depth_std": float(np.std(predicted_depth)),
            "nonfinite_after_cleaning": int(
                np.count_nonzero(~np.isfinite(predicted_depth))
            )
        })

        processed += 1

        # Libération des objets volumineux à chaque itération
        del inputs, outputs, predicted_depth, pil_image, frame_rgb, frame_bgr

        if processed % 30 == 0 or processed == expected_samples:
            elapsed = time.perf_counter() - start_total
            rate = processed / elapsed if elapsed > 0 else 0
            print(
                f"Progression : {processed}/{expected_samples} | "
                f"échecs : {failed} | "
                f"temps : {elapsed:.1f} s | "
                f"débit global : {rate:.2f} images/s"
            )

finally:
    cap.release()
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Export des résultats
# ------------------------------------------------------------

total_elapsed = time.perf_counter() - start_total

df_depth = pd.DataFrame(records)

csv_path = OUTPUT_DIR / "depth_temporal_statistics.csv"
df_depth.to_csv(csv_path, index=False)

# Résumé de l'expérience
summary = {
    "video_path": VIDEO_PATH,
    "source_fps": float(source_fps),
    "source_frames": int(total_frames),
    "duration_s": float(duration_s),
    "sample_fps_target": float(SAMPLE_FPS),
    "frame_step": int(frame_step),
    "samples_expected": int(expected_samples),
    "samples_processed": int(processed),
    "samples_failed": int(failed),
    "processing_time_s": float(total_elapsed),
    "global_processing_fps": float(processed / total_elapsed)
        if total_elapsed > 0 else np.nan,
    "mean_inference_time_s": float(np.mean(inference_times))
        if inference_times else np.nan,
    "median_inference_time_s": float(np.median(inference_times))
        if inference_times else np.nan,
    "p95_inference_time_s": float(np.percentile(inference_times, 95))
        if inference_times else np.nan,
    "depth_maps_directory": str(RAW_DIR),
    "statistics_csv": str(csv_path),
    "depth_units": "relative model output; not meters",
    "ground_truth_available": False
}

summary_path = OUTPUT_DIR / "depth_temporal_metadata.json"

import json

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 5. Rapport final
# ------------------------------------------------------------

print("\n========== RAPPORT FINAL ==========")
print(f"Images traitées             : {processed}/{expected_samples}")
print(f"Échecs de lecture           : {failed}")
print(f"Temps total                 : {total_elapsed:.2f} s")
print(f"Débit global                : {summary['global_processing_fps']:.2f} images/s")
print(f"Inférence moyenne           : {summary['mean_inference_time_s']:.4f} s")
print(f"Inférence médiane           : {summary['median_inference_time_s']:.4f} s")
print(f"P95 inférence               : {summary['p95_inference_time_s']:.4f} s")
print(f"CSV                         : {csv_path}")
print(f"Métadonnées                 : {summary_path}")
print(f"Cartes brutes               : {RAW_DIR}")

if not df_depth.empty:
    print("\nAperçu des résultats :")
    display(df_depth.head())

    print("\nStatistiques globales des sorties :")
    display(
        df_depth[
            [
                "depth_min",
                "depth_max",
                "depth_mean",
                "depth_median",
                "depth_std",
                "inference_time_s"
            ]
        ].describe().round(4)
    )

print("\nUNITÉS : valeurs de profondeur relatives, non métriques.")
print("VALIDATION :", "SUCCESS" if processed > 0 and failed == 0 else "À EXAMINER")


## Étape 05.7 — Vérifier la stabilité temporelle

### Cellule Kaggle — Étape 05.7.1

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — Étape 05.7.1
# Diagnostic de stabilité temporelle des sorties de profondeur
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path(
    "/kaggle/working/ADAS_FUSION_X2030/05_depth"
)

CSV_PATH = OUTPUT_DIR / "depth_temporal_statistics.csv"

assert CSV_PATH.exists(), f"CSV introuvable : {CSV_PATH}"

df = pd.read_csv(CSV_PATH)
df = df.sort_values("timestamp_s").reset_index(drop=True)

required_columns = [
    "timestamp_s",
    "depth_mean",
    "depth_median",
    "depth_std",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Colonnes manquantes : {missing}")

# Variation absolue de la médiane entre échantillons successifs
df["median_abs_change"] = df["depth_median"].diff().abs()

print("========== DIAGNOSTIC TEMPOREL ==========")
print(f"Nombre d'échantillons : {len(df)}")
print(f"Durée couverte        : {df['timestamp_s'].max():.2f} s")

print("\nStatistiques des valeurs entre images :")
display(
    df[
        [
            "depth_mean",
            "depth_median",
            "depth_std",
            "median_abs_change",
        ]
    ].describe(percentiles=[0.50, 0.90, 0.95, 0.99]).round(4)
)

# ------------------------------------------------------------
# Graphique 1 : moyenne et médiane
# ------------------------------------------------------------

plt.figure(figsize=(14, 5))

plt.plot(
    df["timestamp_s"],
    df["depth_mean"],
    label="Moyenne brute",
    linewidth=1.2,
)

plt.plot(
    df["timestamp_s"],
    df["depth_median"],
    label="Médiane brute",
    linewidth=1.2,
)

plt.xlabel("Temps vidéo (s)")
plt.ylabel("Valeur brute prédite")
plt.title("Évolution temporelle des statistiques de profondeur")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Graphique 2 : variations de la médiane
# ------------------------------------------------------------

plt.figure(figsize=(14, 4))

plt.plot(
    df["timestamp_s"],
    df["median_abs_change"],
    linewidth=1,
)

plt.xlabel("Temps vidéo (s)")
plt.ylabel("Variation absolue de la médiane")
plt.title("Variations entre deux estimations successives")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Export du tableau enrichi
# ------------------------------------------------------------

OUTPUT_CSV = OUTPUT_DIR / "depth_temporal_stability.csv"
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nCSV de stabilité enregistré : {OUTPUT_CSV}")
print("UNITÉS : valeurs de modèle relatives, non métriques.")
print("Aucune précision de distance n'est déduite de ce diagnostic.")


## Étape 05.8 — Audit visuel des ruptures

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — Étape 05.8
# Audit visuel des ruptures temporelles de profondeur
# ============================================================

from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VIDEO_PATH = Path(
    "/kaggle/working/ADAS_FUSION_X2030/videos/"
    "ADAS_FUSION_X2030_Reference_180s.mp4"
)

RAW_DIR = Path(
    "/kaggle/working/ADAS_FUSION_X2030/05_depth/raw_maps"
)

CSV_PATH = Path(
    "/kaggle/working/ADAS_FUSION_X2030/05_depth/"
    "depth_temporal_statistics.csv"
)

assert VIDEO_PATH.exists(), f"Vidéo introuvable : {VIDEO_PATH}"
assert CSV_PATH.exists(), f"CSV introuvable : {CSV_PATH}"

df = pd.read_csv(CSV_PATH)
cap = cv2.VideoCapture(str(VIDEO_PATH))

fps = cap.get(cv2.CAP_PROP_FPS)

# Instants autour des ruptures observées
timestamps = [88, 90, 92, 100, 102, 174]

fig, axes = plt.subplots(
    len(timestamps), 2,
    figsize=(14, 4 * len(timestamps))
)

audit_rows = []

for row, timestamp in enumerate(timestamps):

    # Associer l'instant demandé à l'échantillon traité le plus proche
    idx = (df["timestamp_s"] - timestamp).abs().idxmin()
    record = df.loc[idx]

    actual_time = float(record["timestamp_s"])
    frame_index = int(record["frame_index"])

    # Lecture de l'image source
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, frame_bgr = cap.read()

    if not ok:
        axes[row, 0].text(
            0.5, 0.5, "Lecture impossible",
            ha="center", va="center"
        )
        axes[row, 0].axis("off")
        axes[row, 1].axis("off")
        continue

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    # Charger la carte déjà calculée
    depth_path = RAW_DIR / str(record["map_file"])
    depth = np.load(depth_path)

    # Normalisation locale pour visualisation uniquement
    lo, hi = np.percentile(depth, [2, 98])

    if hi > lo:
        depth_vis = np.clip((depth - lo) / (hi - lo), 0, 1)
    else:
        depth_vis = np.zeros_like(depth)

    axes[row, 0].imshow(frame_rgb)
    axes[row, 0].set_title(
        f"Image originale — t={actual_time:.1f} s"
    )
    axes[row, 0].axis("off")

    axes[row, 1].imshow(depth_vis, cmap="inferno")
    axes[row, 1].set_title(
        "Profondeur relative normalisée pour cette image"
    )
    axes[row, 1].axis("off")

    audit_rows.append({
        "requested_time_s": timestamp,
        "actual_time_s": actual_time,
        "frame_index": frame_index,
        "depth_mean": float(record["depth_mean"]),
        "depth_median": float(record["depth_median"]),
        "depth_std": float(record["depth_std"]),
        "map_file": str(record["map_file"]),
    })

cap.release()
plt.tight_layout()
plt.show()

audit_df = pd.DataFrame(audit_rows)

display(audit_df.round(4))

audit_path = Path(
    "/kaggle/working/ADAS_FUSION_X2030/05_depth/"
    "depth_temporal_audit.csv"
)

audit_df.to_csv(audit_path, index=False)

print(f"Audit exporté : {audit_path}")
print(
    "Attention : la normalisation est locale à chaque image ; "
    "les couleurs ne sont pas directement comparables entre images."
)


## Étape 05.9 — Comparaison visuelle avec une échelle commune

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — Étape 05.9
# Comparaison de profondeur avec une échelle commune
# ============================================================

from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VIDEO_PATH = Path(
    "/kaggle/working/ADAS_FUSION_X2030/videos/"
    "ADAS_FUSION_X2030_Reference_180s.mp4"
)

BASE_DIR = Path(
    "/kaggle/working/ADAS_FUSION_X2030/05_depth"
)

RAW_DIR = BASE_DIR / "raw_maps"
CSV_PATH = BASE_DIR / "depth_temporal_statistics.csv"

df = pd.read_csv(CSV_PATH)
cap = cv2.VideoCapture(str(VIDEO_PATH))

assert cap.isOpened(), "Impossible d'ouvrir la vidéo."

# Instants à comparer
timestamps = [88, 90, 92, 94, 96, 98, 100, 102, 174]

# Sélectionner les cartes et images correspondantes
samples = []

for t in timestamps:
    idx = (df["timestamp_s"] - t).abs().idxmin()
    row = df.loc[idx]

    frame_idx = int(row["frame_index"])
    actual_t = float(row["timestamp_s"])

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame_bgr = cap.read()

    if not ok:
        print(f"Image illisible à t={actual_t:.1f} s")
        continue

    depth_path = RAW_DIR / str(row["map_file"])

    if not depth_path.exists():
        print(f"Carte manquante : {depth_path}")
        continue

    depth = np.load(depth_path)

    samples.append({
        "time": actual_t,
        "frame": cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB),
        "depth": depth,
        "mean": float(row["depth_mean"]),
        "median": float(row["depth_median"]),
    })

cap.release()

if not samples:
    raise RuntimeError("Aucun échantillon disponible pour l'audit.")

# Échelle commune déterminée sur les valeurs de toutes les cartes
all_values = np.concatenate([
    s["depth"].ravel()[::20]
    for s in samples
])

vmin, vmax = np.percentile(all_values, [2, 98])

print(f"Échelle commune : [{vmin:.4f}, {vmax:.4f}]")
print("Les valeurs affichées restent des sorties relatives du modèle.")

fig, axes = plt.subplots(
    len(samples), 2,
    figsize=(14, 4 * len(samples))
)

for i, s in enumerate(samples):
    axes[i, 0].imshow(s["frame"])
    axes[i, 0].set_title(f"Image originale — {s['time']:.1f} s")
    axes[i, 0].axis("off")

    im = axes[i, 1].imshow(
        s["depth"],
        cmap="inferno",
        vmin=vmin,
        vmax=vmax
    )

    axes[i, 1].set_title(
        f"Profondeur — médiane={s['median']:.3f}"
    )
    axes[i, 1].axis("off")

fig.colorbar(
    im,
    ax=axes[:, 1].tolist(),
    label="Valeur brute relative prédite",
    shrink=0.7
)

plt.tight_layout()
plt.show()


# Étape 06 — Prédiction des trajectoires des véhicules

## 06.1 — Vérifier les données de suivi disponibles

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — Étape 06.1
# Inspection des données de suivi
# ============================================================

from pathlib import Path
import pandas as pd
import json

TRACKING_DIR = Path(
    "/kaggle/working/ADAS_FUSION_X2030/03_tracking"
)

FILES = {
    "raw": TRACKING_DIR / "tracking_raw.csv",
    "stabilized": TRACKING_DIR / "tracking_stabilized.csv",
    "audit": TRACKING_DIR / "tracking_audit.csv",
    "metadata": TRACKING_DIR / "tracking_metadata.json",
}

for name, path in FILES.items():
    print(f"{name:12s} | existe={path.exists()} | {path}")

if not FILES["stabilized"].exists():
    raise FileNotFoundError(
        "Le fichier tracking_stabilized.csv est introuvable. "
        "Vérifie le dossier 03_tracking."
    )

df_tracks = pd.read_csv(FILES["stabilized"])

print("\n========== STRUCTURE DES DONNÉES ==========")
print("Dimensions :", df_tracks.shape)
print("Colonnes :", df_tracks.columns.tolist())

print("\n========== APERÇU ==========")
display(df_tracks.head(10))

print("\n========== VALEURS MANQUANTES ==========")
display(
    df_tracks.isna().sum()
    .rename("Valeurs manquantes")
    .to_frame()
)

print("\n========== MÉTADONNÉES ==========")

if FILES["metadata"].exists():
    with open(FILES["metadata"], "r", encoding="utf-8") as f:
        tracking_metadata = json.load(f)

    print(json.dumps(
        tracking_metadata,
        indent=2,
        ensure_ascii=False
    ))
else:
    print("Fichier de métadonnées absent.")


## Étape 06.2 — Construire un prédicteur de trajectoire

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — Étape 06.2
# Prédiction cinématique des trajectoires dans le plan image
# Modèle : extrapolation linéaire des centres des boîtes
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/kaggle/working/ADAS_FUSION_X2030")
TRACKING_DIR = BASE_DIR / "03_tracking"
OUTPUT_DIR = BASE_DIR / "06_trajectory"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV = TRACKING_DIR / "tracking_stabilized.csv"

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Fichier introuvable : {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)

required = [
    "timestamp_s", "track_id", "x1", "y1", "x2", "y2",
    "stable_class", "confidence"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Colonnes manquantes : {missing}")

# ------------------------------------------------------------
# 1. Préparation des positions
# ------------------------------------------------------------

df = df.copy()

df["center_x"] = (df["x1"] + df["x2"]) / 2.0
df["center_y"] = (df["y1"] + df["y2"]) / 2.0

df["box_width"] = df["x2"] - df["x1"]
df["box_height"] = df["y2"] - df["y1"]

df = df.sort_values(["track_id", "timestamp_s"])

# Horizons à prédire
HORIZONS = [0.5, 1.0, 2.0]

# Paramètres de l'évaluation
MIN_HISTORY = 5
TIME_TOLERANCE = 0.025  # tolérance pour retrouver une observation future

prediction_rows = []

# ------------------------------------------------------------
# 2. Prédiction pour chaque trajectoire
# ------------------------------------------------------------

for track_id, group in df.groupby("track_id"):

    group = group.sort_values("timestamp_s").reset_index(drop=True)

    if len(group) < MIN_HISTORY + 1:
        continue

    times = group["timestamp_s"].to_numpy(dtype=float)
    xs = group["center_x"].to_numpy(dtype=float)
    ys = group["center_y"].to_numpy(dtype=float)

    for i in range(MIN_HISTORY - 1, len(group) - 1):

        # Historique strictement antérieur ou égal à l'instant de prédiction
        t0 = times[i]

        history_t = times[:i + 1]
        history_x = xs[:i + 1]
        history_y = ys[:i + 1]

        # Régression linéaire sur les dernières observations disponibles.
        # On utilise au maximum les 12 observations les plus récentes.
        window = min(12, len(history_t))

        ht = history_t[-window:]
        hx = history_x[-window:]
        hy = history_y[-window:]

        # Temps relatif pour limiter les problèmes numériques
        tau = ht - ht[-1]

        vx = np.polyfit(tau, hx, 1)[0]
        vy = np.polyfit(tau, hy, 1)[0]

        x0 = xs[i]
        y0 = ys[i]

        for horizon in HORIZONS:

            target_time = t0 + horizon

            # Retrouver une observation future suffisamment proche
            future_idx = np.where(
                np.abs(times - target_time) <= TIME_TOLERANCE
            )[0]

            future_idx = future_idx[future_idx > i]

            if len(future_idx) == 0:
                continue

            j = future_idx[
                np.argmin(np.abs(times[future_idx] - target_time))
            ]

            # Prédiction au temps cible
            x_pred = x0 + vx * horizon
            y_pred = y0 + vy * horizon

            # Position observée à l'instant futur
            x_true = xs[j]
            y_true = ys[j]

            error_px = float(
                np.hypot(x_pred - x_true, y_pred - y_true)
            )

            prediction_rows.append({
                "track_id": int(track_id),
                "class": str(group.loc[i, "stable_class"]),
                "timestamp_s": float(t0),
                "target_timestamp_s": float(times[j]),
                "horizon_s": float(horizon),
                "x0_px": float(x0),
                "y0_px": float(y0),
                "vx_image_px_s": float(vx),
                "vy_image_px_s": float(vy),
                "x_pred_px": float(x_pred),
                "y_pred_px": float(y_pred),
                "x_observed_px": float(x_true),
                "y_observed_px": float(y_true),
                "error_px": error_px,
                "history_observations": int(window),
            })

# ------------------------------------------------------------
# 3. Export des prédictions
# ------------------------------------------------------------

pred_df = pd.DataFrame(prediction_rows)

if pred_df.empty:
    raise RuntimeError(
        "Aucune prédiction évaluable. Vérifie la durée des pistes "
        "et les horodatages disponibles."
    )

PRED_PATH = OUTPUT_DIR / "trajectory_predictions.csv"
pred_df.to_csv(PRED_PATH, index=False)

# ------------------------------------------------------------
# 4. Résumé des erreurs par horizon
# ------------------------------------------------------------

summary_rows = []

for horizon, group in pred_df.groupby("horizon_s"):

    errors = group["error_px"].to_numpy()

    summary_rows.append({
        "horizon_s": float(horizon),
        "n_predictions": int(len(group)),
        "n_tracks": int(group["track_id"].nunique()),
        "MAE_position_px": float(np.mean(errors)),
        "Median_error_px": float(np.median(errors)),
        "P90_error_px": float(np.percentile(errors, 90)),
        "RMSE_position_px": float(np.sqrt(np.mean(errors ** 2))),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("horizon_s")

SUMMARY_PATH = OUTPUT_DIR / "trajectory_evaluation_summary.csv"
summary_df.to_csv(SUMMARY_PATH, index=False)

# ------------------------------------------------------------
# 5. Affichage des résultats
# ------------------------------------------------------------

print("========== PRÉDICTION DES TRAJECTOIRES ==========")
print(f"Observations initiales : {len(df)}")
print(f"Identifiants de suivi  : {df['track_id'].nunique()}")
print(f"Prédictions évaluées   : {len(pred_df)}")

print("\n========== ERREURS PAR HORIZON ==========")
display(summary_df.round(3))

print("\nFichiers enregistrés :")
print(PRED_PATH)
print(SUMMARY_PATH)

# ------------------------------------------------------------
# 6. Visualisation de l'erreur
# ------------------------------------------------------------

plt.figure(figsize=(9, 5))

plt.boxplot(
    [
        pred_df.loc[
            pred_df["horizon_s"] == horizon, "error_px"
        ].to_numpy()
        for horizon in HORIZONS
    ],
    tick_labels=[f"{h:.1f} s" for h in HORIZONS],
    showfliers=False
)

plt.xlabel("Horizon de prédiction")
plt.ylabel("Erreur de position (pixels)")
plt.title("Erreur de prédiction des trajectoires")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(
    "\nNote : les erreurs sont exprimées en pixels. "
    "Elles évaluent une extrapolation dans le plan image, "
    "pas une trajectoire physique en mètres."
)


## Étape 06.3 — Comparer deux modèles de prédiction

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — Étape 06.3
# Baseline constante vs extrapolation linéaire
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/kaggle/working/ADAS_FUSION_X2030")
TRACKING_DIR = BASE_DIR / "03_tracking"
OUTPUT_DIR = BASE_DIR / "06_trajectory"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    TRACKING_DIR / "tracking_stabilized.csv"
).copy()

df["center_x"] = (df["x1"] + df["x2"]) / 2
df["center_y"] = (df["y1"] + df["y2"]) / 2

df = df.sort_values(
    ["track_id", "timestamp_s"]
).reset_index(drop=True)

HORIZONS = [0.5, 1.0, 2.0]

MIN_HISTORY = 5
MAX_HISTORY = 12

# Le suivi est effectué à 24 FPS ; 1 image représente environ 0,0417 s.
# Tolérance de 0,021 s : moins d'une demi-période d'image.
TIME_TOLERANCE = 0.021

rows = []

for track_id, group in df.groupby("track_id"):

    group = group.sort_values(
        "timestamp_s"
    ).reset_index(drop=True)

    if len(group) <= MIN_HISTORY:
        continue

    times = group["timestamp_s"].to_numpy(float)
    xs = group["center_x"].to_numpy(float)
    ys = group["center_y"].to_numpy(float)

    for i in range(MIN_HISTORY - 1, len(group)):

        t0 = times[i]

        # Historique disponible jusqu'à t0 inclus
        start = max(0, i - MAX_HISTORY + 1)

        ht = times[start:i + 1]
        hx = xs[start:i + 1]
        hy = ys[start:i + 1]

        # Régression sur le temps relatif
        tau = ht - t0

        vx = np.polyfit(tau, hx, 1)[0]
        vy = np.polyfit(tau, hy, 1)[0]

        x0, y0 = xs[i], ys[i]

        for horizon in HORIZONS:

            target = t0 + horizon

            # Première observation disponible après t0,
            # suffisamment proche de l'horizon cible.
            candidates = np.where(
                (times > t0) &
                (np.abs(times - target) <= TIME_TOLERANCE)
            )[0]

            if len(candidates) == 0:
                continue

            j = candidates[
                np.argmin(np.abs(times[candidates] - target))
            ]

            x_true, y_true = xs[j], ys[j]

            # Modèle 1 : position constante
            x_const, y_const = x0, y0

            # Modèle 2 : extrapolation linéaire
            # On utilise l'horizon effectivement observé.
            dt = times[j] - t0
            x_linear = x0 + vx * dt
            y_linear = y0 + vy * dt

            error_const = np.hypot(
                x_const - x_true,
                y_const - y_true
            )

            error_linear = np.hypot(
                x_linear - x_true,
                y_linear - y_true
            )

            rows.append({
                "track_id": int(track_id),
                "timestamp_s": float(t0),
                "target_timestamp_s": float(times[j]),
                "horizon_requested_s": float(horizon),
                "horizon_actual_s": float(dt),
                "error_constant_px": float(error_const),
                "error_linear_px": float(error_linear),
            })

results = pd.DataFrame(rows)

if results.empty:
    raise RuntimeError(
        "Aucune prédiction évaluable avec la tolérance temporelle choisie."
    )

# ------------------------------------------------------------
# Résumé statistique
# ------------------------------------------------------------

summary_rows = []

for horizon, g in results.groupby("horizon_requested_s"):

    for model_name, col in [
        ("Position constante", "error_constant_px"),
        ("Extrapolation linéaire", "error_linear_px"),
    ]:

        e = g[col].to_numpy()

        summary_rows.append({
            "horizon_s": horizon,
            "model": model_name,
            "n_predictions": len(g),
            "n_tracks": g["track_id"].nunique(),
            "MAE_px": np.mean(e),
            "Median_px": np.median(e),
            "P90_px": np.percentile(e, 90),
            "RMSE_px": np.sqrt(np.mean(e ** 2)),
        })

summary = pd.DataFrame(summary_rows)

# Comparaison appariée : mêmes instants et observations futures
comparison_rows = []

for horizon, g in results.groupby("horizon_requested_s"):

    mean_const = g["error_constant_px"].mean()
    mean_linear = g["error_linear_px"].mean()

    improvement = (
        100 * (mean_const - mean_linear) / mean_const
        if mean_const > 0 else np.nan
    )

    comparison_rows.append({
        "horizon_s": horizon,
        "MAE_constant_px": mean_const,
        "MAE_linear_px": mean_linear,
        "amelioration_MAE_pct": improvement,
        "n_predictions": len(g),
        "n_tracks": g["track_id"].nunique(),
    })

comparison = pd.DataFrame(comparison_rows)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

results.to_csv(
    OUTPUT_DIR / "trajectory_baseline_comparison.csv",
    index=False
)

summary.to_csv(
    OUTPUT_DIR / "trajectory_baseline_summary.csv",
    index=False
)

comparison.to_csv(
    OUTPUT_DIR / "trajectory_model_comparison.csv",
    index=False
)

print("========== MÉTRIQUES PAR MODÈLE ==========")
display(summary.round(3))

print("\n========== COMPARAISON DIRECTE ==========")
display(comparison.round(3))

print("\nFichiers exportés dans :", OUTPUT_DIR)

# ------------------------------------------------------------
# Visualisation du MAE
# ------------------------------------------------------------

pivot = summary.pivot(
    index="horizon_s",
    columns="model",
    values="MAE_px"
)

ax = pivot.plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_xlabel("Horizon de prédiction (s)")
ax.set_ylabel("MAE (pixels)")
ax.set_title("Comparaison des modèles de trajectoire")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(
    "\nInterprétation : une amélioration positive signifie que "
    "l'extrapolation linéaire réduit le MAE par rapport à la "
    "position constante, sur les mêmes observations."
)


# Étape 07 — Analyse des risques ADAS

## Étape 07.1 — Examiner les données avant de définir le score de risque

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — Étape 07.1
# Diagnostic des indicateurs visuels de risque
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/kaggle/working/ADAS_FUSION_X2030")
TRACKING_DIR = BASE_DIR / "03_tracking"
OUTPUT_DIR = BASE_DIR / "07_risk_analysis"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV = TRACKING_DIR / "tracking_stabilized.csv"
df = pd.read_csv(INPUT_CSV)

# Dimensions de la vidéo validée précédemment
IMAGE_WIDTH = 1920
IMAGE_HEIGHT = 1080

# Centre des boîtes
df["center_x"] = (df["x1"] + df["x2"]) / 2
df["center_y"] = (df["y1"] + df["y2"]) / 2

# Dimensions et surface des boîtes
df["box_width"] = (df["x2"] - df["x1"]).clip(lower=0)
df["box_height"] = (df["y2"] - df["y1"]).clip(lower=0)
df["box_area"] = df["box_width"] * df["box_height"]

# Normalisation géométrique
df["center_x_norm"] = df["center_x"] / IMAGE_WIDTH
df["center_y_norm"] = df["center_y"] / IMAGE_HEIGHT

df["box_area_ratio"] = (
    df["box_area"] / (IMAGE_WIDTH * IMAGE_HEIGHT)
)

# Calcul temporel par piste
df = df.sort_values(["track_id", "timestamp_s"])

df["delta_time_s"] = (
    df.groupby("track_id")["timestamp_s"].diff()
)

df["delta_center_x"] = (
    df.groupby("track_id")["center_x"].diff()
)

df["delta_center_y"] = (
    df.groupby("track_id")["center_y"].diff()
)

df["apparent_speed_px_s"] = (
    np.hypot(df["delta_center_x"], df["delta_center_y"])
    / df["delta_time_s"].replace(0, np.nan)
)

# Diagnostic général
print("========== QUALITÉ ET GÉOMÉTRIE ==========")
print(f"Observations : {len(df)}")
print(f"Pistes       : {df['track_id'].nunique()}")
print(f"Classes      : {df['stable_class'].nunique()}")

print("\n========== DISTRIBUTION DES INDICATEURS ==========")

cols = [
    "center_x_norm",
    "center_y_norm",
    "box_area_ratio",
    "box_width",
    "box_height",
    "apparent_speed_px_s",
]

display(df[cols].describe(percentiles=[0.5, 0.9, 0.95]).round(4))

print("\n========== RÉSUMÉ PAR CLASSE ==========")

class_summary = (
    df.groupby("stable_class")
    .agg(
        observations=("track_id", "size"),
        pistes=("track_id", "nunique"),
        surface_moyenne=("box_area_ratio", "mean"),
        surface_mediane=("box_area_ratio", "median"),
        vitesse_apparente_mediane=("apparent_speed_px_s", "median"),
    )
    .reset_index()
)

display(class_summary.round(5))

class_summary.to_csv(
    OUTPUT_DIR / "visual_risk_diagnostics_by_class.csv",
    index=False
)

# Graphique 1 : position verticale et surface des boîtes
plt.figure(figsize=(9, 6))
plt.scatter(
    df["center_y_norm"],
    df["box_area_ratio"],
    alpha=0.5
)
plt.xlabel("Position verticale normalisée")
plt.ylabel("Surface relative de la boîte")
plt.title("Position verticale et taille apparente des véhicules")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Graphique 2 : vitesse apparente et surface
valid = df[
    np.isfinite(df["apparent_speed_px_s"])
    & np.isfinite(df["box_area_ratio"])
]

plt.figure(figsize=(9, 6))
plt.scatter(
    valid["apparent_speed_px_s"],
    valid["box_area_ratio"],
    alpha=0.5
)
plt.xlabel("Vitesse apparente (pixels/s)")
plt.ylabel("Surface relative de la boîte")
plt.title("Vitesse apparente et taille des objets suivis")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nRésultats enregistrés dans :", OUTPUT_DIR)


## Étape 07.2 — Audit des pistes et des valeurs extrêmes

### Cellule 1 — Identifier les pistes suspectes

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — ÉTAPE 07.2
# Audit des anomalies de suivi avant le calcul du risque visuel
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/kaggle/working/ADAS_FUSION_X2030")
TRACKING_FILE = BASE_DIR / "03_tracking" / "tracking_stabilized.csv"
OUTPUT_DIR = BASE_DIR / "07_risk_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRACKING_FILE.exists(), (
    f"Fichier introuvable : {TRACKING_FILE}"
)

df = pd.read_csv(TRACKING_FILE)

required_columns = [
    "frame", "timestamp_s", "track_id", "stable_class",
    "x1", "y1", "x2", "y2"
]

missing = [c for c in required_columns if c not in df.columns]
assert not missing, f"Colonnes manquantes : {missing}"

# ------------------------------------------------------------
# 1. Géométrie des boîtes
# ------------------------------------------------------------

IMAGE_WIDTH = 1920
IMAGE_HEIGHT = 1080
FPS = 24

df["center_x"] = (df["x1"] + df["x2"]) / 2
df["center_y"] = (df["y1"] + df["y2"]) / 2

df["box_width"] = (df["x2"] - df["x1"]).clip(lower=0)
df["box_height"] = (df["y2"] - df["y1"]).clip(lower=0)

df["box_area_ratio"] = (
    df["box_width"] * df["box_height"]
) / (IMAGE_WIDTH * IMAGE_HEIGHT)

df["center_x_norm"] = df["center_x"] / IMAGE_WIDTH
df["center_y_norm"] = df["center_y"] / IMAGE_HEIGHT

# ------------------------------------------------------------
# 2. Mouvement apparent par piste
# ------------------------------------------------------------

df = df.sort_values(["track_id", "timestamp_s"]).copy()
grouped = df.groupby("track_id", sort=False)

df["dt"] = grouped["timestamp_s"].diff()
df["dx"] = grouped["center_x"].diff()
df["dy"] = grouped["center_y"].diff()

valid_dt = df["dt"].where(df["dt"] > 0)

df["apparent_speed_px_s"] = (
    np.sqrt(df["dx"]**2 + df["dy"]**2) / valid_dt
)

# Variation relative de taille entre observations successives.
previous_area = grouped["box_area_ratio"].shift(1)

df["relative_area_change"] = (
    (df["box_area_ratio"] - previous_area)
    / previous_area.replace(0, np.nan)
)

# ------------------------------------------------------------
# 3. Résumé statistique par piste
# ------------------------------------------------------------

summary = (
    df.groupby("track_id")
      .agg(
          stable_class=("stable_class", "first"),
          observations=("frame", "count"),
          first_time_s=("timestamp_s", "min"),
          last_time_s=("timestamp_s", "max"),
          median_area_ratio=("box_area_ratio", "median"),
          max_area_ratio=("box_area_ratio", "max"),
          median_speed_px_s=("apparent_speed_px_s", "median"),
          p95_speed_px_s=(
              "apparent_speed_px_s",
              lambda s: s.quantile(0.95)
          ),
          max_speed_px_s=("apparent_speed_px_s", "max"),
          max_abs_relative_area_change=(
              "relative_area_change",
              lambda s: s.abs().max()
          )
      )
      .reset_index()
)

summary["duration_s"] = (
    summary["last_time_s"] - summary["first_time_s"]
)

# Indicateurs d'audit, pas des diagnostics définitifs.
summary["flag_high_speed"] = (
    summary["p95_speed_px_s"] > 400
)

summary["flag_large_area_change"] = (
    summary["max_abs_relative_area_change"] > 0.50
)

summary["flag_short_track"] = (
    summary["observations"] < 10
)

summary["audit_flags_count"] = summary[
    [
        "flag_high_speed",
        "flag_large_area_change",
        "flag_short_track"
    ]
].sum(axis=1)

summary["audit_priority"] = np.select(
    [
        summary["audit_flags_count"] >= 2,
        summary["audit_flags_count"] == 1
    ],
    [
        "À examiner en priorité",
        "À vérifier"
    ],
    default="Aucun signal selon ces critères"
)

summary = summary.sort_values(
    ["audit_flags_count", "p95_speed_px_s"],
    ascending=[False, False]
)

# ------------------------------------------------------------
# 4. Export des résultats
# ------------------------------------------------------------

audit_path = OUTPUT_DIR / "track_anomaly_audit.csv"
summary.to_csv(audit_path, index=False)

# Exporter également les observations des pistes prioritaires.
priority_ids = summary.loc[
    summary["audit_flags_count"] >= 1, "track_id"
]

priority_observations = df[
    df["track_id"].isin(priority_ids)
].copy()

observations_path = (
    OUTPUT_DIR / "track_anomaly_observations.csv"
)

priority_observations.to_csv(
    observations_path, index=False
)

metadata = {
    "project": "ADAS-FUSION X 2030",
    "step": "07.2 - Track anomaly audit",
    "source": str(TRACKING_FILE),
    "number_of_observations": int(len(df)),
    "number_of_tracks": int(df["track_id"].nunique()),
    "threshold_p95_speed_px_s": 400,
    "threshold_abs_relative_area_change": 0.50,
    "threshold_short_track_observations": 10,
    "interpretation": (
        "Heuristic audit flags only; not proof of tracking errors "
        "or collision risk. Speed is expressed in image pixels per second."
    )
}

with open(
    OUTPUT_DIR / "track_anomaly_audit_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 5. Affichage
# ------------------------------------------------------------

print("========== BILAN DE L'AUDIT ==========")
print(f"Observations : {len(df)}")
print(f"Pistes       : {df['track_id'].nunique()}")
print(f"Pistes à examiner : {(summary['audit_flags_count'] >= 1).sum()}")
print(f"Pistes prioritaires : {(summary['audit_flags_count'] >= 2).sum()}")

print("\n========== PISTES À EXAMINER ==========")

display(
    summary.loc[
        summary["audit_flags_count"] >= 1,
        [
            "track_id",
            "stable_class",
            "observations",
            "duration_s",
            "median_area_ratio",
            "p95_speed_px_s",
            "max_speed_px_s",
            "max_abs_relative_area_change",
            "audit_priority"
        ]
    ].head(20).round(4)
)

print("\n========== FICHIERS EXPORTÉS ==========")
print(audit_path)
print(observations_path)
print(OUTPUT_DIR / "track_anomaly_audit_metadata.json")


### Étape 07.2 — Récupérer les résultats de l'audit

In [ ]:

import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    "/kaggle/working/ADAS_FUSION_X2030/07_risk_analysis"
)

audit = pd.read_csv(
    OUTPUT_DIR / "track_anomaly_audit.csv"
)

print("========== BILAN DE L'AUDIT ==========")
print("Nombre de pistes :", len(audit))

if "audit_flags_count" in audit.columns:
    print(
        "Pistes signalées :",
        int((audit["audit_flags_count"] >= 1).sum())
    )
    print(
        "Pistes prioritaires :",
        int((audit["audit_flags_count"] >= 2).sum())
    )

columns = [
    "track_id",
    "stable_class",
    "observations",
    "duration_s",
    "median_area_ratio",
    "p95_speed_px_s",
    "max_speed_px_s",
    "max_abs_relative_area_change",
    "audit_priority"
]

columns = [c for c in columns if c in audit.columns]

print("\n========== PISTES À EXAMINER ==========")

display(
    audit.loc[
        audit["audit_flags_count"] >= 1,
        columns
    ].head(20).round(4)
)

print("\n========== RÉPARTITION DES PRIORITÉS ==========")

if "audit_priority" in audit.columns:
    display(
        audit["audit_priority"]
        .value_counts(dropna=False)
        .rename_axis("priorité")
        .to_frame("nombre_de_pistes")
    )


## Étape 07.3 — Construire le score de risque visuel

### Calcul du score

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — ÉTAPE 07.3
# Score heuristique de risque visuel
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/kaggle/working/ADAS_FUSION_X2030")
INPUT_FILE = BASE_DIR / "03_tracking" / "tracking_stabilized.csv"
OUTPUT_DIR = BASE_DIR / "07_risk_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_WIDTH = 1920
IMAGE_HEIGHT = 1080

df = pd.read_csv(INPUT_FILE)

required = [
    "frame", "timestamp_s", "track_id", "stable_class",
    "x1", "y1", "x2", "y2"
]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Colonnes manquantes : {missing}"

# ------------------------------------------------------------
# 1. Géométrie des boîtes
# ------------------------------------------------------------

df["box_width"] = (df["x2"] - df["x1"]).clip(lower=0)
df["box_height"] = (df["y2"] - df["y1"]).clip(lower=0)

df["center_x"] = (df["x1"] + df["x2"]) / 2
df["center_y"] = (df["y1"] + df["y2"]) / 2

df["box_area_ratio"] = (
    df["box_width"] * df["box_height"]
    / (IMAGE_WIDTH * IMAGE_HEIGHT)
)

df["center_y_norm"] = df["center_y"] / IMAGE_HEIGHT

# ------------------------------------------------------------
# 2. Mouvement apparent par piste
# ------------------------------------------------------------

df = df.sort_values(["track_id", "timestamp_s"]).copy()
groups = df.groupby("track_id", sort=False)

df["dt"] = groups["timestamp_s"].diff()
df["dx"] = groups["center_x"].diff()
df["dy"] = groups["center_y"].diff()

valid_dt = df["dt"].where(df["dt"] > 0)

df["apparent_speed_px_s"] = (
    np.sqrt(df["dx"]**2 + df["dy"]**2) / valid_dt
)

# ------------------------------------------------------------
# 3. Normalisation robuste
# ------------------------------------------------------------
# Les percentiles servent à limiter l'influence des valeurs extrêmes.
# Ils ne constituent pas des seuils physiques de sécurité.

def percentile_scale(series, low_q=5, high_q=95):
    s = series.replace([np.inf, -np.inf], np.nan)
    low = s.quantile(low_q / 100)
    high = s.quantile(high_q / 100)

    if pd.isna(low) or pd.isna(high):
        return pd.Series(0.0, index=series.index)

    if high <= low:
        return pd.Series(0.0, index=series.index)

    return ((s - low) / (high - low)).clip(0, 1).fillna(0)

df["score_size"] = percentile_scale(df["box_area_ratio"])

# Une position verticale plus basse reçoit ici un score plus élevé.
# C'est une hypothèse géométrique de priorisation visuelle,
# pas une preuve de proximité physique.
df["score_vertical"] = percentile_scale(df["center_y_norm"])

df["score_motion"] = percentile_scale(df["apparent_speed_px_s"])

# ------------------------------------------------------------
# 4. Qualité minimale de suivi
# ------------------------------------------------------------

track_counts = df.groupby("track_id")["frame"].transform("count")
df["track_observations"] = track_counts

# Indicateur de confiance dans la stabilité statistique de la piste.
# Il ne s'agit pas de la confiance du détecteur.
df["track_quality_factor"] = np.select(
    [
        track_counts < 3,
        track_counts < 10
    ],
    [
        0.25,
        0.60
    ],
    default=1.0
)

# ------------------------------------------------------------
# 5. Score visuel brut et score ajusté
# ------------------------------------------------------------

W_SIZE = 0.50
W_VERTICAL = 0.30
W_MOTION = 0.20

assert abs(W_SIZE + W_VERTICAL + W_MOTION - 1.0) < 1e-9

df["visual_risk_raw"] = 100 * (
    W_SIZE * df["score_size"]
    + W_VERTICAL * df["score_vertical"]
    + W_MOTION * df["score_motion"]
)

df["visual_risk_score"] = (
    df["visual_risk_raw"] * df["track_quality_factor"]
)

# Catégories de visualisation expérimentales.
# Elles ne représentent pas des niveaux de sécurité homologués.
df["visual_risk_category"] = pd.cut(
    df["visual_risk_score"],
    bins=[-0.001, 25, 50, 75, 100],
    labels=[
        "Faible",
        "Modéré",
        "Élevé",
        "Très élevé"
    ]
)

# ------------------------------------------------------------
# 6. Résumé par piste
# ------------------------------------------------------------

track_summary = (
    df.groupby("track_id", observed=False)
      .agg(
          stable_class=("stable_class", "first"),
          observations=("frame", "count"),
          duration_s=("timestamp_s", lambda s: s.max() - s.min()),
          median_size_ratio=("box_area_ratio", "median"),
          median_apparent_speed_px_s=("apparent_speed_px_s", "median"),
          mean_visual_risk=("visual_risk_score", "mean"),
          max_visual_risk=("visual_risk_score", "max"),
          pct_high_or_very_high=(
              "visual_risk_category",
              lambda s: 100 * s.isin(["Élevé", "Très élevé"]).mean()
          )
      )
      .reset_index()
      .sort_values("max_visual_risk", ascending=False)
)

# ------------------------------------------------------------
# 7. Export
# ------------------------------------------------------------

observations_file = OUTPUT_DIR / "visual_risk_scores.csv"
tracks_file = OUTPUT_DIR / "visual_risk_summary_by_track.csv"
metadata_file = OUTPUT_DIR / "visual_risk_metadata.json"

df.to_csv(observations_file, index=False)
track_summary.to_csv(tracks_file, index=False)

metadata = {
    "project": "ADAS-FUSION X 2030",
    "step": "07.3 - Heuristic visual risk score",
    "source": str(INPUT_FILE),
    "weights": {
        "apparent_size": W_SIZE,
        "vertical_position": W_VERTICAL,
        "apparent_motion": W_MOTION
    },
    "normalization": "Empirical 5th and 95th percentiles",
    "track_quality_factor": {
        "fewer_than_3_observations": 0.25,
        "3_to_9_observations": 0.60,
        "10_or_more_observations": 1.0
    },
    "score_range": [0, 100],
    "limitations": [
        "Heuristic visual prioritization score, not collision probability",
        "No metric distance estimation",
        "No calibrated time-to-collision",
        "Weights and category thresholds are experimental",
        "No ground-truth risk labels available"
    ]
}

with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 8. Résultats
# ------------------------------------------------------------

print("========== SCORE DE RISQUE VISUEL ==========")
print("Observations :", len(df))
print("Pistes :", df["track_id"].nunique())
print("Score moyen :", round(df["visual_risk_score"].mean(), 3))
print("Score médian :", round(df["visual_risk_score"].median(), 3))
print("Score maximal :", round(df["visual_risk_score"].max(), 3))

print("\n========== RÉPARTITION DES CATÉGORIES ==========")
display(
    df["visual_risk_category"]
    .value_counts(dropna=False)
    .rename_axis("catégorie")
    .to_frame("observations")
)

print("\n========== TOP 10 DES PISTES ==========")
display(track_summary.head(10).round(4))

print("\n========== EXPORTS ==========")
print(observations_file)
print(tracks_file)
print(metadata_file)

# ------------------------------------------------------------
# 9. Visualisation temporelle
# ------------------------------------------------------------

plt.figure(figsize=(14, 5))

for track_id, track in df.groupby("track_id"):
    if len(track) >= 3:
        plt.plot(
            track["timestamp_s"],
            track["visual_risk_score"],
            alpha=0.55,
            linewidth=1
        )

plt.xlabel("Temps dans le clip (s)")
plt.ylabel("Score heuristique de risque visuel")
plt.title("ADAS-FUSION X 2030 — Évolution du score par piste")
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Distribution globale des scores
plt.figure(figsize=(10, 5))
plt.hist(df["visual_risk_score"], bins=25)
plt.xlabel("Score heuristique")
plt.ylabel("Nombre d'observations")
plt.title("Distribution du score de risque visuel")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Étape 07.4 — Vérification temporelle et sélection des événements

### Cellule Kaggle : extraire les événements prioritaires

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — ÉTAPE 07.4
# Analyse temporelle des événements à score élevé
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("/kaggle/working/ADAS_FUSION_X2030")
RISK_DIR = BASE_DIR / "07_risk_analysis"

SCORES_FILE = RISK_DIR / "visual_risk_scores.csv"

assert SCORES_FILE.exists(), (
    f"Fichier introuvable : {SCORES_FILE}"
)

df = pd.read_csv(SCORES_FILE)

# Vérification des colonnes nécessaires
required = [
    "timestamp_s",
    "track_id",
    "stable_class",
    "visual_risk_score",
    "score_size",
    "score_vertical",
    "score_motion",
    "track_observations",
    "apparent_speed_px_s",
    "box_area_ratio"
]

missing = [c for c in required if c not in df.columns]
assert not missing, f"Colonnes manquantes : {missing}"

# ------------------------------------------------------------
# 1. Les 20 observations aux scores les plus élevés
# ------------------------------------------------------------

top_events = (
    df.sort_values("visual_risk_score", ascending=False)
      .head(20)
      .copy()
)

columns_to_show = [
    "timestamp_s",
    "track_id",
    "stable_class",
    "visual_risk_score",
    "score_size",
    "score_vertical",
    "score_motion",
    "box_area_ratio",
    "apparent_speed_px_s",
    "track_observations"
]

print("========== TOP 20 DES OBSERVATIONS ==========")

display(
    top_events[columns_to_show]
    .round(4)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Résumé par piste prioritaire
# ------------------------------------------------------------

priority_ids = [19, 169, 136]

priority = df[df["track_id"].isin(priority_ids)].copy()

print("\n========== RÉSUMÉ DES PISTES PRIORITAIRES ==========")

priority_summary = (
    priority.groupby("track_id")
    .agg(
        classe=("stable_class", "first"),
        observations=("timestamp_s", "count"),
        temps_debut_s=("timestamp_s", "min"),
        temps_fin_s=("timestamp_s", "max"),
        score_moyen=("visual_risk_score", "mean"),
        score_maximal=("visual_risk_score", "max"),
        taille_moyenne=("score_size", "mean"),
        position_verticale_moyenne=("score_vertical", "mean"),
        mouvement_moyen=("score_motion", "mean"),
        vitesse_apparente_mediane=("apparent_speed_px_s", "median")
    )
    .reset_index()
)

display(priority_summary.round(4))

# ------------------------------------------------------------
# 3. Durée des séquences consécutives à score élevé
# ------------------------------------------------------------
# Une séquence n'est considérée comme consécutive que si deux
# observations successives de la même piste sont espacées
# d'environ une période vidéo, avec une tolérance explicite.

df = df.sort_values(["track_id", "timestamp_s"]).copy()

FRAME_PERIOD = 1 / 24
MAX_GAP_S = 1.5 * FRAME_PERIOD

df["is_high_risk_visual"] = (
    df["visual_risk_score"] >= 50
)

df["time_gap_s"] = (
    df.groupby("track_id")["timestamp_s"].diff()
)

df["new_sequence"] = (
    df["time_gap_s"].isna()
    | (df["time_gap_s"] > MAX_GAP_S)
    | (
        df["is_high_risk_visual"]
        != df.groupby("track_id")["is_high_risk_visual"].shift()
    )
)

df["sequence_id"] = (
    df.groupby("track_id")["new_sequence"].cumsum()
)

high = df[df["is_high_risk_visual"]].copy()

if not high.empty:
    sequences = (
        high.groupby(["track_id", "sequence_id"])
        .agg(
            debut_s=("timestamp_s", "min"),
            fin_s=("timestamp_s", "max"),
            observations=("timestamp_s", "count"),
            score_moyen=("visual_risk_score", "mean"),
            score_maximal=("visual_risk_score", "max")
        )
        .reset_index()
    )

    sequences["duree_s"] = (
        sequences["fin_s"] - sequences["debut_s"]
    )

    sequences = sequences.sort_values(
        ["score_maximal", "observations"],
        ascending=[False, False]
    )

    print("\n========== SÉQUENCES À SCORE ÉLEVÉ ==========")

    display(sequences.head(20).round(4))

    sequences.to_csv(
        RISK_DIR / "high_visual_risk_sequences.csv",
        index=False
    )

# ------------------------------------------------------------
# 4. Export des observations prioritaires
# ------------------------------------------------------------

top_events.to_csv(
    RISK_DIR / "top_visual_risk_observations.csv",
    index=False
)

priority_summary.to_csv(
    RISK_DIR / "priority_track_score_analysis.csv",
    index=False
)

print("\nFichiers enregistrés :")
print(RISK_DIR / "top_visual_risk_observations.csv")
print(RISK_DIR / "priority_track_score_analysis.csv")

if not high.empty:
    print(RISK_DIR / "high_visual_risk_sequences.csv")


## Étape 07.5 — Construire un indicateur de persistance temporelle

In [ ]:

# ============================================================
# ADAS-FUSION X 2030 — ÉTAPE 07.5
# Persistance temporelle du score visuel
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("/kaggle/working/ADAS_FUSION_X2030")
RISK_DIR = BASE_DIR / "07_risk_analysis"

INPUT_FILE = RISK_DIR / "visual_risk_scores.csv"

assert INPUT_FILE.exists(), f"Fichier introuvable : {INPUT_FILE}"

df = pd.read_csv(INPUT_FILE)
df = df.sort_values(["track_id", "timestamp_s"]).copy()

required = [
    "track_id",
    "timestamp_s",
    "visual_risk_score",
    "track_observations"
]

missing = [c for c in required if c not in df.columns]
assert not missing, f"Colonnes manquantes : {missing}"

# ------------------------------------------------------------
# 1. Score élevé instantané
# ------------------------------------------------------------

df["high_score_flag"] = (
    df["visual_risk_score"] >= 50
).astype(int)

# ------------------------------------------------------------
# 2. Indicateurs glissants par piste
# ------------------------------------------------------------

grouped = df.groupby("track_id", sort=False)

df["risk_mean_5obs"] = (
    grouped["visual_risk_score"]
    .transform(
        lambda s: s.rolling(window=5, min_periods=1).mean()
    )
)

df["high_score_ratio_5obs"] = (
    grouped["high_score_flag"]
    .transform(
        lambda s: s.rolling(window=5, min_periods=1).mean()
    )
)

# Persistance mesurée sur les observations disponibles,
# et non sur une fenêtre temporelle physique fixe.
df["persistent_high_flag"] = (
    (df["risk_mean_5obs"] >= 50)
    & (df["high_score_ratio_5obs"] >= 0.60)
)

# ------------------------------------------------------------
# 3. Qualité statistique de la piste
# ------------------------------------------------------------

df["track_duration_s"] = (
    grouped["timestamp_s"].transform("max")
    - grouped["timestamp_s"].transform("min")
)

df["track_quality_label"] = np.select(
    [
        df["track_observations"] < 3,
        df["track_observations"] < 10,
        df["track_duration_s"] < 1.0
    ],
    [
        "Très insuffisante",
        "Limitée",
        "Durée courte"
    ],
    default="Suffisante pour analyse exploratoire"
)

# Ce label ne garantit pas que la piste est correcte.
# Il caractérise uniquement la quantité d'information disponible.

# ------------------------------------------------------------
# 4. Résumé par piste
# ------------------------------------------------------------

summary = (
    df.groupby("track_id")
      .agg(
          observations=("timestamp_s", "count"),
          duree_s=("track_duration_s", "first"),
          score_moyen=("visual_risk_score", "mean"),
          score_maximal=("visual_risk_score", "max"),
          score_glissant_moyen=("risk_mean_5obs", "mean"),
          proportion_score_eleve=(
              "high_score_flag",
              "mean"
          ),
          proportion_persistante=(
              "persistent_high_flag",
              "mean"
          ),
          classe_qualite=("track_quality_label", "first")
      )
      .reset_index()
)

summary["proportion_score_eleve"] *= 100
summary["proportion_persistante"] *= 100

summary = summary.sort_values(
    ["proportion_persistante", "score_glissant_moyen"],
    ascending=False
)

# ------------------------------------------------------------
# 5. Exports
# ------------------------------------------------------------

observations_path = RISK_DIR / "visual_risk_temporal_scores.csv"
summary_path = RISK_DIR / "visual_risk_temporal_summary.csv"
metadata_path = RISK_DIR / "visual_risk_temporal_metadata.json"

df.to_csv(observations_path, index=False)
summary.to_csv(summary_path, index=False)

metadata = {
    "project": "ADAS-FUSION X 2030",
    "step": "07.5 - Temporal persistence",
    "source": str(INPUT_FILE),
    "rolling_window_observations": 5,
    "instantaneous_high_threshold": 50,
    "persistent_mean_threshold": 50,
    "persistent_high_ratio_threshold": 0.60,
    "limitations": [
        "Persistence is computed over observations, not a fixed time window",
        "Track quality label is not a detection confidence score",
        "No collision probability or safety guarantee is inferred"
    ]
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 6. Affichage des résultats
# ------------------------------------------------------------

print("========== BILAN TEMPOREL ==========")
print("Observations :", len(df))
print("Pistes :", df["track_id"].nunique())
print(
    "Observations avec score instantané >= 50 :",
    int(df["high_score_flag"].sum())
)
print(
    "Observations avec score persistant :",
    int(df["persistent_high_flag"].sum())
)

print("\n========== TOP 10 PAR PERSISTANCE ==========")
display(summary.head(10).round(4))

print("\n========== PISTES 19, 169 ET 136 ==========")
display(
    summary[
        summary["track_id"].isin([19, 169, 136])
    ].round(4)
)

print("\n========== FICHIERS EXPORTÉS ==========")
print(observations_path)
print(summary_path)
print(metadata_path)

# ------------------------------------------------------------
# 7. Visualisation : pistes prioritaires
# ------------------------------------------------------------

plt.figure(figsize=(14, 6))

for track_id in [19, 169, 136]:
    track = df[df["track_id"] == track_id]

    if not track.empty:
        plt.plot(
            track["timestamp_s"],
            track["visual_risk_score"],
            marker=".",
            markersize=3,
            linewidth=1,
            label=f"Piste {track_id} — score instantané"
        )

        plt.plot(
            track["timestamp_s"],
            track["risk_mean_5obs"],
            linewidth=2,
            label=f"Piste {track_id} — moyenne glissante"
        )

plt.axhline(50, linestyle="--", linewidth=1, label="Seuil expérimental = 50")

plt.xlabel("Temps (s)")
plt.ylabel("Score visuel")
plt.title("ADAS-FUSION X 2030 — Persistance temporelle")
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


### Étape 07.5.1 — Afficher et vérifier les résultats

In [ ]:

import pandas as pd
from pathlib import Path

RISK_DIR = Path(
    "/kaggle/working/ADAS_FUSION_X2030/07_risk_analysis"
)

scores = pd.read_csv(
    RISK_DIR / "visual_risk_temporal_scores.csv"
)

summary = pd.read_csv(
    RISK_DIR / "visual_risk_temporal_summary.csv"
)

print("========== BILAN TEMPOREL ==========")
print("Observations :", len(scores))
print("Pistes :", scores["track_id"].nunique())

print(
    "Observations avec score instantané >= 50 :",
    int((scores["visual_risk_score"] >= 50).sum())
)

print(
    "Observations avec score persistant :",
    int(scores["persistent_high_flag"].sum())
)

print("\n========== TOP 10 PAR PERSISTANCE ==========")

display(
    summary.head(10).round(4)
)

print("\n========== PISTES 19, 169 ET 136 ==========")

display(
    summary[
        summary["track_id"].isin([19, 169, 136])
    ].round(4)
)


## Étape suivante — 07.6 : visualisation et validation des événements prioritaires

In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

VIDEO_PATH = (
    "/kaggle/working/ADAS_FUSION_X2030/videos/"
    "ADAS_FUSION_X2030_Reference_180s.mp4"
)

OUTPUT_DIR = (
    "/kaggle/working/ADAS_FUSION_X2030/07_risk_analysis/"
    "07_6_visual_validation"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Instants à examiner : autour des événements déjà identifiés.
events = [
    {"track_id": 19,  "timestamp_s": 59.375},
    {"track_id": 19,  "timestamp_s": 61.7083},
    {"track_id": 169, "timestamp_s": 69.375},
    {"track_id": 136, "timestamp_s": 67.54},
]

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError(f"Impossible d'ouvrir la vidéo : {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"FPS : {fps}")
print(f"Nombre de frames : {frame_count}")
print(f"Durée : {frame_count / fps:.2f} s")

saved_images = []

for event in events:
    t = event["timestamp_s"]
    frame_idx = int(round(t * fps))

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        print(f"Échec de lecture à t={t:.3f} s")
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    filename = (
        f"track_{event['track_id']}_"
        f"t_{t:.3f}s.png"
    )
    path = os.path.join(OUTPUT_DIR, filename)

    plt.figure(figsize=(14, 7))
    plt.imshow(frame)
    plt.title(
        f"Piste {event['track_id']} — "
        f"t = {t:.3f} s — frame {frame_idx}"
    )
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()

    saved_images.append({
        "track_id": event["track_id"],
        "timestamp_s": t,
        "frame": frame_idx,
        "image_path": path,
    })

cap.release()

display(pd.DataFrame(saved_images))

print("\nImages enregistrées dans :")
print(OUTPUT_DIR)

# Affichage dans le notebook
for item in saved_images:
    image = plt.imread(item["image_path"])

    plt.figure(figsize=(14, 7))
    plt.imshow(image)
    plt.title(
        f"Piste {item['track_id']} — "
        f"{item['timestamp_s']:.3f} s"
    )
    plt.axis("off")
    plt.show()

## Étape 07.6 — Vérification des pistes sur les images

In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

BASE_DIR = "/kaggle/working/ADAS_FUSION_X2030"

VIDEO_PATH = (
    f"{BASE_DIR}/videos/"
    "ADAS_FUSION_X2030_Reference_180s.mp4"
)

TRACKING_PATH = (
    f"{BASE_DIR}/03_tracking/"
    "tracking_stabilized.csv"
)

RISK_PATH = (
    f"{BASE_DIR}/07_risk_analysis/"
    "visual_risk_temporal_scores.csv"
)

OUTPUT_DIR = (
    f"{BASE_DIR}/07_risk_analysis/"
    "07_6_visual_validation_annotated"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

tracking = pd.read_csv(TRACKING_PATH)
risk = pd.read_csv(RISK_PATH)

print("Colonnes du suivi :")
print(tracking.columns.tolist())

print("\nColonnes du score temporel :")
print(risk.columns.tolist())

# Vérifier les colonnes nécessaires.
required_tracking = {
    "frame", "track_id", "x1", "y1", "x2", "y2"
}
required_risk = {"frame", "track_id"}

if not required_tracking.issubset(tracking.columns):
    raise ValueError(
        f"Colonnes manquantes dans le suivi : "
        f"{required_tracking - set(tracking.columns)}"
    )

if not required_risk.issubset(risk.columns):
    raise ValueError(
        f"Colonnes manquantes dans le score : "
        f"{required_risk - set(risk.columns)}"
    )

# Instants à examiner.
events = [
    {"track_id": 19, "timestamp_s": 59.3750},
    {"track_id": 19, "timestamp_s": 61.7083},
    {"track_id": 169, "timestamp_s": 69.3750},
    {"track_id": 136, "timestamp_s": 67.5400},
]

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir la vidéo.")

fps = cap.get(cv2.CAP_PROP_FPS)
annotated_images = []

for event in events:
    t = event["timestamp_s"]
    target_id = event["track_id"]
    frame_idx = int(round(t * fps))

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()

    if not ok:
        print(f"Frame illisible : {frame_idx}")
        continue

    # Toutes les pistes présentes à cette frame.
    rows = tracking[tracking["frame"] == frame_idx]

    # Informations de score pour la piste cible.
    score_row = risk[
        (risk["frame"] == frame_idx) &
        (risk["track_id"] == target_id)
    ]

    found_target = False

    for _, row in rows.iterrows():
        x1 = int(round(row["x1"]))
        y1 = int(round(row["y1"]))
        x2 = int(round(row["x2"]))
        y2 = int(round(row["y2"]))

        track_id = int(row["track_id"])
        is_target = track_id == target_id

        # Couleurs OpenCV : BGR.
        color = (0, 255, 0) if is_target else (180, 180, 180)
        thickness = 3 if is_target else 1

        cv2.rectangle(
            frame, (x1, y1), (x2, y2),
            color, thickness
        )

        label = f"ID {track_id}"

        if "stable_class" in row.index:
            label += f" | {row['stable_class']}"

        cv2.putText(
            frame, label,
            (x1, max(25, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65 if is_target else 0.45,
            color, 2 if is_target else 1,
            cv2.LINE_AA
        )

        if is_target:
            found_target = True

    if not score_row.empty:
        score_text = (
            f"Score: "
            f"{score_row.iloc[0].get('visual_risk_score', float('nan')):.1f}"
        )
    else:
        score_text = "Score indisponible pour cette frame"

    cv2.putText(
        frame,
        f"t={t:.3f}s | frame={frame_idx} | "
        f"cible ID={target_id} | {score_text}",
        (25, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8, (0, 255, 255), 2, cv2.LINE_AA
    )

    if not found_target:
        cv2.putText(
            frame,
            f"ID {target_id} absent du suivi a cette frame",
            (25, 75),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8, (0, 0, 255), 2, cv2.LINE_AA
        )

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    filename = f"track_{target_id}_frame_{frame_idx}.png"
    output_path = os.path.join(OUTPUT_DIR, filename)

    cv2.imwrite(output_path, frame)

    annotated_images.append({
        "track_id": target_id,
        "timestamp_s": t,
        "frame": frame_idx,
        "target_found": found_target,
        "image_path": output_path,
        "image": frame_rgb,
    })

cap.release()

# Tableau récapitulatif.
display(pd.DataFrame([
    {k: v for k, v in item.items() if k != "image"}
    for item in annotated_images
]))

# Affichage des images annotées.
for item in annotated_images:
    plt.figure(figsize=(16, 8))
    plt.imshow(item["image"])
    plt.title(
        f"Piste {item['track_id']} — "
        f"{item['timestamp_s']:.3f} s"
    )
    plt.axis("off")
    plt.show()

print("\nImages annotées enregistrées dans :")
print(OUTPUT_DIR)

### Étape 07.6.1 — Diagnostiquer les numéros de frame

In [ ]:
import pandas as pd
import numpy as np
import os

BASE_DIR = "/kaggle/working/ADAS_FUSION_X2030"

tracking_path = (
    f"{BASE_DIR}/03_tracking/tracking_stabilized.csv"
)
risk_path = (
    f"{BASE_DIR}/07_risk_analysis/"
    "visual_risk_temporal_scores.csv"
)

tracking = pd.read_csv(tracking_path)
risk = pd.read_csv(risk_path)

events = [
    {"track_id": 19, "timestamp_s": 59.3750, "frame_video": 1425},
    {"track_id": 19, "timestamp_s": 61.7083, "frame_video": 1481},
    {"track_id": 169, "timestamp_s": 69.3750, "frame_video": 1665},
    {"track_id": 136, "timestamp_s": 67.5400, "frame_video": 1621},
]

print("===== DIAGNOSTIC GLOBAL =====")
print("Frames uniques dans tracking :", tracking["frame"].nunique())
print("Plage frames tracking :", tracking["frame"].min(),
      "à", tracking["frame"].max())
print("Plage temporelle tracking :",
      tracking["timestamp_s"].min(),
      "à", tracking["timestamp_s"].max())

print("\n===== DIAGNOSTIC DES PISTES =====")

for event in events:
    tid = event["track_id"]
    t = event["timestamp_s"]
    frame_video = event["frame_video"]

    track_rows = tracking[tracking["track_id"] == tid]

    print(f"\n--- Piste {tid}, t={t:.4f}s, frame vidéo={frame_video} ---")
    print("Nombre d'observations de cette piste :", len(track_rows))

    if track_rows.empty:
        print("Piste absente de tracking_stabilized.csv")
        continue

    print(
        "Frames de la piste :",
        track_rows["frame"].min(),
        "à",
        track_rows["frame"].max()
    )

    nearest_time = track_rows.iloc[
        (track_rows["timestamp_s"] - t).abs().argsort()[:3]
    ]

    print("\nObservations les plus proches dans le temps :")
    print(
        nearest_time[
            ["frame", "timestamp_s", "track_id",
             "stable_class", "x1", "y1", "x2", "y2"]
        ].to_string(index=False)
    )

    exact_frame = tracking[
        (tracking["frame"] == frame_video) &
        (tracking["track_id"] == tid)
    ]

    print("\nCorrespondance exacte frame + piste :", len(exact_frame))

    nearest_frame = tracking[
        tracking["frame"] == frame_video
    ]

    print("Nombre total de détections à cette frame :", len(nearest_frame))

print("\n===== COMPARAISON DES FRAMES DES DEUX FICHIERS =====")
print("Frames communes :", len(
    set(tracking["frame"].unique()) &
    set(risk["frame"].unique())
))
print("Frames absentes du suivi mais présentes dans le score :",
      len(set(risk["frame"].unique()) -
          set(tracking["frame"].unique())))

In [ ]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = "/kaggle/working/ADAS_FUSION_X2030"

VIDEO_PATH = (
    f"{BASE_DIR}/videos/"
    "ADAS_FUSION_X2030_Reference_180s.mp4"
)

TRACKING_PATH = (
    f"{BASE_DIR}/03_tracking/"
    "tracking_stabilized.csv"
)

RISK_PATH = (
    f"{BASE_DIR}/07_risk_analysis/"
    "visual_risk_temporal_scores.csv"
)

OUTPUT_DIR = (
    f"{BASE_DIR}/07_risk_analysis/"
    "07_6_visual_validation_corrected"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

tracking = pd.read_csv(TRACKING_PATH)
risk = pd.read_csv(RISK_PATH)

# Instants et identifiants à vérifier.
events = [
    {"track_id": 19, "timestamp_s": 59.3750},
    {"track_id": 19, "timestamp_s": 61.7083},
    {"track_id": 169, "timestamp_s": 69.3750},
    {"track_id": 136, "timestamp_s": 67.5417},
]

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir la vidéo.")

fps = cap.get(cv2.CAP_PROP_FPS)
results = []

for event in events:
    tid = event["track_id"]
    t = event["timestamp_s"]

    # Retrouver l'observation réelle par piste et timestamp.
    candidates = tracking[
        tracking["track_id"].eq(tid)
    ].copy()

    if candidates.empty:
        print(f"Piste {tid} absente du suivi.")
        continue

    candidates["time_error"] = (
        candidates["timestamp_s"] - t
    ).abs()

    target = candidates.sort_values("time_error").iloc[0]

    # Employer le timestamp associé à l'observation trouvée.
    actual_t = float(target["timestamp_s"])
    local_frame = int(target["frame"])

    # Le timestamp est global : on extrait la frame globale.
    global_frame = int(round(actual_t * fps))

    cap.set(cv2.CAP_PROP_POS_FRAMES, global_frame)
    ok, frame = cap.read()

    if not ok:
        print(f"Impossible de lire la frame {global_frame}")
        continue

    # Toutes les pistes détectées à cet instant.
    frame_rows = tracking[
        tracking["frame"].eq(local_frame)
    ]

    found_target = False

    for _, row in frame_rows.iterrows():
        x1 = int(round(row["x1"]))
        y1 = int(round(row["y1"]))
        x2 = int(round(row["x2"]))
        y2 = int(round(row["y2"]))

        current_id = int(row["track_id"])
        is_target = current_id == tid

        color = (0, 255, 0) if is_target else (180, 180, 180)
        thickness = 3 if is_target else 1

        cv2.rectangle(
            frame, (x1, y1), (x2, y2),
            color, thickness
        )

        label = f"ID {current_id}"

        if "stable_class" in row.index:
            label += f" | {row['stable_class']}"

        cv2.putText(
            frame, label,
            (x1, max(25, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65 if is_target else 0.45,
            color, 2 if is_target else 1,
            cv2.LINE_AA
        )

        if is_target:
            found_target = True

    # Score temporel correspondant à l'observation locale.
    score_candidates = risk[
        risk["track_id"].eq(tid)
        & risk["frame"].eq(local_frame)
    ]

    if not score_candidates.empty:
        score_row = score_candidates.iloc[0]

        score = score_row["visual_risk_score"]
        persistent = score_row["persistent_high_flag"]

        score_text = (
            f"Score visuel={score:.1f} | "
            f"Persistance={persistent}"
        )
    else:
        score_text = "Score temporel indisponible"

    cv2.putText(
        frame,
        f"t={actual_t:.3f}s | ID={tid} | {score_text}",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7, (0, 255, 255), 2, cv2.LINE_AA
    )

    filename = f"track_{tid}_frame_{global_frame}.png"
    output_path = os.path.join(OUTPUT_DIR, filename)

    cv2.imwrite(output_path, frame)

    results.append({
        "track_id": tid,
        "timestamp_s": actual_t,
        "frame_locale": local_frame,
        "frame_globale": global_frame,
        "target_found": found_target,
        "score_available": not score_candidates.empty,
        "image_path": output_path,
        "image": cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    })

cap.release()

summary = pd.DataFrame([
    {k: v for k, v in item.items() if k != "image"}
    for item in results
])

display(summary)

for item in results:
    plt.figure(figsize=(16, 8))
    plt.imshow(item["image"])
    plt.title(
        f"Piste {item['track_id']} — "
        f"{item['timestamp_s']:.3f} s"
    )
    plt.axis("off")
    plt.show()

print("Résultats enregistrés dans :", OUTPUT_DIR)

In [ ]:
import os
import json
import numpy as np
import pandas as pd

# ============================================================
# ADAS-FUSION X 2030 — ETAPE 08.1
# Moteur expérimental de décision basé sur des règles
# ============================================================

BASE_DIR = "/kaggle/working/ADAS_FUSION_X2030"

INPUT_PATH = (
    f"{BASE_DIR}/07_risk_analysis/"
    "visual_risk_temporal_scores.csv"
)

OUTPUT_DIR = f"{BASE_DIR}/08_decision_fusion"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = f"{OUTPUT_DIR}/adas_decisions_observations.csv"
SUMMARY_PATH = f"{OUTPUT_DIR}/adas_decision_summary.csv"
METADATA_PATH = f"{OUTPUT_DIR}/adas_decision_metadata.json"

# ------------------------------------------------------------
# 1. Charger les observations et contrôler les données
# ------------------------------------------------------------

df = pd.read_csv(INPUT_PATH)

required_columns = [
    "frame",
    "timestamp_s",
    "track_id",
    "stable_class",
    "confidence",
    "visual_risk_score",
    "persistent_high_flag",
    "track_quality_label",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(
        f"Colonnes nécessaires absentes du fichier : {missing}"
    )

print("========== DONNEES D'ENTREE ==========")
print(f"Observations : {len(df)}")
print(f"Pistes uniques : {df['track_id'].nunique()}")
print(f"Colonnes disponibles : {len(df.columns)}")

# ------------------------------------------------------------
# 2. Préparer les variables décisionnelles
# ------------------------------------------------------------

df["visual_risk_score"] = pd.to_numeric(
    df["visual_risk_score"], errors="coerce"
)

df["confidence"] = pd.to_numeric(
    df["confidence"], errors="coerce"
)

df["persistent_high_flag"] = (
    df["persistent_high_flag"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "1.0"])
)

df["track_quality_label"] = (
    df["track_quality_label"]
    .fillna("Inconnue")
    .astype(str)
)

df["stable_class"] = (
    df["stable_class"]
    .fillna("unknown")
    .astype(str)
)

# Ne pas prendre de décision si le score est manquant.
df["decision_data_valid"] = (
    df["visual_risk_score"].notna()
    & df["confidence"].notna()
)

# ------------------------------------------------------------
# 3. Définir les règles de décision
# ------------------------------------------------------------

def make_adas_decision(row):

    if not row["decision_data_valid"]:
        return pd.Series({
            "adas_level": -1,
            "adas_decision": "DONNEES_INSUFFISANTES",
            "decision_reason": "Score ou confiance indisponible"
        })

    score = float(row["visual_risk_score"])
    persistent = bool(row["persistent_high_flag"])
    quality = row["track_quality_label"]

    # Les pistes courtes ou limitées ne peuvent pas recevoir
    # le niveau de priorité maximal dans cette version.
    quality_sufficient = (
        quality == "Suffisante pour analyse exploratoire"
    )

    # Niveau 3 : signal visuel très élevé et persistant,
    # avec une piste suffisamment longue pour l'analyse.
    if score >= 75 and persistent and quality_sufficient:
        level = 3
        decision = "PRIORITE_ANALYSE"
        reason = (
            "Score très élevé, signal persistant, "
            "qualité suffisante pour analyse exploratoire"
        )

    # Niveau 2 : signal visuel élevé et persistant.
    elif score >= 50 and persistent:
        level = 2
        decision = "ALERTE_VISUELLE"
        reason = "Score élevé et signal persistant"

    # Niveau 2 également pour un pic très élevé non persistant :
    # on le signale sans le considérer comme un événement confirmé.
    elif score >= 75:
        level = 2
        decision = "ALERTE_VISUELLE"
        reason = "Pic de score très élevé, persistance non confirmée"

    # Niveau 1 : score intermédiaire ou signal persistant
    # dont le score instantané reste modéré.
    elif score >= 25 or persistent:
        level = 1
        decision = "SURVEILLANCE"
        reason = "Signal visuel intermédiaire ou persistance détectée"

    # Niveau 0 : aucun signal visuel prioritaire selon nos règles.
    else:
        level = 0
        decision = "FAIBLE_PRIORITE"
        reason = "Score visuel faible et absence de signal persistant"

    return pd.Series({
        "adas_level": level,
        "adas_decision": decision,
        "decision_reason": reason
    })


decision_columns = df.apply(make_adas_decision, axis=1)
df = pd.concat([df, decision_columns], axis=1)

# ------------------------------------------------------------
# 4. Exporter les décisions par observation
# ------------------------------------------------------------

df.to_csv(OUTPUT_PATH, index=False)

# ------------------------------------------------------------
# 5. Produire les statistiques globales
# ------------------------------------------------------------

summary = (
    df.groupby(
        ["adas_level", "adas_decision"],
        dropna=False
    )
    .agg(
        observations=("track_id", "size"),
        pistes=("track_id", "nunique"),
        score_moyen=("visual_risk_score", "mean"),
        score_maximal=("visual_risk_score", "max"),
    )
    .reset_index()
    .sort_values("adas_level")
)

summary.to_csv(SUMMARY_PATH, index=False)

metadata = {
    "project": "ADAS-FUSION X 2030",
    "module": "08.1 - Rule-based decision fusion",
    "input_file": INPUT_PATH,
    "output_file": OUTPUT_PATH,
    "summary_file": SUMMARY_PATH,
    "number_of_observations": int(len(df)),
    "number_of_tracks": int(df["track_id"].nunique()),
    "decision_levels": {
        "-1": "DONNEES_INSUFFISANTES",
        "0": "FAIBLE_PRIORITE",
        "1": "SURVEILLANCE",
        "2": "ALERTE_VISUELLE",
        "3": "PRIORITE_ANALYSE"
    },
    "limitations": [
        "Prototype expérimental fondé sur des règles heuristiques",
        "Le score visuel n'est pas une probabilité de collision",
        "Aucun TTC métrique calculé",
        "Aucune validation sur vérité terrain dans cette étape",
        "Aucune commande de freinage ou de direction"
    ]
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------
# 6. Afficher les résultats pour validation
# ------------------------------------------------------------

print("\n========== REPARTITION DES DECISIONS ==========")
display(summary)

print("\n========== EXEMPLES DE DECISIONS ==========")

columns_to_show = [
    "timestamp_s",
    "track_id",
    "stable_class",
    "visual_risk_score",
    "persistent_high_flag",
    "track_quality_label",
    "adas_level",
    "adas_decision",
    "decision_reason",
]

display(
    df[columns_to_show]
    .sort_values(
        ["adas_level", "visual_risk_score"],
        ascending=[False, False]
    )
    .head(15)
)

print("\n========== FICHIERS EXPORTES ==========")
print(OUTPUT_PATH)
print(SUMMARY_PATH)
print(METADATA_PATH)

In [ ]:
import os
import json
import numpy as np
import pandas as pd

BASE_DIR = "/kaggle/working/ADAS_FUSION_X2030"

INPUT_PATH = (
    f"{BASE_DIR}/08_decision_fusion/"
    "adas_decisions_observations.csv"
)

OUTPUT_DIR = f"{BASE_DIR}/08_decision_fusion"
OUTPUT_PATH = f"{OUTPUT_DIR}/adas_track_decision_summary.csv"
METADATA_PATH = f"{OUTPUT_DIR}/adas_track_decision_metadata.json"

df = pd.read_csv(INPUT_PATH)

# Vérification des colonnes nécessaires.
required = [
    "track_id",
    "timestamp_s",
    "adas_level",
    "visual_risk_score",
    "persistent_high_flag",
    "confidence",
    "track_quality_label",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Colonnes manquantes : {missing}")

# Convertir explicitement les variables numériques.
for col in [
    "timestamp_s",
    "adas_level",
    "visual_risk_score",
    "confidence",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Convertir les booléens CSV de façon robuste.
df["persistent_high_flag"] = (
    df["persistent_high_flag"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "1.0"])
)

# ----------------------------------------------------------
# Consolidation descriptive par piste
# ----------------------------------------------------------

def summarize_track(group):
    group = group.sort_values("timestamp_s")

    valid_levels = group["adas_level"].dropna()
    valid_scores = group["visual_risk_score"].dropna()

    n = len(group)

    high_level = group["adas_level"].isin([2, 3])
    priority_level = group["adas_level"].eq(3)

    duration = (
        group["timestamp_s"].max() -
        group["timestamp_s"].min()
    ) if n > 1 else 0.0

    # Niveau maximal observé, conservé comme indicateur descriptif.
    max_level = (
        int(valid_levels.max())
        if not valid_levels.empty else -1
    )

    if max_level == 3:
        provisional_decision = "PRIORITE_ANALYSE"
    elif max_level == 2:
        provisional_decision = "ALERTE_VISUELLE"
    elif max_level == 1:
        provisional_decision = "SURVEILLANCE"
    elif max_level == 0:
        provisional_decision = "FAIBLE_PRIORITE"
    else:
        provisional_decision = "DONNEES_INSUFFISANTES"

    return pd.Series({
        "observations": n,
        "debut_s": group["timestamp_s"].min(),
        "fin_s": group["timestamp_s"].max(),
        "duree_s": duration,
        "score_moyen": valid_scores.mean(),
        "score_maximal": valid_scores.max(),
        "niveau_maximal_observe": max_level,
        "observations_niveau_2_ou_3": int(high_level.sum()),
        "observations_niveau_3": int(priority_level.sum()),
        "proportion_niveau_2_ou_3_pct": 100 * high_level.mean(),
        "proportion_niveau_3_pct": 100 * priority_level.mean(),
        "observations_persistantes": int(
            group["persistent_high_flag"].sum()
        ),
        "confiance_moyenne": group["confidence"].mean(),
        "qualite_suivi": (
            group["track_quality_label"].mode().iloc[0]
            if not group["track_quality_label"].mode().empty
            else "Inconnue"
        ),
        "decision_provisoire": provisional_decision,
    })


track_summary = (
    df.groupby("track_id")
    .apply(summarize_track, include_groups=False)
    .reset_index()
)

track_summary = track_summary.sort_values(
    ["niveau_maximal_observe", "score_maximal"],
    ascending=[False, False]
)

track_summary.to_csv(OUTPUT_PATH, index=False)

metadata = {
    "project": "ADAS-FUSION X 2030",
    "module": "08.2 - Consolidation descriptive par piste",
    "input_file": INPUT_PATH,
    "output_file": OUTPUT_PATH,
    "number_of_observations": int(len(df)),
    "number_of_tracks": int(df["track_id"].nunique()),
    "interpretation": (
        "Le niveau maximal observé est un indicateur descriptif. "
        "Il ne constitue pas une validation de risque réel ni "
        "une décision de sécurité automobile."
    ),
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("========== BILAN PAR PISTE ==========")
print("Nombre de pistes :", len(track_summary))

display(track_summary.head(15))

print("\n========== PISTES PRIORITAIRES ==========")
display(
    track_summary[
        track_summary["niveau_maximal_observe"] >= 2
    ]
)

print("\n========== FICHIERS EXPORTES ==========")
print(OUTPUT_PATH)
print(METADATA_PATH)

In [ ]:
import os
import json
import numpy as np
import pandas as pd

BASE_DIR = "/kaggle/working/ADAS_FUSION_X2030"

INPUT_PATH = (
    f"{BASE_DIR}/08_decision_fusion/"
    "adas_track_decision_summary.csv"
)

OUTPUT_DIR = f"{BASE_DIR}/08_decision_fusion"

OUTPUT_PATH = (
    f"{OUTPUT_DIR}/adas_track_decision_consolidated.csv"
)

METADATA_PATH = (
    f"{OUTPUT_DIR}/adas_track_decision_consolidated_metadata.json"
)

df = pd.read_csv(INPUT_PATH)

# -------------------------------------------------------
# 1. Vérifier les données
# -------------------------------------------------------

required = [
    "track_id",
    "observations",
    "duree_s",
    "score_moyen",
    "score_maximal",
    "proportion_niveau_2_ou_3_pct",
    "proportion_niveau_3_pct",
    "proportion_niveau_2_ou_3_pct",
    "qualite_suivi",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Colonnes manquantes : {missing}")

# -------------------------------------------------------
# 2. Définir les règles consolidées
# -------------------------------------------------------

def consolidated_decision(row):

    score_mean = row["score_moyen"]
    score_max = row["score_maximal"]
    duration = row["duree_s"]
    high_pct = row["proportion_niveau_2_ou_3_pct"]
    quality = row["qualite_suivi"]

    quality_sufficient = (
        quality == "Suffisante pour analyse exploratoire"
    )

    # Niveau 3 :
    # signal élevé en moyenne, fréquent et suivi suffisamment long.
    if (
        score_mean >= 50
        and high_pct >= 60
        and duration >= 1.0
        and quality_sufficient
    ):
        return pd.Series({
            "niveau_consolide": 3,
            "decision_consolidee": "PRIORITE_ANALYSE",
            "motif_consolide": (
                "Score moyen élevé, signal fréquent, "
                "durée et qualité de suivi suffisantes"
            )
        })

    # Niveau 2 :
    # signal élevé récurrent ou score maximal très élevé,
    # sous réserve d'une durée minimale de suivi.
    if (
        duration >= 0.5
        and (
            (score_mean >= 35 and high_pct >= 30)
            or (score_max >= 75 and high_pct >= 20)
        )
    ):
        return pd.Series({
            "niveau_consolide": 2,
            "decision_consolidee": "ALERTE_VISUELLE",
            "motif_consolide": (
                "Signal élevé récurrent ou pic important "
                "sur une piste suffisamment longue"
            )
        })

    # Niveau 1 :
    # signal intermédiaire, ou pic important sur une piste courte.
    if score_mean >= 20 or score_max >= 50:
        return pd.Series({
            "niveau_consolide": 1,
            "decision_consolidee": "SURVEILLANCE",
            "motif_consolide": (
                "Signal intermédiaire ou pic à examiner, "
                "sans critères suffisants pour une alerte consolidée"
            )
        })

    # Niveau 0 :
    return pd.Series({
        "niveau_consolide": 0,
        "decision_consolidee": "FAIBLE_PRIORITE",
        "motif_consolide": (
            "Score global faible selon les règles expérimentales"
        )
    })


decision = df.apply(consolidated_decision, axis=1)
df = pd.concat([df, decision], axis=1)

# -------------------------------------------------------
# 3. Exporter les résultats
# -------------------------------------------------------

df = df.sort_values(
    ["niveau_consolide", "score_moyen"],
    ascending=[False, False]
)

df.to_csv(OUTPUT_PATH, index=False)

metadata = {
    "project": "ADAS-FUSION X 2030",
    "module": "08.3 - Consolidation expérimentale par piste",
    "input_file": INPUT_PATH,
    "output_file": OUTPUT_PATH,
    "number_of_tracks": int(len(df)),
    "thresholds_are_heuristic": True,
    "not_collision_probability": True,
    "not_validated_for_vehicle_control": True,
    "limitations": [
        "Seuils choisis expérimentalement",
        "Absence de vérité terrain de risque",
        "Score visuel non équivalent à un risque de collision",
        "Pas de distance métrique ni de TTC validé"
    ]
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# -------------------------------------------------------
# 4. Afficher les résultats
# -------------------------------------------------------

print("========== DECISION CONSOLIDEE PAR PISTE ==========")

display(
    df[
        [
            "track_id",
            "observations",
            "duree_s",
            "score_moyen",
            "score_maximal",
            "proportion_niveau_2_ou_3_pct",
            "qualite_suivi",
            "niveau_consolide",
            "decision_consolidee",
            "motif_consolide",
        ]
    ]
)

print("\n========== REPARTITION DES DECISIONS ==========")

display(
    df.groupby(
        ["niveau_consolide", "decision_consolidee"]
    ).agg(
        nombre_pistes=("track_id", "nunique"),
        score_moyen=("score_moyen", "mean"),
    ).reset_index()
)

print("\nFichier exporté :", OUTPUT_PATH)
print("Métadonnées :", METADATA_PATH)

In [ ]:
import os
import pandas as pd

BASE_DIR = "/kaggle/working/ADAS_FUSION_X2030"

TRACKING_PATH = (
    f"{BASE_DIR}/03_tracking/"
    "tracking_stabilized.csv"
)

OBSERVATION_DECISIONS_PATH = (
    f"{BASE_DIR}/08_decision_fusion/"
    "adas_decisions_observations.csv"
)

TRACK_DECISIONS_PATH = (
    f"{BASE_DIR}/08_decision_fusion/"
    "adas_track_decision_consolidated.csv"
)

tracking = pd.read_csv(TRACKING_PATH)
observations = pd.read_csv(OBSERVATION_DECISIONS_PATH)
track_decisions = pd.read_csv(TRACK_DECISIONS_PATH)

# Vérifier l'unicité de la décision consolidée par piste.
duplicates = track_decisions["track_id"].duplicated().sum()

print("========== CONTROLE D'INTEGRATION ==========")
print("Observations de suivi :", len(tracking))
print("Observations avec décision instantanée :", len(observations))
print("Pistes consolidées :", len(track_decisions))
print("Décisions consolidées dupliquées :", duplicates)

# Préparer la table des décisions à joindre.
decision_map = track_decisions[
    [
        "track_id",
        "niveau_consolide",
        "decision_consolidee",
        "score_moyen",
        "score_maximal",
        "duree_s",
    ]
].copy()

# Vérifier la correspondance des identifiants.
tracking_ids = set(tracking["track_id"].dropna().unique())
decision_ids = set(decision_map["track_id"].dropna().unique())

print("Pistes présentes dans le suivi :", len(tracking_ids))
print("Pistes présentes dans les décisions :", len(decision_ids))
print("Pistes sans décision consolidée :", len(tracking_ids - decision_ids))
print("Décisions sans piste de suivi :", len(decision_ids - tracking_ids))

# Jointure sans modifier les fichiers originaux.
merged = tracking.merge(
    decision_map,
    on="track_id",
    how="left",
    validate="many_to_one"
)

print("\nObservations associées à une décision :",
      merged["decision_consolidee"].notna().sum())

print("\nExemples de correspondance :")
display(
    merged[
        [
            "frame",
            "timestamp_s",
            "track_id",
            "stable_class",
            "niveau_consolide",
            "decision_consolidee",
        ]
    ].head(10)
)

print("\nRépartition des observations par décision consolidée :")
display(
    merged["decision_consolidee"]
    .value_counts(dropna=False)
    .rename_axis("decision_consolidee")
    .reset_index(name="observations")
)

In [ ]:
import os
import json
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ============================================================
# ADAS-FUSION X 2030 — ETAPE 08.5
# Visualisation intégrée des décisions
# ============================================================

BASE_DIR = "/kaggle/working/ADAS_FUSION_X2030"

VIDEO_PATH = (
    f"{BASE_DIR}/videos/"
    "ADAS_FUSION_X2030_Reference_180s.mp4"
)

TRACKING_PATH = (
    f"{BASE_DIR}/03_tracking/"
    "tracking_stabilized.csv"
)

SCORES_PATH = (
    f"{BASE_DIR}/07_risk_analysis/"
    "visual_risk_temporal_scores.csv"
)

DECISIONS_PATH = (
    f"{BASE_DIR}/08_decision_fusion/"
    "adas_track_decision_consolidated.csv"
)

OUTPUT_DIR = f"{BASE_DIR}/08_decision_fusion/video"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_VIDEO = f"{OUTPUT_DIR}/ADAS_FUSION_X2030_Decision.mp4"
OUTPUT_METADATA = f"{OUTPUT_DIR}/video_metadata.json"

# ------------------------------------------------------------
# 1. Charger les données
# ------------------------------------------------------------

tracking = pd.read_csv(TRACKING_PATH)
scores = pd.read_csv(SCORES_PATH)
decisions = pd.read_csv(DECISIONS_PATH)

# Assurer un type numérique cohérent pour les clés.
for frame in [tracking, scores, decisions]:
    frame["track_id"] = pd.to_numeric(
        frame["track_id"], errors="raise"
    ).astype(int)

tracking["frame"] = pd.to_numeric(
    tracking["frame"], errors="raise"
).astype(int)

scores["frame"] = pd.to_numeric(
    scores["frame"], errors="raise"
).astype(int)

# Vérifier les colonnes nécessaires.
tracking_required = [
    "frame", "timestamp_s", "track_id",
    "stable_class", "x1", "y1", "x2", "y2"
]
score_required = [
    "frame", "track_id", "visual_risk_score"
]
decision_required = [
    "track_id", "niveau_consolide", "decision_consolidee"
]

for name, data, required in [
    ("tracking", tracking, tracking_required),
    ("scores", scores, score_required),
    ("decisions", decisions, decision_required),
]:
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(
            f"Colonnes manquantes dans {name} : {missing}"
        )

# Une seule décision consolidée par piste.
if decisions["track_id"].duplicated().any():
    raise ValueError(
        "Plusieurs décisions consolidées existent pour une même piste."
    )

# ------------------------------------------------------------
# 2. Joindre les scores et les décisions
# ------------------------------------------------------------

# Contrôler l'unicité de la clé frame + track_id.
if tracking.duplicated(["frame", "track_id"]).any():
    raise ValueError(
        "Doublons détectés pour les couples frame + track_id."
    )

if scores.duplicated(["frame", "track_id"]).any():
    raise ValueError(
        "Doublons détectés dans les scores pour frame + track_id."
    )

merged = tracking.merge(
    scores[["frame", "track_id", "visual_risk_score"]],
    on=["frame", "track_id"],
    how="left",
    validate="one_to_one"
)

merged = merged.merge(
    decisions[
        ["track_id", "niveau_consolide", "decision_consolidee"]
    ],
    on="track_id",
    how="left",
    validate="many_to_one"
)

print("Observations de suivi :", len(tracking))
print("Observations après fusion :", len(merged))
print(
    "Scores manquants :",
    int(merged["visual_risk_score"].isna().sum())
)
print(
    "Décisions manquantes :",
    int(merged["decision_consolidee"].isna().sum())
)

# ------------------------------------------------------------
# 3. Définir les styles visuels
# ------------------------------------------------------------

# Couleurs OpenCV en BGR.
COLORS = {
    0: (80, 180, 80),      # Faible priorité
    1: (0, 200, 255),      # Surveillance
    2: (0, 140, 255),      # Alerte visuelle
    3: (0, 0, 255),        # Priorité d'analyse
}

LABELS = {
    0: "FAIBLE PRIORITE",
    1: "SURVEILLANCE",
    2: "ALERTE VISUELLE",
    3: "PRIORITE ANALYSE",
}

# ------------------------------------------------------------
# 4. Ouvrir la vidéo et définir le segment
# ------------------------------------------------------------

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(
        f"Vidéo introuvable : {VIDEO_PATH}"
    )

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir la vidéo source.")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

start_s = 55.0
end_s = 70.0

start_frame = int(round(start_s * fps))
end_frame = min(
    int(round(end_s * fps)),
    total_frames
)

# mp4v : codec généralement disponible dans OpenCV.
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError(
        "Impossible de créer la vidéo de sortie."
    )

# Index rapide des observations par frame.
frame_groups = {
    int(frame_id): group
    for frame_id, group in merged.groupby("frame")
}

# ------------------------------------------------------------
# 5. Générer les annotations
# ------------------------------------------------------------

frames_written = 0
frames_with_tracks = 0
observations_drawn = 0

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

for global_frame in tqdm(
    range(start_frame, end_frame),
    desc="Génération vidéo ADAS"
):
    ret, frame = cap.read()

    if not ret:
        print(
            f"Arrêt : lecture impossible à la frame {global_frame}"
        )
        break

    # Dans les données de suivi, la frame 1 correspond à
    # la première frame du segment de 15 secondes.
    local_frame = global_frame - start_frame + 1

    group = frame_groups.get(local_frame)

    if group is not None and len(group) > 0:
        frames_with_tracks += 1

        for _, row in group.iterrows():

            level_value = row["niveau_consolide"]

            if pd.isna(level_value):
                continue

            level = int(level_value)
            color = COLORS.get(level, (200, 200, 200))

            x1 = int(np.clip(round(row["x1"]), 0, width - 1))
            y1 = int(np.clip(round(row["y1"]), 0, height - 1))
            x2 = int(np.clip(round(row["x2"]), 0, width - 1))
            y2 = int(np.clip(round(row["y2"]), 0, height - 1))

            if x2 <= x1 or y2 <= y1:
                continue

            track_id = int(row["track_id"])
            class_name = str(row["stable_class"])

            score = row["visual_risk_score"]

            if pd.isna(score):
                score_text = "Score indisponible"
            else:
                score_text = f"Score visuel: {score:.1f}/100"

            label = (
                f"ID {track_id} | {class_name} | "
                f"{LABELS[level]}"
            )

            # Boîte englobante.
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                color,
                3
            )

            # Étiquette principale.
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.62
            thickness = 2

            (text_w, text_h), baseline = cv2.getTextSize(
                label, font, font_scale, thickness
            )

            label_y = max(y1 - 10, text_h + 8)

            cv2.rectangle(
                frame,
                (x1, label_y - text_h - 8),
                (min(width - 1, x1 + text_w + 8), label_y + baseline),
                color,
                -1
            )

            cv2.putText(
                frame,
                label,
                (x1 + 4, label_y - 4),
                font,
                font_scale,
                (255, 255, 255),
                thickness,
                cv2.LINE_AA
            )

            # Score visuel sur une deuxième ligne.
            cv2.putText(
                frame,
                score_text,
                (x1, min(height - 8, y2 + 24)),
                font,
                0.58,
                color,
                2,
                cv2.LINE_AA
            )

            observations_drawn += 1

    # Bandeau supérieur.
    overlay = frame.copy()
    cv2.rectangle(
        overlay,
        (0, 0),
        (width, 92),
        (20, 20, 20),
        -1
    )
    frame = cv2.addWeighted(overlay, 0.72, frame, 0.28, 0)

    elapsed_s = global_frame / fps

    cv2.putText(
        frame,
        "ADAS-FUSION X 2030 | PROTOTYPE EXPERIMENTAL",
        (24, 32),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.82,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    cv2.putText(
        frame,
        f"Temps video: {elapsed_s:.2f} s | FPS source: {fps:.1f}",
        (24, 66),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.68,
        (230, 230, 230),
        2,
        cv2.LINE_AA
    )

    # Légende.
    legend_y = height - 28
    legend_items = [
        (0, "Faible"),
        (1, "Surveillance"),
        (2, "Alerte"),
        (3, "Priorite analyse"),
    ]

    x_cursor = 20

    for level, legend_text in legend_items:
        color = COLORS[level]

        cv2.rectangle(
            frame,
            (x_cursor, legend_y - 15),
            (x_cursor + 18, legend_y + 3),
            color,
            -1
        )

        cv2.putText(
            frame,
            legend_text,
            (x_cursor + 25, legend_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.52,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )

        x_cursor += 220

    # Avertissement méthodologique permanent.
    cv2.putText(
        frame,
        "Score visuel heuristique - ne represente pas une probabilite de collision",
        (20, height - 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.52,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )

    writer.write(frame)
    frames_written += 1

cap.release()
writer.release()

# ------------------------------------------------------------
# 6. Vérifier et exporter les métadonnées
# ------------------------------------------------------------

metadata = {
    "project": "ADAS-FUSION X 2030",
    "module": "08.5 - Integrated decision visualization",
    "source_video": VIDEO_PATH,
    "output_video": OUTPUT_VIDEO,
    "source_fps": float(fps),
    "source_width": width,
    "source_height": height,
    "source_total_frames": total_frames,
    "segment_start_s": start_s,
    "segment_end_s": end_s,
    "frames_written": frames_written,
    "frames_with_tracks": frames_with_tracks,
    "observations_drawn": observations_drawn,
    "decision_source": DECISIONS_PATH,
    "score_source": SCORES_PATH,
    "interpretation_limit": (
        "Prototype visuel expérimental. Le score ne constitue "
        "pas une probabilité de collision et aucune commande "
        "de conduite n'est générée."
    ),
}

with open(OUTPUT_METADATA, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\n========== RESULTAT VIDEO ADAS ==========")
print("Frames écrites :", frames_written)
print("Frames avec pistes :", frames_with_tracks)
print("Observations dessinées :", observations_drawn)
print("Vidéo :", OUTPUT_VIDEO)
print("Métadonnées :", OUTPUT_METADATA)

if os.path.exists(OUTPUT_VIDEO):
    print(
        "Taille vidéo (Mo) :",
        round(os.path.getsize(OUTPUT_VIDEO) / (1024 ** 2), 2)
    )

# Contrôle rapide de la vidéo générée.
check = cv2.VideoCapture(OUTPUT_VIDEO)
print("Vidéo générée lisible :", check.isOpened())
print(
    "Nombre de frames en sortie :",
    int(check.get(cv2.CAP_PROP_FRAME_COUNT))
)
check.release()

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

video_path = Path(
    "/kaggle/working/ADAS_FUSION_X2030/"
    "08_decision_fusion/video/ADAS_FUSION_X2030_Decision.mp4"
)

assert video_path.exists(), f"Vidéo introuvable : {video_path}"

cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir la vidéo exportée.")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Examiner une image située au milieu de l'extrait.
target_frame = total_frames // 2
cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)

success, frame = cap.read()
cap.release()

if not success:
    raise RuntimeError("Impossible de lire l'image sélectionnée.")

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

print(f"Vidéo : {video_path.name}")
print(f"FPS : {fps:.2f}")
print(f"Nombre d'images : {total_frames}")
print(f"Image examinée : {target_frame}")

plt.figure(figsize=(16, 9))
plt.imshow(frame_rgb)
plt.axis("off")
plt.title("Audit visuel — vidéo de décision ADAS")
plt.show()

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/working/ADAS_FUSION_X2030")
DECISION_DIR = BASE / "08_decision_fusion"

path = DECISION_DIR / "adas_track_decision_consolidated.csv"
df = pd.read_csv(path)

print("========== AUDIT DES DÉCISIONS ADAS ==========")
print(f"Nombre de pistes : {len(df)}")

# Vérification des colonnes disponibles
print("\nColonnes disponibles :")
print(df.columns.tolist())

# Distribution des décisions
decision_col = "decision_consolidee"

if decision_col not in df.columns:
    candidates = [
        c for c in df.columns
        if "decision" in c.lower() or "niveau" in c.lower()
    ]
    print("\nColonnes candidates pour les décisions :", candidates)
else:
    print("\nDistribution des décisions :")
    print(df[decision_col].value_counts(dropna=False))

# Recherche des valeurs manquantes
print("\nValeurs manquantes par colonne :")
print(df.isna().sum())

# Affichage des pistes prioritaires
priority_cols = [
    c for c in df.columns
    if c in [
        "track_id",
        "duree_s",
        "score_moyen",
        "score_max",
        "decision_consolidee",
        "pourcentage_niveaux_2_3"
    ]
]

print("\nPistes à examiner :")
print(df[priority_cols].head(10).to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/working/ADAS_FUSION_X2030")
path = BASE / "08_decision_fusion" / "adas_track_decision_consolidated.csv"

df = pd.read_csv(path)

# Vérification des identifiants
assert df["track_id"].notna().all(), "Identifiants manquants"
assert df["track_id"].is_unique, "Plusieurs décisions pour un même identifiant"

# Vérification des variables numériques essentielles
numeric_cols = [
    "observations",
    "duree_s",
    "score_moyen",
    "score_maximal",
    "proportion_niveau_2_ou_3_pct",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

assert df[numeric_cols].notna().all().all(), \
    "Des valeurs numériques sont manquantes ou invalides"

assert df["duree_s"].ge(0).all(), "Durée négative détectée"
assert df["score_moyen"].between(0, 100).all(), "Score moyen hors limites"
assert df["score_maximal"].between(0, 100).all(), "Score maximal hors limites"
assert df["proportion_niveau_2_ou_3_pct"].between(0, 100).all(), \
    "Pourcentage hors limites"

print("========== CONTRÔLES STRUCTURELS ==========")
print("Identifiants uniques :", df["track_id"].nunique())
print("Valeurs numériques valides :", df[numeric_cols].notna().all().all())
print("Scores dans [0, 100] :", df["score_moyen"].between(0, 100).all()
      and df["score_maximal"].between(0, 100).all())
print("Pourcentages dans [0, 100] :",
      df["proportion_niveau_2_ou_3_pct"].between(0, 100).all())

print("\n========== PISTES À AUDITER ==========")

ids = [19, 169, 3, 136, 1]

cols = [
    "track_id",
    "observations",
    "duree_s",
    "score_moyen",
    "score_maximal",
    "proportion_niveau_2_ou_3_pct",
    "qualite_suivi",
    "niveau_consolide",
    "decision_consolidee",
    "motif_consolide",
]

print(
    df.loc[df["track_id"].isin(ids), cols]
      .sort_values("track_id")
      .to_string(index=False)
)

print("\n========== AUDIT TERMINÉ ==========")

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/working/ADAS_FUSION_X2030")
INPUT = (
    BASE / "08_decision_fusion"
    / "adas_track_decision_consolidated.csv"
)

OUTPUT_DIR = BASE / "08_decision_fusion" / "sensitivity_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT)

# Conversion et vérification des variables nécessaires.
required = [
    "track_id",
    "duree_s",
    "score_moyen",
    "score_maximal",
    "proportion_niveau_2_ou_3_pct",
    "qualite_suivi",
    "decision_consolidee",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Colonnes manquantes : {missing}")

for col in [
    "duree_s",
    "score_moyen",
    "score_maximal",
    "proportion_niveau_2_ou_3_pct",
]:
    df[col] = pd.to_numeric(df[col], errors="raise")

assert df["track_id"].is_unique
assert df[required].notna().all().all()

# Reproduction des règles de consolidation initiales.
def classify(row, delta=0):
    mean = row["score_moyen"]
    maximum = row["score_maximal"]
    high_pct = row["proportion_niveau_2_ou_3_pct"]
    duration = row["duree_s"]
    quality = row["qualite_suivi"]

    # Variation symétrique des seuils numériques.
    if (
        mean >= 50 + delta
        and high_pct >= 60
        and duration >= 1.0
        and quality == "Suffisante pour analyse exploratoire"
    ):
        return "PRIORITE_ANALYSE"

    if (
        duration >= 0.5
        and (
            (
                mean >= 35 + delta
                and high_pct >= 30
            )
            or (
                maximum >= 75 + delta
                and high_pct >= 20
            )
        )
    ):
        return "ALERTE_VISUELLE"

    if mean >= 20 + delta or maximum >= 50 + delta:
        return "SURVEILLANCE"

    return "FAIBLE_PRIORITE"


# Sensibilité à des variations de seuils de score.
scenarios = {
    "seuils_minus_5": -5,
    "seuils_initiaux": 0,
    "seuils_plus_5": 5,
    "seuils_plus_10": 10,
}

results = df[["track_id", "decision_consolidee"]].copy()

for name, delta in scenarios.items():
    results[name] = df.apply(
        lambda row: classify(row, delta=delta),
        axis=1,
    )

# Comparaison des décisions à la référence enregistrée.
for name in scenarios:
    results[f"{name}_identique_reference"] = (
        results[name] == results["decision_consolidee"]
    )

summary = []

for name in scenarios:
    summary.append({
        "scenario": name,
        "variation_seuils": scenarios[name],
        "nombre_pistes": len(results),
        "decisions_identiques_reference": int(
            results[f"{name}_identique_reference"].sum()
        ),
        "decisions_modifiees": int(
            (~results[f"{name}_identique_reference"]).sum()
        ),
        "accord_reference_pct": round(
            100 * results[f"{name}_identique_reference"].mean(), 2
        ),
    })

summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    OUTPUT_DIR / "sensitivity_summary.csv",
    index=False,
)
results.to_csv(
    OUTPUT_DIR / "sensitivity_by_track.csv",
    index=False,
)

print("========== AUDIT DE SENSIBILITÉ ==========")
print(summary_df.to_string(index=False))

print("\n========== PISTES DONT LA DÉCISION CHANGE ==========")

for name in ["seuils_minus_5", "seuils_plus_5", "seuils_plus_10"]:
    changed = results[
        results[name] != results["decision_consolidee"]
    ]

    print(f"\n{name} : {len(changed)} piste(s) modifiée(s)")

    if not changed.empty:
        print(
            changed[
                ["track_id", "decision_consolidee", name]
            ].to_string(index=False)
        )

print(f"\nRésultats enregistrés dans : {OUTPUT_DIR}")

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/working/ADAS_FUSION_X2030")
DECISION_DIR = BASE / "08_decision_fusion"

decisions = pd.read_csv(
    DECISION_DIR / "adas_track_decision_consolidated.csv"
)

sensitivity = pd.read_csv(
    DECISION_DIR / "sensitivity_audit" / "sensitivity_by_track.csv"
)

# 1. Contrôle de la couverture des pistes
assert decisions["track_id"].is_unique
assert set(decisions["track_id"]) == set(sensitivity["track_id"])

# 2. Vérification des justifications
assert decisions["motif_consolide"].notna().all()
assert decisions["motif_consolide"].astype(str).str.strip().ne("").all()

# 3. Vérification de la cohérence niveau / décision
mapping = {
    0: "FAIBLE_PRIORITE",
    1: "SURVEILLANCE",
    2: "ALERTE_VISUELLE",
    3: "PRIORITE_ANALYSE",
}

decisions["decision_attendue"] = decisions["niveau_consolide"].map(mapping)

decisions["coherent_niveau_decision"] = (
    decisions["decision_consolidee"] == decisions["decision_attendue"]
)

# 4. Sensibilité à +10 points
sensitivity["change_plus_10"] = (
    sensitivity["seuils_plus_10"]
    != sensitivity["decision_consolidee"]
)

print("========== AUDIT DE COHÉRENCE ==========")
print("Nombre de pistes :", len(decisions))
print("Justifications renseignées :",
      decisions["motif_consolide"].notna().sum())
print("Niveaux cohérents avec les décisions :",
      int(decisions["coherent_niveau_decision"].sum()), "/", len(decisions))
print("Pistes modifiées avec +10 points :",
      int(sensitivity["change_plus_10"].sum()))

print("\n========== DISTRIBUTION PAR NIVEAU ==========")
print(
    decisions.groupby(
        ["niveau_consolide", "decision_consolidee"],
        dropna=False
    ).size().rename("nombre_pistes").reset_index().to_string(index=False)
)

print("\n========== PISTES SENSIBLES ==========")
sensitive_ids = sensitivity.loc[
    sensitivity["change_plus_10"], "track_id"
]

cols = [
    "track_id",
    "score_moyen",
    "score_maximal",
    "proportion_niveau_2_ou_3_pct",
    "decision_consolidee",
    "motif_consolide",
]

print(
    decisions.loc[decisions["track_id"].isin(sensitive_ids), cols]
    .to_string(index=False)
)

print("\n========== FIN DE L'AUDIT ==========")

In [ ]:

# ================================================================
# ADAS-FUSION X 2030 — FULL VIDEO TRACKING & RESEARCH DASHBOARD
# Détection YOLO + suivi ByteTrack + vitesse apparente + vidéo
# Environnement cible : Kaggle / Tesla T4
#
# IMPORTANT :
# - Vitesse en pixels/s, PAS en km/h.
# - Score de risque visuel heuristique, PAS une probabilité.
# - Les résultats ne constituent pas une validation de sécurité.
# ================================================================

import os
import cv2
import json
import time
import math
import shutil
import warnings
import numpy as np
import pandas as pd
import torch
import ultralytics

from pathlib import Path
from collections import defaultdict, deque
from ultralytics import YOLO

warnings.filterwarnings("ignore")

# ------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------
BASE = Path("/kaggle/working/ADAS_FUSION_X2030")
VIDEO = BASE / "videos" / "ADAS_FUSION_X2030_Reference_180s.mp4"

OUT = BASE / "09_full_video_analysis"
DATA = OUT / "data"
VIDEO_OUT = OUT / "video"
FIGURES = OUT / "figures"

for folder in [DATA, VIDEO_OUT, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)

TRACKING_CSV = DATA / "tracking_full.csv"
SPEED_CSV = DATA / "vehicle_speeds_apparent.csv"
DECISIONS_CSV = DATA / "adas_decisions.csv"
VIDEO_FINAL = VIDEO_OUT / "ADAS_FUSION_X2030_FULL_DASHBOARD.mp4"
METADATA_JSON = OUT / "metadata.json"
LOG_FILE = OUT / "execution_log.txt"

MODEL_NAME = "yolo26n.pt"
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.45

# Réglage du traitement
DEVICE = 0 if torch.cuda.is_available() else "cpu"
HALF_PRECISION = bool(torch.cuda.is_available())

# Catégories routières COCO pertinentes
VEHICLE_CLASSES = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck",
    9: "traffic_light",
    11: "stop_sign"
}

print("=" * 72)
print("ADAS-FUSION X 2030 — ANALYSE INTÉGRALE")
print("=" * 72)
print("Vidéo :", VIDEO)
print("GPU :", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")
print("Ultralytics :", ultralytics.__version__)

if not VIDEO.exists():
    raise FileNotFoundError(f"Vidéo source introuvable : {VIDEO}")

# ------------------------------------------------
# 2. MÉTADONNÉES VIDÉO
# ------------------------------------------------
cap = cv2.VideoCapture(str(VIDEO))

if not cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir la vidéo source.")

FPS = cap.get(cv2.CAP_PROP_FPS)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION = N_FRAMES / FPS

if FPS <= 0 or N_FRAMES <= 0:
    cap.release()
    raise RuntimeError("Métadonnées vidéo invalides.")

print(f"Résolution : {WIDTH} x {HEIGHT}")
print(f"FPS source : {FPS:.3f}")
print(f"Images : {N_FRAMES}")
print(f"Durée : {DURATION:.2f} s")

# ------------------------------------------------
# 3. MODÈLE YOLO
# ------------------------------------------------
print("\nChargement du modèle :", MODEL_NAME)
model = YOLO(MODEL_NAME)

# Le suivi utilise ByteTrack. persist=True conserve les identifiants
# entre les images successives.
#
# On traite une image à la fois pour limiter l'usage mémoire.
# Les observations sont sauvegardées périodiquement.
# ------------------------------------------------

rows = []
track_history = defaultdict(lambda: deque(maxlen=30))
track_class_history = defaultdict(lambda: deque(maxlen=15))

# Les seuils ci-dessous ne sont pas des seuils de sécurité automobile.
RISK_THRESHOLDS = {
    "surveillance": 25.0,
    "alerte_visuelle": 50.0,
    "priorite_analyse": 75.0
}

# ------------------------------------------------
# 4. OUTIL : RISQUE VISUEL EXPLORATOIRE
# ------------------------------------------------
def calculate_visual_score(box, image_width, image_height,
                           apparent_speed_px_s):
    """
    Score heuristique entre 0 et 100.
    Il combine taille apparente, position verticale et mouvement apparent.
    Ce score n'est pas une probabilité de collision.
    """
    x1, y1, x2, y2 = box

    bw = max(0.0, x2 - x1)
    bh = max(0.0, y2 - y1)

    area_ratio = (bw * bh) / float(image_width * image_height)
    center_y_norm = ((y1 + y2) / 2.0) / image_height

    # Fonctions bornées, choisies comme heuristiques exploratoires.
    score_size = np.clip(area_ratio / 0.12, 0, 1)
    score_vertical = np.clip((center_y_norm - 0.35) / 0.60, 0, 1)
    score_motion = np.clip(apparent_speed_px_s / 600.0, 0, 1)

    score = 100.0 * (
        0.50 * score_size +
        0.30 * score_vertical +
        0.20 * score_motion
    )

    return float(np.clip(score, 0, 100))


def decision_from_score(score):
    if score >= RISK_THRESHOLDS["priorite_analyse"]:
        return 3, "PRIORITE_ANALYSE"
    if score >= RISK_THRESHOLDS["alerte_visuelle"]:
        return 2, "ALERTE_VISUELLE"
    if score >= RISK_THRESHOLDS["surveillance"]:
        return 1, "SURVEILLANCE"
    return 0, "FAIBLE_PRIORITE"


# ------------------------------------------------
# 5. TABLEAU DE BORD VIDÉO
# ------------------------------------------------
# La vidéo est produite en 1920x1080, avec une bande d'information
# sombre en haut et une bande de statut en bas.
# Les boîtes sont tracées directement sur l'image.
# ------------------------------------------------

DASHBOARD_HEIGHT = 150
FOOTER_HEIGHT = 48
OUT_WIDTH = WIDTH
OUT_HEIGHT = HEIGHT + DASHBOARD_HEIGHT + FOOTER_HEIGHT

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    str(VIDEO_FINAL), fourcc, FPS, (OUT_WIDTH, OUT_HEIGHT)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Impossible de créer la vidéo de sortie.")

start_time = time.time()
last_checkpoint = start_time
frames_processed = 0
frames_with_tracks = 0
total_observations = 0

# On sauvegarde régulièrement les données pour limiter les pertes
# en cas d'interruption de session.
CHECKPOINT_INTERVAL = 240

print("\nDébut du traitement intégral...")

# ------------------------------------------------
# 6. TRAITEMENT DE TOUTES LES IMAGES
# ------------------------------------------------
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame_idx = frames_processed
        timestamp_s = frame_idx / FPS

        # Suivi persistant ByteTrack
        result = model.track(
            source=frame,
            persist=True,
            tracker="bytetrack.yaml",
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            device=DEVICE,
            half=HALF_PRECISION,
            verbose=False
        )[0]

        annotated = frame.copy()
        current_tracks = []

        if result.boxes is not None and len(result.boxes) > 0:
            boxes_xyxy = result.boxes.xyxy.cpu().numpy()
            confidences = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy().astype(int)

            if result.boxes.id is not None:
                ids = result.boxes.id.cpu().numpy().astype(int)
            else:
                ids = np.full(len(boxes_xyxy), -1, dtype=int)

            for box, conf, cls_id, track_id in zip(
                boxes_xyxy, confidences, classes, ids
            ):
                if cls_id not in VEHICLE_CLASSES:
                    continue

                x1, y1, x2, y2 = map(float, box)
                cx = (x1 + x2) / 2.0
                cy = (y1 + y2) / 2.0
                bw = max(0.0, x2 - x1)
                bh = max(0.0, y2 - y1)

                # Vitesse apparente du centre de boîte en pixels/s.
                apparent_speed = 0.0
                vx_px_s = 0.0
                vy_px_s = 0.0

                if track_id >= 0:
                    hist = track_history[track_id]

                    if len(hist) > 0:
                        old_t, old_cx, old_cy = hist[-1]
                        dt = timestamp_s - old_t

                        if dt > 0:
                            vx_px_s = (cx - old_cx) / dt
                            vy_px_s = (cy - old_cy) / dt
                            apparent_speed = math.hypot(vx_px_s, vy_px_s)

                    hist.append((timestamp_s, cx, cy))
                    track_class_history[track_id].append(cls_id)

                class_name = VEHICLE_CLASSES[cls_id]

                # Score heuristique visuel
                risk_score = calculate_visual_score(
                    box, WIDTH, HEIGHT, apparent_speed
                )
                decision_level, decision = decision_from_score(risk_score)

                # Couleurs BGR selon la décision
                colors = {
                    0: (100, 200, 100),
                    1: (0, 220, 255),
                    2: (0, 140, 255),
                    3: (0, 0, 255)
                }
                color = colors[decision_level]

                # Boîte englobante et identité de piste
                ix1, iy1, ix2, iy2 = map(int, [x1, y1, x2, y2])
                cv2.rectangle(
                    annotated, (ix1, iy1), (ix2, iy2), color, 2
                )

                if track_id >= 0:
                    label = (
                        f"ID {track_id} | {class_name} | "
                        f"{apparent_speed:.0f}px/s | R:{risk_score:.0f}"
                    )
                else:
                    label = f"{class_name} | non suivi"

                label_y = max(22, iy1 - 8)
                cv2.putText(
                    annotated, label, (ix1, label_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52,
                    color, 2, cv2.LINE_AA
                )

                # Historique des positions : trajectoire image-plane
                if track_id >= 0:
                    hist_points = list(track_history[track_id])
                    for j in range(1, len(hist_points)):
                        p1 = tuple(map(int, hist_points[j - 1][1:]))
                        p2 = tuple(map(int, hist_points[j][1:]))
                        cv2.line(annotated, p1, p2, color, 2)

                row = {
                    "frame": frame_idx,
                    "timestamp_s": timestamp_s,
                    "track_id": int(track_id),
                    "class": class_name,
                    "class_id": int(cls_id),
                    "confidence": float(conf),
                    "x1": x1, "y1": y1,
                    "x2": x2, "y2": y2,
                    "center_x": cx, "center_y": cy,
                    "box_width": bw, "box_height": bh,
                    "box_area_ratio": (bw * bh) / (WIDTH * HEIGHT),
                    "vx_px_s": vx_px_s,
                    "vy_px_s": vy_px_s,
                    "apparent_speed_px_s": apparent_speed,
                    "visual_risk_score": risk_score,
                    "decision_level": decision_level,
                    "decision": decision
                }
                rows.append(row)
                current_tracks.append(row)
                total_observations += 1

        if current_tracks:
            frames_with_tracks += 1

        # ----------------------------
        # Bandeau supérieur
        # ----------------------------
        header = np.zeros(
            (DASHBOARD_HEIGHT, WIDTH, 3), dtype=np.uint8
        )
        header[:] = (22, 25, 31)

        elapsed = time.time() - start_time
        fps_proc = (frames_processed + 1) / max(elapsed, 1e-6)
        progress = 100.0 * (frames_processed + 1) / N_FRAMES

        cv2.putText(
            header, "ADAS-FUSION X 2030 | FULL VIDEO ANALYSIS",
            (28, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.88,
            (245, 245, 245), 2, cv2.LINE_AA
        )

        stats = [
            f"TIME {timestamp_s:06.2f}s / {DURATION:.0f}s",
            f"PROGRESS {progress:5.1f}%",
            f"TRACKS {len(set(r['track_id'] for r in rows if r['track_id'] >= 0))}",
            f"FPS PROCESS {fps_proc:.2f}",
        ]

        for i, txt in enumerate(stats):
            x = 28 + i * 350
            cv2.putText(
                header, txt, (x, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.62,
                (100, 220, 255), 2, cv2.LINE_AA
            )

        # Liste des pistes prioritaires observées sur l'image courante
        current_tracks_sorted = sorted(
            current_tracks,
            key=lambda z: z["visual_risk_score"],
            reverse=True
        )[:4]

        for i, item in enumerate(current_tracks_sorted):
            txt = (
                f"ID {item['track_id']} {item['class']} | "
                f"{item['apparent_speed_px_s']:.0f}px/s | "
                f"{item['decision']}"
            )
            cv2.putText(
                header, txt, (28 + i * 460, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.48,
                (230, 230, 230), 1, cv2.LINE_AA
            )

        # ----------------------------
        # Bandeau inférieur
        # ----------------------------
        footer = np.zeros((FOOTER_HEIGHT, WIDTH, 3), dtype=np.uint8)
        footer[:] = (22, 25, 31)

        footer_text = (
            "VITESSE = mouvement apparent en pixels/s | "
            "RISQUE = score heuristique | "
            "PAS DE DISTANCE METRIQUE NI DE VALIDATION DE SECURITE"
        )

        cv2.putText(
            footer, footer_text, (22, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.53,
            (190, 200, 210), 1, cv2.LINE_AA
        )

        dashboard_frame = np.vstack([header, annotated, footer])
        writer.write(dashboard_frame)

        frames_processed += 1

        # ----------------------------
        # Sauvegarde intermédiaire
        # ----------------------------
        if frames_processed % CHECKPOINT_INTERVAL == 0:
            pd.DataFrame(rows).to_csv(TRACKING_CSV, index=False)

            elapsed = time.time() - start_time
            print(
                f"[{frames_processed}/{N_FRAMES}] "
                f"{100*frames_processed/N_FRAMES:.1f}% | "
                f"{timestamp_s:.1f}s | "
                f"{fps_proc:.2f} FPS | "
                f"{total_observations} observations | "
                f"{elapsed/60:.1f} min"
            )

finally:
    cap.release()
    writer.release()

# ------------------------------------------------
# 7. EXPORT FINAL
# ------------------------------------------------
tracking_df = pd.DataFrame(rows)

if tracking_df.empty:
    raise RuntimeError(
        "Aucune observation enregistrée. Vérifie le modèle et la vidéo."
    )

tracking_df.to_csv(TRACKING_CSV, index=False)

# Tableau des vitesses apparentes
speed_df = tracking_df[
    tracking_df["track_id"] >= 0
].copy()

speed_df.to_csv(SPEED_CSV, index=False)

# Consolidation par piste
track_summary = (
    speed_df.groupby(["track_id", "class"], dropna=False)
    .agg(
        observations=("frame", "count"),
        debut_s=("timestamp_s", "min"),
        fin_s=("timestamp_s", "max"),
        vitesse_apparente_moy_px_s=("apparent_speed_px_s", "mean"),
        vitesse_apparente_med_px_s=("apparent_speed_px_s", "median"),
        vitesse_apparente_max_px_s=("apparent_speed_px_s", "max"),
        score_visuel_moyen=("visual_risk_score", "mean"),
        score_visuel_max=("visual_risk_score", "max"),
        niveau_decision_max=("decision_level", "max")
    )
    .reset_index()
)

def decision_track(row):
    # Consolidation descriptive ; ce n'est pas une décision de sécurité.
    if row["score_visuel_moyen"] >= 50 and row["observations"] >= 10:
        return "PRIORITE_ANALYSE"
    if row["score_visuel_moyen"] >= 35 and row["observations"] >= 5:
        return "ALERTE_VISUELLE"
    if row["score_visuel_max"] >= 25:
        return "SURVEILLANCE"
    return "FAIBLE_PRIORITE"

track_summary["decision_consolidee"] = track_summary.apply(
    decision_track, axis=1
)
track_summary.to_csv(DECISIONS_CSV, index=False)

# ------------------------------------------------
# 8. METADONNÉES ET AUDIT
# ------------------------------------------------
elapsed_total = time.time() - start_time

cap_check = cv2.VideoCapture(str(VIDEO_FINAL))
video_ok = cap_check.isOpened()
out_frames = int(cap_check.get(cv2.CAP_PROP_FRAME_COUNT)) if video_ok else 0
out_fps = cap_check.get(cv2.CAP_PROP_FPS) if video_ok else 0
out_w = int(cap_check.get(cv2.CAP_PROP_FRAME_WIDTH)) if video_ok else 0
out_h = int(cap_check.get(cv2.CAP_PROP_FRAME_HEIGHT)) if video_ok else 0
cap_check.release()

metadata = {
    "project": "ADAS-FUSION X 2030",
    "mode": "full_video_exploratory_analysis",
    "source_video": str(VIDEO),
    "model": MODEL_NAME,
    "tracker": "ByteTrack",
    "device": str(torch.cuda.get_device_name(0))
              if torch.cuda.is_available() else "CPU",
    "source_fps": FPS,
    "source_frames": N_FRAMES,
    "source_duration_s": DURATION,
    "processed_frames": frames_processed,
    "processed_duration_s": frames_processed / FPS,
    "observations": len(tracking_df),
    "unique_track_ids": int(
        tracking_df.loc[tracking_df["track_id"] >= 0, "track_id"].nunique()
    ),
    "frames_with_tracks": frames_with_tracks,
    "processing_time_s": elapsed_total,
    "processing_fps": frames_processed / max(elapsed_total, 1e-6),
    "output_video": str(VIDEO_FINAL),
    "output_video_readable": bool(video_ok),
    "output_video_frames": out_frames,
    "output_video_fps": out_fps,
    "output_video_width": out_w,
    "output_video_height": out_h,
    "limitations": [
        "Apparent speed in pixels/s; not physical speed in km/h.",
        "Visual risk scores are heuristic and are not collision probabilities.",
        "No ground-truth annotations are available for accuracy evaluation.",
        "Track IDs may switch or fragment.",
        "No metric depth or calibrated camera geometry is assumed."
    ]
}

with open(METADATA_JSON, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

with open(LOG_FILE, "w", encoding="utf-8") as f:
    f.write(json.dumps(metadata, indent=2, ensure_ascii=False))

print("\n" + "=" * 72)
print("TRAITEMENT TERMINÉ")
print("=" * 72)
print("Images traitées :", frames_processed, "/", N_FRAMES)
print("Couverture temporelle :", round(frames_processed / FPS, 2), "s")
print("Observations :", len(tracking_df))
print("Pistes uniques :", metadata["unique_track_ids"])
print("FPS de traitement :", round(metadata["processing_fps"], 2))
print("Durée de calcul :", round(elapsed_total / 60, 2), "min")
print("Vidéo lisible :", video_ok)
print("Images de sortie :", out_frames)
print("Résolution de sortie :", out_w, "x", out_h)
print("\nFichiers produits :")
print(" -", VIDEO_FINAL)
print(" -", TRACKING_CSV)
print(" -", SPEED_CSV)
print(" -", DECISIONS_CSV)
print(" -", METADATA_JSON)
print(" -", LOG_FILE)

print("\nTop pistes selon le score visuel moyen :")
print(
    track_summary.sort_values(
        "score_visuel_moyen", ascending=False
    ).head(10).to_string(index=False)
)


In [ ]:

from pathlib import Path
import json
import pandas as pd
import cv2
import os

BASE = Path("/kaggle/working/ADAS_FUSION_X2030")
OUT = BASE / "09_full_video_analysis"

files_to_check = {
    "Tracking CSV": OUT / "data" / "tracking_full.csv",
    "Vitesses apparentes": OUT / "data" / "vehicle_speeds_apparent.csv",
    "Décisions ADAS": OUT / "data" / "adas_decisions.csv",
    "Vidéo dashboard": OUT / "video" / "ADAS_FUSION_X2030_FULL_DASHBOARD.mp4",
    "Métadonnées": OUT / "metadata.json",
    "Journal": OUT / "execution_log.txt",
}

print("=" * 65)
print("DIAGNOSTIC ADAS-FUSION X 2030")
print("=" * 65)

for name, path in files_to_check.items():
    print(f"\n{name}")
    print("  Présent :", path.exists())

    if path.exists():
        print("  Taille :", round(path.stat().st_size / (1024**2), 2), "Mo")

        if path.name == "tracking_full.csv":
            try:
                df = pd.read_csv(path)
                print("  Observations :", len(df))
                print("  Dernière image :", df["frame"].max() if len(df) else None)
                print("  Dernier instant :", round(df["timestamp_s"].max(), 2)
                      if len(df) else None, "s")
            except Exception as e:
                print("  Lecture CSV impossible :", repr(e))

        elif path.name == "metadata.json":
            try:
                print(json.dumps(json.loads(path.read_text()), indent=2,
                                 ensure_ascii=False))
            except Exception as e:
                print("  Métadonnées incomplètes :", repr(e))

        elif path.suffix == ".mp4":
            cap = cv2.VideoCapture(str(path))
            if cap.isOpened():
                fps = cap.get(cv2.CAP_PROP_FPS)
                frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                print("  Vidéo lisible :", True)
                print("  Images :", frames)
                print("  FPS :", fps)
                print("  Durée :", round(frames / fps, 2) if fps else None, "s")
            else:
                print("  Vidéo lisible :", False)
            cap.release()

print("\nDiagnostic terminé.")


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/kaggle/working/ADAS_FUSION_X2030")
DATA = BASE / "09_full_video_analysis" / "data"

tracking = pd.read_csv(DATA / "tracking_full.csv")

print("=" * 72)
print("AUDIT DES PISTES ET DES VITESSES APPARENTES")
print("=" * 72)

# 1. Couverture temporelle
print("\n[1] COUVERTURE")
print("Observations :", len(tracking))
print("Images distinctes :", tracking["frame"].nunique())
print("Première image :", tracking["frame"].min())
print("Dernière image :", tracking["frame"].max())
print("Pistes distinctes :", tracking["track_id"].nunique())

# 2. Classes
print("\n[2] OBSERVATIONS PAR CLASSE")
print(tracking["class"].value_counts().to_string())

# 3. Durée et nombre d'observations par piste
summary = (
    tracking[tracking["track_id"] >= 0]
    .groupby(["track_id", "class"])
    .agg(
        observations=("frame", "count"),
        debut_s=("timestamp_s", "min"),
        fin_s=("timestamp_s", "max"),
        vitesse_mediane_px_s=("apparent_speed_px_s", "median"),
        vitesse_max_px_s=("apparent_speed_px_s", "max"),
        score_moyen=("visual_risk_score", "mean"),
        score_max=("visual_risk_score", "max")
    )
    .reset_index()
)

summary["duree_s"] = summary["fin_s"] - summary["debut_s"]

print("\n[3] DURÉE DES PISTES")
print(summary["duree_s"].describe().round(3).to_string())

print("\n[4] QUALITÉ DE SUIVI — NOMBRE D'OBSERVATIONS")
print(
    summary["observations"]
    .describe(percentiles=[.25, .5, .75, .9])
    .round(2)
    .to_string()
)

# 5. Distribution des vitesses apparentes
speed = tracking["apparent_speed_px_s"].replace(
    [np.inf, -np.inf], np.nan
).dropna()

print("\n[5] VITESSES APPARENTES — pixels/s")
print(speed.describe(percentiles=[.5, .9, .95, .99]).round(2).to_string())

print("\nObservations à vitesse nulle :", int((speed == 0).sum()))
print("Observations > 500 px/s :", int((speed > 500).sum()))
print("Observations > 1000 px/s :", int((speed > 1000).sum()))

# 6. Cohérence temporelle par piste
ordered = tracking[tracking["track_id"] >= 0].sort_values(
    ["track_id", "timestamp_s"]
).copy()

ordered["dt_s"] = ordered.groupby("track_id")["timestamp_s"].diff()

print("\n[6] INTERVALLES TEMPORELS ENTRE OBSERVATIONS")
print(ordered["dt_s"].dropna().describe().round(4).to_string())

# 7. Résumé des pistes les plus persistantes
print("\n[7] 15 PISTES LES PLUS PERSISTANTES")
print(
    summary.sort_values(
        ["observations", "duree_s"], ascending=False
    ).head(15).round(3).to_string(index=False)
)

# 8. Export de la synthèse
summary_path = DATA / "track_quality_summary.csv"
summary.to_csv(summary_path, index=False)

print("\nSynthèse exportée :", summary_path)
print("AUDIT TERMINÉ")


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/kaggle/working/ADAS_FUSION_X2030")
DATA = BASE / "09_full_video_analysis" / "data"

df = pd.read_csv(DATA / "tracking_full.csv")
df = df.sort_values(["track_id", "timestamp_s", "frame"]).copy()

print("=" * 72)
print("AUDIT AVANCÉ — IDENTITÉS, VITESSES ET CONTINUITÉ")
print("=" * 72)

# ---------------------------------------------------------
# 1. Cohérence des classes pour chaque identifiant
# ---------------------------------------------------------
class_audit = (
    df[df["track_id"] >= 0]
    .groupby("track_id")
    .agg(
        observations=("frame", "count"),
        nombre_classes=("class", "nunique"),
        classes=("class", lambda x: ", ".join(sorted(set(x.astype(str)))))
    )
    .reset_index()
)

class_changes = class_audit[class_audit["nombre_classes"] > 1]

print("\n[1] COHÉRENCE DES CLASSES")
print("IDs analysés :", len(class_audit))
print("IDs associés à plusieurs classes :", len(class_changes))

if not class_changes.empty:
    print(class_changes.sort_values(
        "nombre_classes", ascending=False
    ).head(20).to_string(index=False))
else:
    print("Aucun changement de classe détecté.")

# ---------------------------------------------------------
# 2. Déplacements entre observations consécutives d'une piste
# ---------------------------------------------------------
df["dt_s"] = df.groupby("track_id")["timestamp_s"].diff()
df["dx_px"] = df.groupby("track_id")["center_x"].diff()
df["dy_px"] = df.groupby("track_id")["center_y"].diff()

df["deplacement_px"] = np.hypot(df["dx_px"], df["dy_px"])

valid_dt = df["dt_s"] > 0
df["vitesse_recalculee_px_s"] = np.nan
df.loc[valid_dt, "vitesse_recalculee_px_s"] = (
    df.loc[valid_dt, "deplacement_px"] / df.loc[valid_dt, "dt_s"]
)

# Comparaison avec la vitesse enregistrée
df["ecart_vitesse_px_s"] = (
    df["vitesse_recalculee_px_s"] - df["apparent_speed_px_s"]
).abs()

print("\n[2] COHÉRENCE DES VITESSES")
print("Vitesse recalculée médiane :",
      round(df["vitesse_recalculee_px_s"].median(), 2), "px/s")
print("Vitesse recalculée P95 :",
      round(df["vitesse_recalculee_px_s"].quantile(.95), 2), "px/s")
print("Vitesse recalculée maximale :",
      round(df["vitesse_recalculee_px_s"].max(), 2), "px/s")
print("Écart médian avec la vitesse enregistrée :",
      round(df["ecart_vitesse_px_s"].median(), 4), "px/s")

# ---------------------------------------------------------
# 3. Observations extrêmes à inspecter
# ---------------------------------------------------------
extreme = df[
    (df["vitesse_recalculee_px_s"] > 1000) &
    (df["track_id"] >= 0)
].copy()

print("\n[3] OBSERVATIONS À VITESSE APPARENTE EXTRÊME")
print("Nombre d'observations > 1000 px/s :", len(extreme))

cols = [
    "frame", "timestamp_s", "track_id", "class",
    "confidence", "center_x", "center_y",
    "dt_s", "dx_px", "dy_px",
    "vitesse_recalculee_px_s"
]

if not extreme.empty:
    print(
        extreme[cols]
        .sort_values("vitesse_recalculee_px_s", ascending=False)
        .head(20)
        .round(3)
        .to_string(index=False)
    )

# ---------------------------------------------------------
# 4. Interruptions temporelles des pistes
# ---------------------------------------------------------
df["gap_s"] = df.groupby("track_id")["timestamp_s"].diff()

gaps = df[df["gap_s"] > (1.5 / 24)].copy()

print("\n[4] INTERRUPTIONS TEMPORELLES")
print("Intervalles > 1,5 image :", len(gaps))

if not gaps.empty:
    print(
        gaps[[
            "track_id", "class", "frame", "timestamp_s", "gap_s"
        ]]
        .sort_values("gap_s", ascending=False)
        .head(15)
        .round(3)
        .to_string(index=False)
    )

# ---------------------------------------------------------
# 5. Durée et stabilité des pistes
# ---------------------------------------------------------
track_summary = (
    df[df["track_id"] >= 0]
    .groupby("track_id")
    .agg(
        classe_majoritaire=(
            "class",
            lambda x: x.mode().iloc[0] if not x.mode().empty else "unknown"
        ),
        observations=("frame", "count"),
        debut_s=("timestamp_s", "min"),
        fin_s=("timestamp_s", "max"),
        vitesse_mediane_px_s=("vitesse_recalculee_px_s", "median"),
        vitesse_max_px_s=("vitesse_recalculee_px_s", "max")
    )
    .reset_index()
)

track_summary["duree_s"] = (
    track_summary["fin_s"] - track_summary["debut_s"]
)

track_summary["qualite_exploratoire"] = np.select(
    [
        track_summary["observations"] < 3,
        track_summary["observations"] < 10,
        track_summary["observations"] < 30
    ],
    [
        "Très courte",
        "Courte",
        "Intermédiaire"
    ],
    default="Persistante"
)

print("\n[5] RÉPARTITION DES PISTES")
print(track_summary["qualite_exploratoire"].value_counts().to_string())

print("\n[6] PISTES LES PLUS PERSISTANTES")
print(
    track_summary.sort_values(
        ["observations", "duree_s"], ascending=False
    ).head(15).round(3).to_string(index=False)
)

# ---------------------------------------------------------
# 7. Exports d'audit
# ---------------------------------------------------------
class_changes.to_csv(DATA / "audit_class_changes.csv", index=False)
extreme.to_csv(DATA / "audit_extreme_speeds.csv", index=False)
track_summary.to_csv(DATA / "audit_track_quality.csv", index=False)

print("\nExports créés :")
print(DATA / "audit_class_changes.csv")
print(DATA / "audit_extreme_speeds.csv")
print(DATA / "audit_track_quality.csv")
print("\nAUDIT TERMINÉ")

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/kaggle/working/ADAS_FUSION_X2030")
DATA = BASE / "09_full_video_analysis" / "data"

INPUT = DATA / "tracking_full.csv"
OUTPUT = DATA / "tracking_stabilized_full.csv"
AUDIT = DATA / "tracking_stabilization_audit.csv"

df = pd.read_csv(INPUT)
df = df[df["track_id"] >= 0].copy()
df = df.sort_values(["track_id", "timestamp_s", "frame"]).reset_index(drop=True)

# ---------------------------------------------------------
# 1. Classe dominante par identifiant
# ---------------------------------------------------------
majority_class = (
    df.groupby("track_id")["class"]
      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else "unknown")
      .rename("stable_class")
)

class_count = (
    df.groupby("track_id")["class"]
      .nunique()
      .rename("number_of_classes")
)

df = df.join(majority_class, on="track_id")
df = df.join(class_count, on="track_id")
df["class_changed"] = df["number_of_classes"] > 1

# ---------------------------------------------------------
# 2. Segmentation temporelle des pistes
# Une interruption > 0.25 s démarre un nouveau segment.
# ---------------------------------------------------------
df["dt_s"] = df.groupby("track_id")["timestamp_s"].diff()

df["new_segment"] = (
    df["dt_s"].isna() | (df["dt_s"] > 0.25) | (df["dt_s"] <= 0)
)

df["segment_number"] = (
    df.groupby("track_id")["new_segment"].cumsum().astype(int)
)

df["segment_id"] = (
    df["track_id"].astype(str) + "_" +
    df["segment_number"].astype(str)
)

# ---------------------------------------------------------
# 3. Recalcul des vitesses dans chaque segment
# ---------------------------------------------------------
group = df.groupby("segment_id", sort=False)

df["dx_px"] = group["center_x"].diff()
df["dy_px"] = group["center_y"].diff()
df["segment_dt_s"] = group["timestamp_s"].diff()

df["vx_raw_px_s"] = df["dx_px"] / df["segment_dt_s"]
df["vy_raw_px_s"] = df["dy_px"] / df["segment_dt_s"]

df["speed_raw_px_s"] = np.hypot(
    df["vx_raw_px_s"], df["vy_raw_px_s"]
)

# Première observation de chaque segment : vitesse inconnue.
first = df["segment_dt_s"].isna()
df.loc[first, ["vx_raw_px_s", "vy_raw_px_s", "speed_raw_px_s"]] = np.nan

# ---------------------------------------------------------
# 4. Détection des valeurs extrêmes
# Seuil exploratoire, pas seuil physique de sécurité.
# ---------------------------------------------------------
df["speed_anomaly"] = (
    df["speed_raw_px_s"] > 1000
)

# ---------------------------------------------------------
# 5. Filtrage médian temporel
# Le filtre est appliqué séparément à chaque segment.
# Les vitesses brutes restent conservées.
# ---------------------------------------------------------
df["vx_filtered_px_s"] = (
    df.groupby("segment_id")["vx_raw_px_s"]
      .transform(lambda x: x.rolling(
          window=5, min_periods=2, center=True
      ).median())
)

df["vy_filtered_px_s"] = (
    df.groupby("segment_id")["vy_raw_px_s"]
      .transform(lambda x: x.rolling(
          window=5, min_periods=2, center=True
      ).median())
)

df["speed_filtered_px_s"] = np.hypot(
    df["vx_filtered_px_s"],
    df["vy_filtered_px_s"]
)

# ---------------------------------------------------------
# 6. Qualité exploratoire des segments
# ---------------------------------------------------------
segment_summary = (
    df.groupby("segment_id")
      .agg(
          track_id=("track_id", "first"),
          stable_class=("stable_class", "first"),
          observations=("frame", "count"),
          start_s=("timestamp_s", "min"),
          end_s=("timestamp_s", "max"),
          anomalies=("speed_anomaly", "sum"),
          class_changed=("class_changed", "max")
      )
      .reset_index()
)

segment_summary["duration_s"] = (
    segment_summary["end_s"] - segment_summary["start_s"]
)

segment_summary["tracking_quality"] = np.select(
    [
        segment_summary["observations"] < 3,
        segment_summary["observations"] < 10,
        segment_summary["observations"] < 30
    ],
    [
        "INSUFFISANTE",
        "FAIBLE",
        "INTERMEDIAIRE"
    ],
    default="PERSISTANTE"
)

df = df.merge(
    segment_summary[["segment_id", "duration_s", "tracking_quality"]],
    on="segment_id",
    how="left",
    validate="many_to_one"
)

# ---------------------------------------------------------
# 7. Export
# ---------------------------------------------------------
df.to_csv(OUTPUT, index=False)
segment_summary.to_csv(AUDIT, index=False)

print("=" * 72)
print("STABILISATION TERMINÉE")
print("=" * 72)

print("Observations conservées :", len(df))
print("IDs initiaux :", df["track_id"].nunique())
print("Segments temporels :", df["segment_id"].nunique())
print("IDs avec changements de classe :", 
      df.groupby("track_id")["class_changed"].max().sum())

print("\nVitesses brutes (px/s) :")
print(df["speed_raw_px_s"].describe(
    percentiles=[.5, .9, .95, .99]
).round(2))

print("\nVitesses filtrées (px/s) :")
print(df["speed_filtered_px_s"].describe(
    percentiles=[.5, .9, .95, .99]
).round(2))

print("\nObservations extrêmes :", int(df["speed_anomaly"].sum()))

print("\nQualité des segments :")
print(segment_summary["tracking_quality"].value_counts())

print("\nFichiers exportés :")
print(OUTPUT)
print(AUDIT)


In [ ]:

import os
import cv2
import glob
import json
import numpy as np

ROOT = "/kaggle/working/ADAS_FUSION_X2030"

video_main = os.path.join(
    ROOT, "videos/ADAS_FUSION_X2030_Reference_180s.mp4"
)

video_seg = os.path.join(
    ROOT, "04_segmentation/ADAS_Segmentation_180s.mp4"
)

depth_dir = os.path.join(ROOT, "05_depth/raw_maps")

tracking_file = os.path.join(
    ROOT, "09_full_video_analysis/data/tracking_stabilized_full.csv"
)

def inspect_video(path, name):
    print(f"\n--- {name} ---")
    if not os.path.exists(path):
        print("FICHIER INTROUVABLE :", path)
        return

    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print("Chemin :", path)
    print("Dimensions :", width, "x", height)
    print("FPS :", fps)
    print("Frames :", frames)
    print("Durée :", round(frames / fps, 3) if fps else None)

    ok, frame = cap.read()
    print("Première frame lisible :", ok)
    if ok:
        print("Shape :", frame.shape)

    cap.release()

inspect_video(video_main, "VIDÉO PRINCIPALE")
inspect_video(video_seg, "SEGMENTATION")

print("\n--- CARTES DE PROFONDEUR ---")
depth_files = sorted(glob.glob(os.path.join(depth_dir, "depth_*.npy")))
print("Nombre de cartes :", len(depth_files))

if depth_files:
    sample = np.load(depth_files[0])
    print("Première carte :", os.path.basename(depth_files[0]))
    print("Dimensions :", sample.shape)
    print("Type :", sample.dtype)
    print("Min / Max :", float(np.nanmin(sample)), float(np.nanmax(sample)))

print("\n--- TRACKING STABILISÉ ---")
print("Fichier présent :", os.path.exists(tracking_file))

print("\n--- ESPACE DISQUE ---")
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print("Espace libre :", round(free / (1024**3), 2), "GB")

print("\nAUDIT TERMINÉ")


In [ ]:

import os
import json
import pandas as pd
import numpy as np

ROOT = "/kaggle/working/ADAS_FUSION_X2030"
DATA = os.path.join(ROOT, "09_full_video_analysis/data")

files = {
    "TRACKING STABILISÉ": os.path.join(DATA, "tracking_stabilized_full.csv"),
    "AUDIT TRACKING": os.path.join(DATA, "tracking_stabilization_audit.csv"),
    "DÉCISIONS ADAS": os.path.join(DATA, "adas_decisions.csv"),
    "VITESSES APPARENTES": os.path.join(DATA, "vehicle_speeds_apparent.csv"),
}

for name, path in files.items():
    print("\n" + "=" * 75)
    print(name)
    print("=" * 75)

    if not os.path.exists(path):
        print("Fichier absent :", path)
        continue

    df = pd.read_csv(path)
    print("Dimensions :", df.shape)
    print("Colonnes :", df.columns.tolist())
    print("\nAperçu :")
    display(df.head(3))

    print("\nValeurs manquantes (principales colonnes) :")
    print(df.isna().sum().sort_values(ascending=False).head(10))

# Métadonnées du traitement vidéo
metadata_path = os.path.join(DATA, "../metadata.json")

print("\n" + "=" * 75)
print("MÉTADONNÉES")
print("=" * 75)

if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    print(json.dumps(metadata, indent=2, ensure_ascii=False))
else:
    print("Fichier metadata.json non trouvé.")

# Contrôle de la cohérence temporelle du tracking
tracking_path = files["TRACKING STABILISÉ"]

if os.path.exists(tracking_path):
    df = pd.read_csv(tracking_path)

    if "frame" in df.columns:
        print("\nPlage des frames :", df["frame"].min(), "à", df["frame"].max())

    if "segment_id" in df.columns:
        print("Nombre de segments :", df["segment_id"].nunique())

    if "tracking_quality" in df.columns:
        print("\nQualité des segments :")
        print(df["tracking_quality"].value_counts())

print("\nAUDIT DES STRUCTURES TERMINÉ")


In [ ]:

import os
import cv2
import glob
import json
import numpy as np
import pandas as pd
from collections import defaultdict, deque
from tqdm.auto import tqdm

# ============================================================
# 1. CONFIGURATION
# ============================================================

ROOT = "/kaggle/working/ADAS_FUSION_X2030"

VIDEO_PATH = os.path.join(
    ROOT, "videos/ADAS_FUSION_X2030_Reference_180s.mp4"
)

SEG_PATH = os.path.join(
    ROOT, "04_segmentation/ADAS_Segmentation_180s.mp4"
)

DEPTH_DIR = os.path.join(ROOT, "05_depth/raw_maps")

TRACK_PATH = os.path.join(
    ROOT, "09_full_video_analysis/data/tracking_stabilized_full.csv"
)

OUT_DIR = os.path.join(
    ROOT, "10_multimodal_dashboard"
)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "video"), exist_ok=True)

OUTPUT_VIDEO = os.path.join(
    OUT_DIR, "video/ADAS_FUSION_X2030_MULTIMODAL_180s.mp4"
)

OUTPUT_METADATA = os.path.join(OUT_DIR, "dashboard_metadata.json")

# Dimensions du rendu
W, H = 1920, 1080
MAIN_W, MAIN_H = 1280, 720
PANEL_W, PANEL_H = 640, 360

TOP_H = 150
BOTTOM_H = 210

FPS = 24
SEG_INTERVAL = 6
DEPTH_INTERVAL = 12

# ============================================================
# 2. CHARGEMENT DU TRACKING
# ============================================================

print("Chargement du tracking stabilisé...")

df = pd.read_csv(TRACK_PATH)

required = [
    "frame", "segment_id", "stable_class",
    "x1", "y1", "x2", "y2",
    "center_x", "center_y",
    "confidence", "speed_filtered_px_s",
    "speed_anomaly", "visual_risk_score",
    "decision_level", "decision",
    "tracking_quality"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Colonnes manquantes : {missing}")

df["frame"] = df["frame"].astype(int)
df = df.sort_values(["frame", "segment_id"])

# Indexation par frame
frame_data = {
    int(frame): group.to_dict("records")
    for frame, group in df.groupby("frame")
}

print("Observations :", len(df))
print("Segments :", df["segment_id"].nunique())
print("Frames avec détections :", len(frame_data))

# ============================================================
# 3. OUVERTURE DES VIDÉOS
# ============================================================

main_cap = cv2.VideoCapture(VIDEO_PATH)
seg_cap = cv2.VideoCapture(SEG_PATH)

if not main_cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir la vidéo principale.")

if not seg_cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir la vidéo de segmentation.")

main_frames = int(main_cap.get(cv2.CAP_PROP_FRAME_COUNT))
main_fps = main_cap.get(cv2.CAP_PROP_FPS)

seg_frames = int(seg_cap.get(cv2.CAP_PROP_FRAME_COUNT))
seg_fps = seg_cap.get(cv2.CAP_PROP_FPS)

if main_frames != 4320 or abs(main_fps - 24) > 0.01:
    raise ValueError("Métadonnées de la vidéo principale inattendues.")

if seg_frames != 720 or abs(seg_fps - 4) > 0.01:
    raise ValueError("Métadonnées de segmentation inattendues.")

# ============================================================
# 4. INDEXATION DES CARTES DE PROFONDEUR
# ============================================================

depth_files = sorted(glob.glob(
    os.path.join(DEPTH_DIR, "depth_*.npy")
))

if len(depth_files) != 360:
    raise ValueError(
        f"360 cartes attendues, {len(depth_files)} trouvées."
    )

depth_paths = {}

for path in depth_files:
    name = os.path.basename(path)
    frame_index = int(
        name.replace("depth_", "").replace(".npy", "")
    )
    depth_paths[frame_index] = path

# Normalisation globale robuste : échantillonnage spatial
print("\nCalcul des bornes globales de profondeur...")

depth_samples = []

for path in tqdm(depth_files, desc="Audit profondeur"):
    arr = np.load(path)
    sample = arr[::8, ::8]
    sample = sample[np.isfinite(sample)]
    if sample.size:
        depth_samples.append(sample)

depth_samples = np.concatenate(depth_samples)

DEPTH_LOW = float(np.percentile(depth_samples, 2))
DEPTH_HIGH = float(np.percentile(depth_samples, 98))

del depth_samples

print("Percentile 2 :", DEPTH_LOW)
print("Percentile 98 :", DEPTH_HIGH)

if DEPTH_HIGH <= DEPTH_LOW:
    raise ValueError("Bornes de profondeur invalides.")

# ============================================================
# 5. FONCTIONS GRAPHIQUES
# ============================================================

FONT = cv2.FONT_HERSHEY_SIMPLEX

def put_text(img, text, x, y, scale=0.6,
             color=(255, 255, 255), thickness=1):
    cv2.putText(
        img, str(text), (int(x), int(y)),
        FONT, scale, color, thickness, cv2.LINE_AA
    )

def panel_title(img, title):
    cv2.rectangle(img, (0, 0), (img.shape[1], 38),
                  (25, 35, 48), -1)
    put_text(img, title, 14, 26, 0.65, (255, 255, 255), 2)

def risk_color(level):
    try:
        level = int(level)
    except:
        level = 0

    if level >= 2:
        return (0, 0, 255)
    elif level == 1:
        return (0, 180, 255)
    return (80, 220, 100)

def format_speed(value):
    if pd.isna(value):
        return "N/A"
    return f"{value:.1f} px/s"

def make_depth_panel(depth):
    # Normalisation globale identique pour toutes les images
    normalized = np.clip(
        (depth - DEPTH_LOW) / (DEPTH_HIGH - DEPTH_LOW),
        0, 1
    )

    depth_uint8 = (normalized * 255).astype(np.uint8)

    depth_color = cv2.applyColorMap(
        depth_uint8, cv2.COLORMAP_TURBO
    )

    panel = cv2.resize(
        depth_color, (PANEL_W, PANEL_H),
        interpolation=cv2.INTER_AREA
    )

    panel_title(panel, "DEPTH ANYTHING V2 | PROFONDEUR RELATIVE")

    put_text(
        panel, "Echelle visuelle normalisee - pas de distance en metres",
        12, PANEL_H - 14, 0.43, (255, 255, 255), 1
    )

    return panel

# ============================================================
# 6. INITIALISATION DU RENDU
# ============================================================

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    OUTPUT_VIDEO, fourcc, FPS, (W, H)
)

if not writer.isOpened():
    raise RuntimeError("Impossible de créer la vidéo de sortie.")

last_seg_panel = np.zeros(
    (PANEL_H, PANEL_W, 3), dtype=np.uint8
)

last_depth_panel = np.zeros(
    (PANEL_H, PANEL_W, 3), dtype=np.uint8
)

last_seg_age = 0
last_depth_age = 0

trajectory_history = defaultdict(
    lambda: deque(maxlen=30)
)

# Statistiques globales
total_anomalies = int(df["speed_anomaly"].fillna(False).sum())

quality_counts = (
    df.drop_duplicates("segment_id")["tracking_quality"]
    .value_counts().to_dict()
)

decision_counts = (
    df["decision"].value_counts().to_dict()
)

print("\nDébut du rendu multimodal...")
print("Fichier de sortie :", OUTPUT_VIDEO)

# ============================================================
# 7. TRAITEMENT DES 4320 FRAMES
# ============================================================

for frame_idx in tqdm(range(main_frames), desc="Dashboard 180s"):

    ok, frame = main_cap.read()

    if not ok:
        print("Arrêt : frame non lisible", frame_idx)
        break

    timestamp = frame_idx / FPS

    # --------------------------------------------------------
    # Segmentation : nouvelle image toutes les 6 frames
    # --------------------------------------------------------

    if frame_idx % SEG_INTERVAL == 0:
        ok_seg, seg_frame = seg_cap.read()

        if ok_seg:
            last_seg_panel = cv2.resize(
                seg_frame,
                (PANEL_W, PANEL_H),
                interpolation=cv2.INTER_AREA
            )
            last_seg_age = 0

    else:
        last_seg_age += 1

    seg_panel = last_seg_panel.copy()
    panel_title(seg_panel, "SEGFORMER | SEGMENTATION SEMANTIQUE")

    put_text(
        seg_panel,
        f"Image source approx. {frame_idx - last_seg_age}",
        12, PANEL_H - 14, 0.43, (255, 255, 255), 1
    )

    # --------------------------------------------------------
    # Profondeur : nouvelle carte toutes les 12 frames
    # --------------------------------------------------------

    if frame_idx % DEPTH_INTERVAL == 0:
        if frame_idx in depth_paths:
            depth = np.load(depth_paths[frame_idx])
            last_depth_panel = make_depth_panel(depth)
            last_depth_age = 0
    else:
        last_depth_age += 1

    depth_panel = last_depth_panel.copy()

    put_text(
        depth_panel,
        f"Carte de la frame {frame_idx - last_depth_age}",
        12, PANEL_H - 34, 0.43, (255, 255, 255), 1
    )

    # --------------------------------------------------------
    # Image principale
    # --------------------------------------------------------

    main_view = cv2.resize(
        frame, (MAIN_W, MAIN_H),
        interpolation=cv2.INTER_AREA
    )

    scale_x = MAIN_W / 1920
    scale_y = MAIN_H / 1080

    current_rows = frame_data.get(frame_idx, [])

    current_segments = set()
    current_speeds = []
    current_risks = []

    for row in current_rows:

        segment_id = str(row["segment_id"])
        current_segments.add(segment_id)

        x1 = int(float(row["x1"]) * scale_x)
        y1 = int(float(row["y1"]) * scale_y)
        x2 = int(float(row["x2"]) * scale_x)
        y2 = int(float(row["y2"]) * scale_y)

        x1 = np.clip(x1, 0, MAIN_W - 1)
        x2 = np.clip(x2, 0, MAIN_W - 1)
        y1 = np.clip(y1, 0, MAIN_H - 1)
        y2 = np.clip(y2, 0, MAIN_H - 1)

        cx = int(float(row["center_x"]) * scale_x)
        cy = int(float(row["center_y"]) * scale_y)

        level = row["decision_level"]
        color = risk_color(level)

        # Trajectoire propre au segment temporel
        trajectory_history[segment_id].append((cx, cy))

        points = list(trajectory_history[segment_id])

        if len(points) >= 2:
            for j in range(1, len(points)):
                cv2.line(
                    main_view,
                    points[j - 1],
                    points[j],
                    (255, 210, 0),
                    2,
                    cv2.LINE_AA
                )

        # Boîte de détection
        cv2.rectangle(
            main_view, (x1, y1), (x2, y2),
            color, 2
        )

        stable_class = str(row["stable_class"])
        speed = row["speed_filtered_px_s"]
        confidence = float(row["confidence"])

        speed_text = format_speed(speed)

        label = (
            f"{stable_class} | S:{segment_id} | {speed_text}"
        )

        label_y = max(18, y1 - 7)

        cv2.putText(
            main_view, label,
            (x1, label_y), FONT, 0.43,
            color, 1, cv2.LINE_AA
        )

        # Anomalies : marquage explicite
        if bool(row["speed_anomaly"]):
            cv2.circle(
                main_view, (cx, cy), 8,
                (0, 0, 255), 2
            )
            put_text(
                main_view, "ANOMALIE",
                cx + 8, cy - 8, 0.4, (0, 0, 255), 2
            )

        if pd.notna(speed):
            current_speeds.append(float(speed))

        if pd.notna(row["visual_risk_score"]):
            current_risks.append(float(row["visual_risk_score"]))

    # Nettoyage de l'historique des segments disparus
    # On conserve uniquement un historique borné par deque.

    # --------------------------------------------------------
    # Bandeau supérieur
    # --------------------------------------------------------

    canvas = np.zeros((H, W, 3), dtype=np.uint8)

    canvas[:TOP_H, :, :] = (18, 27, 39)

    put_text(
        canvas, "ADAS-FUSION X 2030",
        25, 43, 1.05, (255, 255, 255), 2
    )

    put_text(
        canvas, "MULTIMODAL PERCEPTION & EXPLORATORY DECISION SUPPORT",
        25, 78, 0.52, (160, 205, 230), 1
    )

    progress = 100 * (frame_idx + 1) / main_frames

    put_text(
        canvas,
        f"TIME {timestamp:06.2f}s / 180s",
        25, 119, 0.65, (255, 255, 255), 2
    )

    put_text(
        canvas,
        f"FRAME {frame_idx + 1}/4320",
        410, 119, 0.60, (255, 255, 255), 1
    )

    put_text(
        canvas,
        f"PROGRESSION {progress:.1f}%",
        720, 119, 0.60, (100, 220, 150), 2
    )

    put_text(
        canvas,
        f"DETECTIONS {len(current_rows)}",
        1120, 43, 0.60, (255, 255, 255), 1
    )

    put_text(
        canvas,
        f"SEGMENTS ACTIFS {len(current_segments)}",
        1120, 78, 0.60, (255, 255, 255), 1
    )

    put_text(
        canvas,
        f"ANOMALIES CUMULEES {total_anomalies}",
        1120, 119, 0.55, (0, 150, 255), 1
    )

    # Barre de progression
    cv2.rectangle(
        canvas, (25, 135), (W - 25, 143),
        (55, 65, 80), -1
    )

    cv2.rectangle(
        canvas, (25, 135),
        (25 + int((W - 50) * progress / 100), 143),
        (50, 190, 130), -1
    )

    # --------------------------------------------------------
    # Zone centrale : vidéo + deux panneaux
    # --------------------------------------------------------

    canvas[
        TOP_H:TOP_H + MAIN_H,
        0:MAIN_W
    ] = main_view

    canvas[
        TOP_H:TOP_H + PANEL_H,
        MAIN_W:W
    ] = seg_panel

    canvas[
        TOP_H + PANEL_H:TOP_H + 2 * PANEL_H,
        MAIN_W:W
    ] = depth_panel

    # --------------------------------------------------------
    # Bandeau inférieur
    # --------------------------------------------------------

    bottom_y = TOP_H + MAIN_H

    canvas[bottom_y:, :, :] = (18, 27, 39)

    put_text(
        canvas, "TRACKING QUALITY",
        25, bottom_y + 38, 0.65, (255, 255, 255), 2
    )

    quality_x = 25

    for quality in [
        "PERSISTANTE", "INTERMEDIAIRE",
        "FAIBLE", "INSUFFISANTE"
    ]:
        count = quality_counts.get(quality, 0)

        put_text(
            canvas,
            f"{quality}: {count}",
            quality_x, bottom_y + 75,
            0.48, (190, 210, 225), 1
        )

        quality_x += 300

    # Indicateurs courants
    median_speed = (
        float(np.median(current_speeds))
        if current_speeds else 0.0
    )

    max_risk = (
        float(np.max(current_risks))
        if current_risks else 0.0
    )

    put_text(
        canvas,
        f"MEDIAN APPARENT SPEED: {median_speed:.1f} px/s",
        25, bottom_y + 120, 0.58, (255, 255, 255), 1
    )

    put_text(
        canvas,
        f"MAX VISUAL RISK SCORE: {max_risk:.1f}",
        600, bottom_y + 120, 0.58, (0, 180, 255), 1
    )

    put_text(
        canvas,
        "WARNING: image-plane speed only | relative depth only | heuristic risk, not collision probability",
        25, bottom_y + 165, 0.48, (120, 190, 220), 1
    )

    put_text(
        canvas,
        f"SEG AGE: {last_seg_age / FPS:.2f}s",
        25, bottom_y + 195, 0.40, (170, 190, 205), 1
    )

    put_text(
        canvas,
        f"DEPTH AGE: {last_depth_age / FPS:.2f}s",
        230, bottom_y + 195, 0.40, (170, 190, 205), 1
    )

    # Écriture
    writer.write(canvas)

# ============================================================
# 8. FERMETURE ET VALIDATION
# ============================================================

writer.release()
main_cap.release()
seg_cap.release()

print("\n" + "=" * 70)
print("RENDU MULTIMODAL TERMINÉ")
print("=" * 70)

if os.path.exists(OUTPUT_VIDEO):
    size_gb = os.path.getsize(OUTPUT_VIDEO) / (1024**3)

    check = cv2.VideoCapture(OUTPUT_VIDEO)

    out_frames = int(check.get(cv2.CAP_PROP_FRAME_COUNT))
    out_fps = check.get(cv2.CAP_PROP_FPS)
    out_w = int(check.get(cv2.CAP_PROP_FRAME_WIDTH))
    out_h = int(check.get(cv2.CAP_PROP_FRAME_HEIGHT))

    ok, test_frame = check.read()
    check.release()

    print("Fichier :", OUTPUT_VIDEO)
    print("Taille :", round(size_gb, 3), "GB")
    print("Frames :", out_frames)
    print("FPS :", out_fps)
    print("Résolution :", out_w, "x", out_h)
    print("Première frame lisible :", ok)

    metadata = {
        "project": "ADAS-FUSION X 2030",
        "video": OUTPUT_VIDEO,
        "duration_s": 180,
        "fps": FPS,
        "frames": out_frames,
        "resolution": [out_w, out_h],
        "tracking_observations": len(df),
        "tracking_segments": int(df["segment_id"].nunique()),
        "depth_maps": len(depth_files),
        "segmentation_frames": seg_frames,
        "depth_normalization_percentiles": [2, 98],
        "depth_global_min": DEPTH_LOW,
        "depth_global_max": DEPTH_HIGH,
        "limitations": [
            "Apparent image-plane speed in pixels per second.",
            "Relative depth is not metric distance.",
            "Visual risk score is heuristic.",
            "Track identities may switch or fragment.",
            "Segmentation temporal alignment assumes sampling starts at frame 0.",
            "No ground-truth accuracy evaluation."
        ]
    }

    with open(OUTPUT_METADATA, "w") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    print("Métadonnées :", OUTPUT_METADATA)

else:
    print("ERREUR : fichier vidéo non généré.")


In [ ]:

import cv2
import os
from IPython.display import display, Image

video_path = (
    "/kaggle/working/ADAS_FUSION_X2030/"
    "10_multimodal_dashboard/video/"
    "ADAS_FUSION_X2030_MULTIMODAL_180s.mp4"
)

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir le dashboard.")

fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

times = [5, 30, 60, 120, 175]

print("Contrôle visuel du dashboard")
print("Durée :", total / fps, "secondes")

for t in times:
    frame_number = int(t * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

    ok, frame = cap.read()

    if ok:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        print(f"\nInstant : {t} secondes")
        display(Image(data=cv2.imencode(
            ".jpg",
            cv2.cvtColor(frame, cv2.COLOR_RGB2BGR),
            [cv2.IMWRITE_JPEG_QUALITY, 90]
        )[1].tobytes()))

cap.release()


In [ ]:
import os

print("Contenu de /kaggle/working :")

for item in os.listdir("/kaggle/working"):
    path = os.path.join("/kaggle/working", item)

    if os.path.isdir(path):
        print("[DOSSIER]", item)
    else:
        print("[FICHIER]", item)
    

In [ ]:
import os
import zipfile
import pandas as pd

zip_path = "/kaggle/working/ADAS_FUSION_X2030_RESULTS.zip"

with zipfile.ZipFile(zip_path, "r") as archive:
    files = archive.infolist()

    inventory = pd.DataFrame([
        {
            "File": f.filename,
            "Extension": os.path.splitext(f.filename)[1].lower(),
            "Size_MB": round(f.file_size / (1024 * 1024), 3)
        }
        for f in files if not f.is_dir()
    ])

print("Nombre total de fichiers :", len(inventory))
print("\nRépartition par extension :")
print(inventory["Extension"].value_counts())

display(inventory.sort_values("Size_MB", ascending=False))

inventory.to_csv(
    "/kaggle/working/ADAS_FUSION_X2030_ARCHIVE_INVENTORY.csv",
    index=False
)

In [ ]:
!kaggle kernels output khourdouchamine/adas-fusion-x2030 -p /kaggle/working/ADAS_FUSION_X2030_Output